# RoadLens - test the trained models

This notebook **trains nothing**. It loads the `best.pt` checkpoints that `RoadLens_standalone.ipynb` saved on Google Drive and tests them. Tests that the experiments already ran are read from their cache, so they finish at once.

**How to run:** `Runtime > Change runtime type > T4 GPU`, check `PROFILE` and `SEED` in the first code cell (they must match the training run), then `Runtime > Run all` and approve the Drive prompt.

| Section | What it does |
|---|---|
| 0. Project code | writes the project files (needed to rebuild the networks) |
| 1. Setup and checkpoints | lists every finished model on Drive |
| 2. Data | GTSRB test set, Belgian signs, corrupted test sets (restored from Drive or built) |
| 3. Tests | clean accuracy; 14 corruptions x 5 severities with mCE; Belgian signs |
| 4. Look inside | confusion matrix, example predictions, your own photos |
| 5. Save | tables and figures to `MyDrive/RoadLens_standalone/test_results*/` and a zip |

Use the code generated together with the training notebook: a checkpoint is only loaded if its configuration has not changed since training.

In [ ]:
DIRS = ['configs', 'configs/experiments', 'configs/profiles', 'configs/runs', 'roadlens', 'roadlens/augment', 'roadlens/data', 'roadlens/data/mappings', 'roadlens/eval', 'roadlens/models', 'roadlens/tta', 'scripts', 'tests']
ORDER = ['smallcnn_base', 'resnet18_base', 'deit_s_base', 'resnet18_mix', 'resnet18_mix_jsd', 'resnet18_full', 'deit_s_full', 'ensemble_avg', 'ensemble_conf', 'ensemble_views', 'fusion_full', 'resnet18_jsd_prime', 'resnet18_jsd_nogeom', 'resnet18_in', 'deit_s_in', 'fusion_v2', 'ensemble_avg_in', 'ensemble_views_in', 'resnet18_ln', 'resnet18_ln_apr', 'resnet18_v3', 'resnet18_v3_64', 'resnet18_v3_ssl', 'convnext_atto_v3', 'efficientnet_b0_v3', 'mobilenetv3_v3', 'deit_s_v3', 'student_v3', 'student_mbv3', 'ensemble_v3']
ENSEMBLES = ['ensemble_avg', 'ensemble_conf', 'ensemble_views', 'ensemble_avg_in', 'ensemble_views_in', 'ensemble_v3']
# ---- Settings: change these, then Runtime > Run all ----
PROFILE = 'pilot'      # must match the training notebook: 'pilot' | 'smoke' | '' (full)
SEED    = 0            # which trained seed to test (0, 1 or 2 for the full run)
MODELS  = []           # [] = every trained model found, or e.g. ['resnet18_full', 'fusion_full']
STORE   = '/content/drive/MyDrive/RoadLens_standalone'   # the training notebook's Drive folder
GENERATE_MISSING_C = True   # build corrupted test sets that are not saved on Drive (can take a while)
TEST_BTSD = True            # also test on Belgian signs (BTSD)
FORCE = False               # evaluate again even when a cached result exists

import glob, io, json, os, shutil, subprocess, sys, time
CODE = '/content/RoadLens'                 # the project is written here by the cells below
DATA = '/content/roadlens_data'            # datasets on the fast local disk
from google.colab import drive
drive.mount('/content/drive')
if not os.path.isdir(STORE):
    raise FileNotFoundError(f'{STORE} not found: run RoadLens_standalone.ipynb first, or fix STORE')
CACHE = f'{STORE}/corrupted_cache'
for d in DIRS:
    os.makedirs(f'{CODE}/{d}', exist_ok=True)
PROF = f'--profile {PROFILE}' if PROFILE else ''
SUFFIX = f'_{PROFILE}' if PROFILE else ''
RUNS_DIR = f'{STORE}/runs{SUFFIX}'
OUT = f'{STORE}/test_results{SUFFIX}'
os.makedirs(OUT, exist_ok=True)
print('profile:', PROFILE or 'full', '| seed', SEED, '| checkpoints:', RUNS_DIR, '| results ->', OUT)

## 0. Project code

The same 106 files as in the training notebook.

### `roadlens/`

In [ ]:
%%writefile /content/RoadLens/roadlens/__init__.py
"""RoadLens: traffic sign classification robust to image corruptions."""

__version__ = "0.1.0"


In [ ]:
%%writefile /content/RoadLens/roadlens/augment/__init__.py
from .augmix import AugMix, ColorJitter, TrainAugment, jsd_loss  # noqa: F401
from .ops import OP_SETS, OPS, resolve_ops  # noqa: F401


In [ ]:
%%writefile /content/RoadLens/roadlens/augment/augmix.py
"""AugMix-style mixing, colour jitter and the JSD consistency loss (all batched, on the GPU)."""

from __future__ import annotations

import math

import torch
import torch.nn.functional as F

from .ops import AFFINE_KIND, OPS, affine, resolve_ops


class AugMix:
    """x_aug = (1 - m) * x + m * sum_k w_k * chain_k(x)

    with `width` chains of `depth` randomly chosen ops each, w ~ Dirichlet(alpha, ..., alpha) and
    m ~ Beta(alpha, alpha), all sampled per image. Op magnitudes are U(0.1, severity) / 10 as in
    the AugMix reference implementation.
    """

    def __init__(self, ops, width: int = 3, depth=(1, 3), alpha: float = 1.0, severity: float = 3.0):
        self.names = list(ops)
        self.width = int(width)
        self.depth = (int(depth), int(depth)) if isinstance(depth, (int, float)) else (int(depth[0]), int(depth[1]))
        self.alpha = float(alpha)
        self.severity = float(severity)
        # Ops are applied group by group. Every op is its own group, except the affine ones, which
        # share a single resampling call (GPU time here is dominated by the number of calls).
        self.groups, group_of, kind_of = [], [], []
        affine_group = None
        for name in self.names:
            kind_of.append(AFFINE_KIND.get(name, -1))
            if name in AFFINE_KIND:
                if affine_group is None:
                    affine_group = len(self.groups)
                    self.groups.append(None)                         # filled in below
                group_of.append(affine_group)
            else:
                group_of.append(len(self.groups))
                self.groups.append(lambda x, level, idx, op=OPS[name]: op(x, level))
        if affine_group is not None:
            self.groups[affine_group] = lambda x, level, idx: affine(x, level, self._kind_of[idx])
        self._group_of, self._kind_of = torch.tensor(group_of), torch.tensor(kind_of)

    @torch.no_grad()
    def __call__(self, x: torch.Tensor, n_views: int = 1) -> list[torch.Tensor]:
        """Return `n_views` independently augmented versions of the batch (computed in one pass)."""
        b, dev = x.shape[0], x.device
        if self._group_of.device != dev:
            self._group_of, self._kind_of = self._group_of.to(dev), self._kind_of.to(dev)
        rows = n_views * self.width
        n, n_groups = rows * b, len(self.groups)
        chains = x.repeat(rows, 1, 1, 1)                             # [views * width * B, C, H, W]
        depth = torch.randint(self.depth[0], self.depth[1] + 1, (n,), device=dev)
        for step in range(self.depth[1]):
            op_idx = torch.randint(len(self.names), (n,), device=dev)
            level = torch.empty(n, device=dev).uniform_(0.1, self.severity) / 10.0
            # sort rows by group so that each group is one contiguous slice; finished chains go last
            key = torch.where(depth > step, self._group_of[op_idx], n_groups)
            order = torch.argsort(key)
            counts = torch.bincount(key, minlength=n_groups + 1).tolist()
            xs, lv, oi = chains[order], level[order], op_idx[order]
            start = 0
            for g in range(n_groups):
                end = start + counts[g]
                if end > start:
                    xs[start:end] = self.groups[g](xs[start:end], lv[start:end], oi[start:end]).to(xs.dtype)
                start = end
            chains[order] = xs
        conc = torch.full((n_views, b, self.width), self.alpha, device=dev)
        w = torch.distributions.Dirichlet(conc).sample()             # [views, B, width]
        a = torch.full((n_views, b), self.alpha, device=dev)
        m = torch.distributions.Beta(a, a).sample().view(n_views, b, 1, 1, 1)
        chains = chains.view(n_views, self.width, *x.shape)
        mixed = (chains * w.permute(0, 2, 1).reshape(n_views, self.width, b, 1, 1, 1)).sum(1)
        out = ((1 - m) * x + m * mixed).clamp_(0, 1)
        return list(out.unbind(0))


class ColorJitter:
    """Mild per-image brightness / contrast / saturation / hue jitter (the only baseline augmentation).

    No horizontal flip anywhere: it would turn 'keep left' into 'keep right'.
    """

    def __init__(self, brightness: float = 0.2, contrast: float = 0.2, saturation: float = 0.2, hue: float = 0.02):
        self.b, self.c, self.s, self.h = float(brightness), float(contrast), float(saturation), float(hue)

    @staticmethod
    def _factor(n, amount, dev):
        return torch.empty(n, 1, 1, 1, device=dev).uniform_(1 - amount, 1 + amount)

    @staticmethod
    def _gray(x):
        return (0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3])

    @torch.no_grad()
    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        n, dev = x.shape[0], x.device
        if self.b > 0:
            x = (x * self._factor(n, self.b, dev)).clamp_(0, 1)
        if self.c > 0:
            mean = self._gray(x).mean(dim=(2, 3), keepdim=True)
            x = ((x - mean) * self._factor(n, self.c, dev) + mean).clamp_(0, 1)
        if self.s > 0:
            gray = self._gray(x)
            x = ((x - gray) * self._factor(n, self.s, dev) + gray).clamp_(0, 1)
        if self.h > 0:
            # hue rotation = rotation of the chroma plane in YIQ space
            ang = torch.empty(n, device=dev).uniform_(-self.h, self.h) * 2 * math.pi
            cos, sin = torch.cos(ang), torch.sin(ang)
            to_yiq = torch.tensor([[0.299, 0.587, 0.114], [0.596, -0.274, -0.322], [0.211, -0.523, 0.312]],
                                  device=dev, dtype=x.dtype)
            rot = torch.zeros(n, 3, 3, device=dev, dtype=x.dtype)
            rot[:, 0, 0] = 1
            rot[:, 1, 1], rot[:, 1, 2], rot[:, 2, 1], rot[:, 2, 2] = cos, -sin, sin, cos
            m = torch.linalg.inv(to_yiq) @ rot @ to_yiq                # [B, 3, 3]
            x = torch.einsum("bij,bjhw->bihw", m, x).clamp_(0, 1)
        return x


class TrainAugment:
    """Turns a clean batch into the views the loss needs.

    mode 'none': [jitter(x)]                          standard training
    mode 'mix' : [augmix(jitter(x))]                  AugMix without the consistency loss
    mode 'jsd' : [jitter(x), augmix(.), augmix(.)]    clean + two mixed views for the JSD loss
    """

    def __init__(self, cfg: dict):
        self.mode = cfg.get("mode", "none")
        if self.mode not in ("none", "mix", "jsd"):
            raise ValueError(f"augment.mode must be none | mix | jsd, got {self.mode!r}")
        cj = cfg.get("color_jitter") or {}
        self.jitter = ColorJitter(**cj) if any(v > 0 for v in cj.values()) else None
        self.augmix = None
        if self.mode != "none":
            self.op_names = resolve_ops(cfg["ops"], cfg.get("exclude_ops"))
            self.augmix = AugMix(self.op_names, cfg.get("width", 3), cfg.get("depth", (1, 3)),
                                 cfg.get("alpha", 1.0), cfg.get("severity", 3.0))

    @property
    def n_views(self) -> int:
        return 3 if self.mode == "jsd" else 1

    @torch.no_grad()
    def __call__(self, x: torch.Tensor) -> list[torch.Tensor]:
        if self.jitter is not None:
            x = self.jitter(x)
        if self.mode == "none":
            return [x]
        if self.mode == "mix":
            return self.augmix(x, 1)
        return [x] + self.augmix(x, 2)


def jsd_loss(logits_views: list[torch.Tensor]) -> torch.Tensor:
    """Jensen-Shannon divergence between the predictive distributions of the views."""
    probs = [F.softmax(z.float(), dim=1) for z in logits_views]
    log_mix = torch.clamp(torch.stack(probs).mean(0), 1e-7, 1.0).log()
    return sum(F.kl_div(log_mix, p, reduction="batchmean") for p in probs) / len(probs)


In [ ]:
%%writefile /content/RoadLens/roadlens/augment/ops.py
"""Batched GPU image operations used as AugMix-style training augmentations.

Every op has the signature ``op(x, level) -> x'`` with

    x      float tensor [B, 3, H, W] in [0, 1]
    level  float tensor [B] in [0, 1]: per-sample magnitude (0 = weakest)

and draws its randomness from torch's global generator on ``x.device``.

Two families:

* AugMix ops (Hendrycks et al. 2020) minus the ones that reproduce a test corruption
  (brightness, contrast, colour, sharpness are left out): autocontrast, equalize, posterize,
  solarize, rotate, shear_x/y, translate_x/y. Magnitude ranges follow the AugMix reference code.
* PRIME-style primitives (Modas et al. 2022), re-implemented here with their own documented
  parameterisation: fourier_noise, spectral_filter, spatial_warp, color_curve.

None of the ops is one of the 14 test corruptions. Related pairs are listed in the README
("Augmentation / test-corruption overlap") so they can be ablated with `augment.exclude_ops`.
"""

from __future__ import annotations

import math

import torch
import torch.nn.functional as F
from torchvision.transforms.v2 import functional as TF


def _sign(level: torch.Tensor) -> torch.Tensor:
    return torch.where(torch.rand_like(level) < 0.5, -1.0, 1.0)


def _to_u8(x: torch.Tensor) -> torch.Tensor:
    return (x * 255.0).round().clamp_(0, 255).to(torch.uint8)


def _from_u8(x8: torch.Tensor) -> torch.Tensor:
    return x8.float().div_(255.0)


def _affine(x: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    """theta [B, 2, 3] maps output coordinates to input coordinates (normalised, centre origin).
    Pixels that fall outside the source image are black, as in AugMix."""
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)


def _theta(a, b, c, d, e, f) -> torch.Tensor:
    return torch.stack([torch.stack([a, b, c], dim=1), torch.stack([d, e, f], dim=1)], dim=1)


# ----------------------------------------------------------------------------------------
# AugMix ops
# ----------------------------------------------------------------------------------------
def autocontrast(x, level):
    """Stretch every channel of every image to the full [0, 1] range (level unused)."""
    lo = x.amin(dim=(2, 3), keepdim=True)
    hi = x.amax(dim=(2, 3), keepdim=True)
    span = hi - lo
    return torch.where(span > 1e-6, (x - lo) / span.clamp_min(1e-6), x)


def equalize(x, level):
    """Per-channel histogram equalisation (level unused)."""
    return _from_u8(TF.equalize(_to_u8(x)))


def posterize(x, level):
    """Keep the 4 - floor(4 * level) most significant bits (at least 1)."""
    bits = (4 - torch.floor(level * 4)).clamp(1, 8)
    shift = (8 - bits).to(torch.uint8).view(-1, 1, 1, 1)
    x8 = _to_u8(x)
    return _from_u8(torch.bitwise_left_shift(torch.bitwise_right_shift(x8, shift), shift))


def solarize(x, level):
    """Invert pixel values at or above 256 - floor(256 * level) (on the 0..255 scale)."""
    thr = (256 - torch.floor(level * 256)).view(-1, 1, 1, 1)
    x8 = _to_u8(x).float()
    return torch.where(x8 >= thr, 255.0 - x8, x8) / 255.0


def rotate(x, level):
    """Rotate by up to +-30 degrees."""
    ang = level * math.radians(30.0) * _sign(level)
    cos, sin, zero = torch.cos(ang), torch.sin(ang), torch.zeros_like(ang)
    return _affine(x, _theta(cos, -sin, zero, sin, cos, zero))


def shear_x(x, level):
    """Horizontal shear by up to +-0.3."""
    s, one, zero = level * 0.3 * _sign(level), torch.ones_like(level), torch.zeros_like(level)
    return _affine(x, _theta(one, s, zero, zero, one, zero))


def shear_y(x, level):
    """Vertical shear by up to +-0.3."""
    s, one, zero = level * 0.3 * _sign(level), torch.ones_like(level), torch.zeros_like(level)
    return _affine(x, _theta(one, zero, zero, s, one, zero))


def translate_x(x, level):
    """Horizontal shift by up to +-1/3 of the image width."""
    t, one, zero = level * (2.0 / 3.0) * _sign(level), torch.ones_like(level), torch.zeros_like(level)
    return _affine(x, _theta(one, zero, t, zero, one, zero))


def translate_y(x, level):
    """Vertical shift by up to +-1/3 of the image height."""
    t, one, zero = level * (2.0 / 3.0) * _sign(level), torch.ones_like(level), torch.zeros_like(level)
    return _affine(x, _theta(one, zero, zero, zero, one, t))


AFFINE_KIND = {"rotate": 0, "shear_x": 1, "shear_y": 2, "translate_x": 3, "translate_y": 4}


def affine(x, level, kind):
    """rotate / shear_x / shear_y / translate_x / translate_y in one resampling call.

    kind: int tensor [B] with the AFFINE_KIND code of each row. Same result as calling the five ops
    separately; AugMix uses this so that a batch needs one grid_sample instead of five.
    """
    s = _sign(level)
    ang, sh, tr = level * math.radians(30.0) * s, level * 0.3 * s, level * (2.0 / 3.0) * s
    one, zero = torch.ones_like(level), torch.zeros_like(level)
    rot = kind == 0
    cos, sin = torch.where(rot, torch.cos(ang), one), torch.where(rot, torch.sin(ang), zero)
    return _affine(x, _theta(cos, torch.where(kind == 1, sh, -sin), torch.where(kind == 3, tr, zero),
                             torch.where(kind == 2, sh, sin), cos, torch.where(kind == 4, tr, zero)))


# ----------------------------------------------------------------------------------------
# PRIME-style primitives
# ----------------------------------------------------------------------------------------
def fourier_noise(x, level, max_amp: float = 0.5):
    """Add one random planar wave (a single Fourier basis) to every channel.

    Per sample and channel: spatial frequency with radius ~ U[1, H/2] cycles/image and random
    orientation, random phase, amplitude = level * max_amp. This is structured, band-limited
    noise; it is not i.i.d. pixel noise (gaussian / shot / impulse are test corruptions).
    """
    b, c, h, w = x.shape
    dev = x.device
    radius = torch.empty(b, c, 1, 1, device=dev).uniform_(1.0, min(h, w) / 2.0)
    angle = torch.empty(b, c, 1, 1, device=dev).uniform_(0.0, math.pi)
    fy, fx = torch.round(radius * torch.sin(angle)), torch.round(radius * torch.cos(angle))
    fx = torch.where((fy == 0) & (fx == 0), torch.ones_like(fx), fx)
    phase = torch.empty(b, c, 1, 1, device=dev).uniform_(0.0, 2 * math.pi)
    yy = torch.arange(h, device=dev, dtype=x.dtype).view(1, 1, h, 1) / h
    xx = torch.arange(w, device=dev, dtype=x.dtype).view(1, 1, 1, w) / w
    wave = torch.cos(2 * math.pi * (fy * yy + fx * xx) + phase)
    return (x + (level * max_amp).view(-1, 1, 1, 1) * wave).clamp_(0, 1)


def spectral_filter(x, level, max_sigma: float = 1.0, k: int = 3):
    """Convolve with a random k x k kernel: identity + sigma * zero-sum Gaussian noise,
    sigma = level * max_sigma, one kernel per sample shared by its channels.

    The perturbation sums to zero, so the DC gain stays 1: the op reshapes the spatial spectrum
    (sharpen / emboss / slight smoothing) without changing mean brightness.
    """
    b, c, h, w = x.shape
    noise = torch.randn(b, 1, k, k, device=x.device, dtype=x.dtype)
    noise = noise - noise.mean(dim=(2, 3), keepdim=True)
    kernel = noise * (level * max_sigma).view(-1, 1, 1, 1)
    kernel[:, :, k // 2, k // 2] += 1.0
    weight = kernel.repeat_interleave(c, dim=0)                       # [B*C, 1, k, k]
    xp = F.pad(x.reshape(1, b * c, h, w), [k // 2] * 4, mode="reflect")
    return F.conv2d(xp, weight, groups=b * c).reshape(b, c, h, w).clamp_(0, 1)


def spatial_warp(x, level, max_disp: float = 0.1, modes: int = 4):
    """Smooth random diffeomorphism-like warp.

    Displacement field = sum over i, j <= modes of c_ij sin(i pi u) sin(j pi v) with
    c_ij ~ N(0, 1 / (i^2 + j^2)), rescaled so that the largest displacement is
    level * max_disp of the image side. The field vanishes on the border.
    """
    b, c, h, w = x.shape
    dev, dt = x.device, x.dtype
    freqs = torch.arange(1, modes + 1, device=dev, dtype=dt)
    u = (torch.arange(h, device=dev, dtype=dt) + 0.5) / h
    v = (torch.arange(w, device=dev, dtype=dt) + 0.5) / w
    su = torch.sin(math.pi * freqs[:, None] * u[None, :])             # [M, H]
    sv = torch.sin(math.pi * freqs[:, None] * v[None, :])             # [M, W]
    std = 1.0 / torch.sqrt(freqs[:, None] ** 2 + freqs[None, :] ** 2)
    coef = torch.randn(b, 2, modes, modes, device=dev, dtype=dt) * std
    field = torch.einsum("bdij,ih,jw->bdhw", coef, su, sv)            # [B, 2, H, W] (x, y)
    peak = field.abs().amax(dim=(1, 2, 3), keepdim=True).clamp_min(1e-8)
    field = field / peak * (level * max_disp * 2.0).view(-1, 1, 1, 1)  # grid units span [-1, 1]
    eye = torch.eye(2, 3, device=dev, dtype=dt).unsqueeze(0).expand(b, 2, 3)
    grid = F.affine_grid(eye, [b, c, h, w], align_corners=False) + field.permute(0, 2, 3, 1)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)


def color_curve(x, level, scale: float = 0.5, n_freq: int = 4):
    """Random smooth tone curve per channel: x + sum_n b_n sin(n pi x), b_n ~ N(0, (level*scale/n)^2).

    The curve fixes 0 and 1, so black and white stay put while mid-tones move.
    """
    b, c, _, _ = x.shape
    n = torch.arange(1, n_freq + 1, device=x.device, dtype=x.dtype)
    coef = torch.randn(b, c, n_freq, device=x.device, dtype=x.dtype) * (level * scale).view(-1, 1, 1) / n
    out = x.clone()
    for i in range(n_freq):
        out += coef[:, :, i, None, None] * torch.sin(math.pi * n[i] * x)
    return out.clamp_(0, 1)


def amplitude_mix(x, level, max_mix: float = 2.0):
    """Amplitude-phase recombination (APR, Chen et al., ICCV 2021), RoadLens v3.

    Keep every image's Fourier phase and move its amplitude spectrum towards that of another
    image of the batch: A' = (1 - lam) A_x + lam A_other, lam = min(1, level * max_mix).
    Phase carries the shape (edges, symbols); amplitude carries contrast and texture energy, which
    is what noise, haze and contrast loss disturb. Training on images whose amplitude no longer
    matches their content pushes the network to rely on phase. It is not one of the test
    corruptions: no noise is added and nothing is blurred; the spectrum comes from a real sign.
    """
    b = x.shape[0]
    if b < 2:
        return x
    f = torch.fft.fft2(x.float())
    lam = (level * max_mix).clamp(0, 1).view(-1, 1, 1, 1)
    amp = (1 - lam) * f.abs() + lam * f.abs()[torch.randperm(b, device=x.device)]
    out = torch.fft.ifft2(torch.polar(amp, torch.angle(f))).real
    return out.clamp_(0, 1).to(x.dtype)


OPS = {
    "autocontrast": autocontrast, "equalize": equalize, "posterize": posterize, "solarize": solarize,
    "rotate": rotate, "shear_x": shear_x, "shear_y": shear_y,
    "translate_x": translate_x, "translate_y": translate_y,
    "fourier_noise": fourier_noise, "spectral_filter": spectral_filter,
    "spatial_warp": spatial_warp, "color_curve": color_curve,
    "amplitude_mix": amplitude_mix,
}

OP_SETS = {
    "augmix": ["autocontrast", "equalize", "posterize", "solarize", "rotate",
               "shear_x", "shear_y", "translate_x", "translate_y"],
    "prime": ["fourier_noise", "spectral_filter", "spatial_warp", "color_curve"],
    "apr": ["amplitude_mix"],                     # RoadLens v3
    # ops that move pixels; they are related to the affine / warp parts of elastic_transform
    "geometric": ["rotate", "shear_x", "shear_y", "translate_x", "translate_y", "spatial_warp"],
}


def resolve_ops(spec, exclude=None) -> list[str]:
    """Expand a list of op names / set names (or one name) into op names, minus `exclude`."""
    spec = [spec] if isinstance(spec, str) else list(spec)

    def expand(items):
        out = []
        for item in items:
            names = OP_SETS.get(item, [item])
            for name in names:
                if name not in OPS:
                    raise KeyError(f"unknown augmentation op or set: {name!r}")
                if name not in out:
                    out.append(name)
        return out

    drop = set(expand(exclude or []))
    ops = [name for name in expand(spec) if name not in drop]
    if not ops:
        raise ValueError("augmentation op set is empty")
    return ops


In [ ]:
%%writefile /content/RoadLens/roadlens/data/__init__.py



In [ ]:
%%writefile /content/RoadLens/roadlens/data/classes.py
"""GTSRB class names (index = official ClassId)."""

GTSRB_CLASSES = [
    "Speed limit 20",
    "Speed limit 30",
    "Speed limit 50",
    "Speed limit 60",
    "Speed limit 70",
    "Speed limit 80",
    "End of speed limit 80",
    "Speed limit 100",
    "Speed limit 120",
    "No passing",
    "No passing (trucks)",
    "Right-of-way at next intersection",
    "Priority road",
    "Yield",
    "Stop",
    "No vehicles",
    "No trucks",
    "No entry",
    "General caution",
    "Dangerous curve left",
    "Dangerous curve right",
    "Double curve",
    "Bumpy road",
    "Slippery road",
    "Road narrows on the right",
    "Road work",
    "Traffic signals",
    "Pedestrians",
    "Children crossing",
    "Bicycles crossing",
    "Beware of ice/snow",
    "Wild animals crossing",
    "End of all limits",
    "Turn right ahead",
    "Turn left ahead",
    "Ahead only",
    "Go straight or right",
    "Go straight or left",
    "Keep right",
    "Keep left",
    "Roundabout mandatory",
    "End of no passing",
    "End of no passing (trucks)",
]

# Short labels for confusion-matrix axes.
GTSRB_SHORT = [
    "20", "30", "50", "60", "70", "80", "End 80", "100", "120", "No pass",
    "No pass trk", "Right-of-way", "Priority", "Yield", "Stop", "No vehicles",
    "No trucks", "No entry", "Caution", "Curve L", "Curve R", "Double curve",
    "Bumpy", "Slippery", "Narrows R", "Road work", "Signals", "Pedestrians",
    "Children", "Bicycles", "Ice/snow", "Animals", "End limits", "Turn R",
    "Turn L", "Ahead", "Ahead/R", "Ahead/L", "Keep R", "Keep L", "Roundabout",
    "End no pass", "End no pass trk",
]

assert len(GTSRB_CLASSES) == len(GTSRB_SHORT) == 43


In [ ]:
%%writefile /content/RoadLens/roadlens/data/corrupted.py
"""Generate once / reuse: GTSRB-C (test corruptions x severities) and the small validation-C."""

from __future__ import annotations

import hashlib
import json
import os
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
from tqdm import tqdm

from ..utils import get_logger
from . import corruptions as C

log = get_logger()


def _fingerprint(x: np.ndarray, names: list[str], seed: int) -> str:
    """Hash of everything that determines the corrupted pixels."""
    h = hashlib.sha1()
    h.update(np.ascontiguousarray(x).tobytes())
    spec = {"seed": seed, "shape": list(x.shape), "frost_scale": C.FROST_TEXTURE_SCALE,
            "params": {n: C.PARAMS[n] for n in sorted(names)}}
    h.update(json.dumps(spec, sort_keys=True, default=str).encode())
    return h.hexdigest()


def _work(args):
    x, name, severity, seed, start = args
    import cv2
    cv2.setNumThreads(0)
    return start, C.corrupt_batch(x, name, severity, seed, start)


def _save_npy(path: Path, arr: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp.npy")
    np.save(tmp, arr)
    os.replace(tmp, path)


def gtsrb_c_dir(data_root: Path, size: int) -> Path:
    return data_root / f"gtsrb_c_s{size}"


def generate_corrupted_sets(x: np.ndarray, y: np.ndarray, out_dir: Path, names: list[str],
                            severities: list[int], seed: int, workers: int = 0,
                            chunk: int = 256, overwrite: bool = False) -> None:
    """Write out_dir/<corruption>/<severity>.npy (uint8 NHWC) + labels.npy + manifest.json.

    Resumable: finished files are skipped. Refuses to mix files generated from different
    data / seed / parameters unless `overwrite`.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    size = x.shape[1]
    if not 0.75 * C.REFERENCE_SIZE <= size <= 1.5 * C.REFERENCE_SIZE:
        log.warning(f"Corruption constants are calibrated for ~{C.REFERENCE_SIZE}px images; at {size}px "
                    f"blur/weather severities will not be comparable. See CORRUPTIONS.md.")
    fp = _fingerprint(x, names, seed)
    manifest_path = out_dir / "manifest.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
    if manifest and manifest["fingerprint"] != fp:
        if not overwrite:
            raise RuntimeError(f"{out_dir} was generated with different data/seed/parameters. "
                               f"Re-run with --overwrite or delete the directory.")
        manifest = None
    if manifest is None:
        manifest = {"fingerprint": fp, "seed": seed, "n": int(len(x)), "image_size": int(size),
                    "params": {n: C.PARAMS[n] for n in names}, "done": []}
        for p in out_dir.glob("*/*.npy"):
            p.unlink()
    np.save(out_dir / "labels.npy", y)

    jobs = [(n, s) for n in names for s in severities if f"{n}/{s}" not in manifest["done"]]
    if not jobs:
        log.info(f"{out_dir}: all {len(names) * len(severities)} sets already present.")
        return
    workers = workers or min(os.cpu_count() or 1, 8)
    starts = list(range(0, len(x), chunk))
    pool = ProcessPoolExecutor(workers) if workers > 1 else None
    try:
        with tqdm(total=len(jobs) * len(x), unit="img", desc="corrupting") as bar:
            for name, sev in jobs:
                bar.set_postfix_str(f"{name} s{sev}")
                tasks = [(x[s:s + chunk], name, sev, seed, s) for s in starts]
                out = np.empty_like(x)
                for start, res in (pool.map(_work, tasks) if pool else map(_work, tasks)):
                    out[start:start + len(res)] = res
                    bar.update(len(res))
                _save_npy(out_dir / name / f"{sev}.npy", out)
                manifest["done"].append(f"{name}/{sev}")
                manifest_path.write_text(json.dumps(manifest, indent=1, default=str))
    finally:
        if pool:
            pool.shutdown()


def load_corrupted(out_dir: Path, name: str, severity: int, mmap: bool = True):
    """Return (x uint8 NHWC, y) of one pre-generated corrupted set."""
    path = out_dir / name / f"{severity}.npy"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run: python scripts/make_gtsrb_c.py")
    return np.load(path, mmap_mode="r" if mmap else None), np.load(out_dir / "labels.npy")


def val_c_path(data_root: Path, size: int) -> Path:
    return data_root / f"gtsrb_val_c_s{size}.npz"


def generate_val_c(x: np.ndarray, y: np.ndarray, path: Path, names: list[str],
                   severities: list[int], seed: int, overwrite: bool = False) -> None:
    """Validation-C: every validation image gets exactly one (held-out corruption, severity),
    balanced over all combinations. Same size as the clean validation split."""
    fp = _fingerprint(x, names, seed) + ":" + ",".join(map(str, severities))
    if path.exists() and not overwrite:
        with np.load(path) as d:
            if str(d["fingerprint"]) == fp:
                log.info(f"{path}: already present.")
                return
        raise RuntimeError(f"{path} was generated with different data/seed/parameters; use --overwrite.")
    combos = [(n, s) for n in names for s in severities]
    order = np.random.RandomState(seed).permutation(len(x))
    cid = np.empty(len(x), dtype=np.int64)
    sev = np.empty(len(x), dtype=np.int64)
    out = np.empty_like(x)
    for k, i in enumerate(tqdm(order, desc="validation-C")):
        name, s = combos[k % len(combos)]
        out[i] = C.corrupt(x[i], name, s, C.derive_seed(seed, name, s, int(i)))
        cid[i], sev[i] = C.CORRUPTION_IDS[name], s
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez(tmp, x=out, y=y, corruption_id=cid, severity=sev, fingerprint=fp)
    os.replace(tmp, path)
    log.info(f"Saved {path} {out.shape}")


def load_val_c(data_root: Path, size: int):
    path = val_c_path(data_root, size)
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run: python scripts/make_gtsrb_c.py")
    with np.load(path) as d:
        return d["x"], d["y"], d["corruption_id"], d["severity"]


In [ ]:
%%writefile /content/RoadLens/roadlens/data/corruptions.py
"""Image corruptions calibrated for ~64 px traffic-sign crops.

The algorithms and helper kernels come from the `imagecorruptions` package (itself a port of
Hendrycks & Dietterich's ImageNet-C code). The package hard-codes severity constants tuned for
224+ px images, so every corruption here is re-parameterised; see CORRUPTIONS.md for the table
of changes and the reasons. All functions take a uint8 HxWx3 RGB array and return uint8.

Randomness goes through the global `np.random` state only, so `corrupt(..., seed=...)` is
fully reproducible.
"""

from __future__ import annotations

import os
import sys
import types
import warnings
from functools import lru_cache
from io import BytesIO

import cv2
import numpy as np
import skimage.color
from PIL import Image
from scipy.ndimage import map_coordinates
from scipy.ndimage import zoom as scizoom
from skimage.filters import gaussian


def _import_package_helpers():
    """Import imagecorruptions.corruptions, shimming modules it imports that newer stacks dropped."""
    try:
        import pkg_resources  # noqa: F401
    except Exception:  # removed from recent setuptools; the package only needs resource_filename
        shim = types.ModuleType("pkg_resources")
        shim.resource_filename = lambda pkg, rel: os.path.join(
            os.path.dirname(sys.modules[pkg].__file__), rel)
        sys.modules["pkg_resources"] = shim
    try:
        import scipy.ndimage.interpolation  # noqa: F401
    except Exception:
        shim = types.ModuleType("scipy.ndimage.interpolation")
        shim.map_coordinates = map_coordinates
        sys.modules["scipy.ndimage.interpolation"] = shim
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        import imagecorruptions.corruptions as pkg
    return pkg


_pkg = _import_package_helpers()
_FROST_DIR = os.path.join(os.path.dirname(_pkg.__file__), "frost")
_FROST_FILES = ["frost1.png", "frost2.png", "frost3.png", "frost4.jpg", "frost5.jpg", "frost6.jpg"]

# Resolution the constants below were calibrated for.
REFERENCE_SIZE = 64

# --------------------------------------------------------------------------------------
# Severity constants (index = severity - 1). See CORRUPTIONS.md.
# --------------------------------------------------------------------------------------
PARAMS = {
    # noise: std of additive noise on [0, 1] / photon count / fraction of flipped values
    "gaussian_noise": [0.08, 0.12, 0.18, 0.26, 0.38],
    "shot_noise": [60, 25, 12, 5, 3],
    "impulse_noise": [0.03, 0.06, 0.09, 0.17, 0.27],
    # blur: (disk radius px, anti-alias sigma) / (kernel radius, sigma) px / zoom factors
    "defocus_blur": [(1, 0.1), (1.5, 0.1), (2, 0.1), (3, 0.5), (4, 0.5)],
    "motion_blur": [(4, 1.2), (6, 2), (6, 3.2), (6, 4.8), (8, 6)],
    "zoom_blur": [(1.16, 0.02), (1.24, 0.02), (1.32, 0.03), (1.40, 0.03), (1.50, 0.04)],
    # weather
    "snow": [(0.1, 0.2, 1, 0.6, 8, 3, 0.8), (0.1, 0.2, 1, 0.5, 10, 4, 0.8),
             (0.15, 0.3, 1.75, 0.55, 10, 4, 0.7), (0.25, 0.3, 2.25, 0.6, 12, 6, 0.65),
             (0.3, 0.3, 1.25, 0.65, 14, 12, 0.6)],
    "fog": [(0.7, 2.5), (1.1, 2.2), (1.5, 2), (2.2, 1.7), (3, 1.4)],
    "frost": [(1, 0.35), (0.9, 0.5), (0.8, 0.6), (0.7, 0.7), (0.6, 0.8)],
    "brightness": [0.1, 0.2, 0.3, 0.4, 0.5],
    # digital
    "contrast": [0.4, 0.3, 0.2, 0.1, 0.05],
    "pixelate": [0.65, 0.52, 0.43, 0.36, 0.3],
    # (RMS warp displacement, warp smoothness sigma, affine jitter), each as a fraction of the image side.
    # RoadLens ladder (not Tiny-ImageNet-C): 0.75 -> 4 px average warp at 64 px; see CORRUPTIONS.md.
    "elastic_transform": [(0.012, 0.0625, 0.02), (0.023, 0.0625, 0.03), (0.035, 0.0625, 0.04),
                          (0.047, 0.0625, 0.05), (0.0625, 0.0625, 0.06)],
    "jpeg_compression": [45, 30, 20, 12, 8],
    # held-out (validation-C only)
    "speckle_noise": [0.15, 0.2, 0.25, 0.3, 0.35],
    "gaussian_blur": [0.5, 0.75, 1, 1.25, 1.5],
    "spatter": [(0.62, 0.1, 0.7, 0.7, 0.6, 0), (0.65, 0.1, 0.8, 0.7, 0.6, 0),
                (0.65, 0.3, 1, 0.69, 0.6, 0), (0.65, 0.1, 0.7, 0.68, 0.6, 1),
                (0.65, 0.1, 0.5, 0.67, 0.6, 1)],
    "saturate": [(0.3, 0), (0.1, 0), (2, 0), (5, 0.1), (30, 0.2)],
}

# Fraction by which frost textures are shrunk before cropping, at REFERENCE_SIZE.
FROST_TEXTURE_SCALE = 0.3

# Stable ids used for seeding and for validation-C metadata. Append only; never reorder.
CORRUPTION_IDS = {name: i for i, name in enumerate(PARAMS)}


# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------
def _clipped_zoom(img: np.ndarray, zoom_factor: float) -> np.ndarray:
    """Centre-crop then zoom back to (at least) the original size, trimmed symmetrically."""
    h, w = img.shape[:2]
    ch, cw = int(np.ceil(h / zoom_factor)), int(np.ceil(w / zoom_factor))
    top, left = (h - ch) // 2, (w - cw) // 2
    out = scizoom(img[top:top + ch, left:left + cw], (zoom_factor, zoom_factor, 1), order=1)
    tt, tl = (out.shape[0] - h) // 2, (out.shape[1] - w) // 2
    return out[tt:tt + h, tl:tl + w]


def _plasma_fractal(mapsize: int, wibbledecay: float) -> np.ndarray:
    """Diamond-square heightmap in [0, 1] (package version uses the removed `np.float_`)."""
    assert mapsize & (mapsize - 1) == 0
    maparray = np.empty((mapsize, mapsize), dtype=np.float64)
    maparray[0, 0] = 0
    stepsize = mapsize
    wibble = 100.0

    def wibbledmean(array):
        return array / 4 + wibble * np.random.uniform(-wibble, wibble, array.shape)

    while stepsize >= 2:
        half = stepsize // 2
        # squares
        cornerref = maparray[0:mapsize:stepsize, 0:mapsize:stepsize]
        squareaccum = cornerref + np.roll(cornerref, shift=-1, axis=0)
        squareaccum += np.roll(squareaccum, shift=-1, axis=1)
        maparray[half:mapsize:stepsize, half:mapsize:stepsize] = wibbledmean(squareaccum)
        # diamonds
        drgrid = maparray[half:mapsize:stepsize, half:mapsize:stepsize]
        ulgrid = maparray[0:mapsize:stepsize, 0:mapsize:stepsize]
        ldrsum = drgrid + np.roll(drgrid, 1, axis=0)
        lulsum = ulgrid + np.roll(ulgrid, -1, axis=1)
        maparray[0:mapsize:stepsize, half:mapsize:stepsize] = wibbledmean(ldrsum + lulsum)
        tdrsum = drgrid + np.roll(drgrid, 1, axis=1)
        tulsum = ulgrid + np.roll(ulgrid, -1, axis=0)
        maparray[half:mapsize:stepsize, 0:mapsize:stepsize] = wibbledmean(tdrsum + tulsum)
        stepsize //= 2
        wibble /= wibbledecay

    maparray -= maparray.min()
    return maparray / maparray.max()


@lru_cache(maxsize=None)
def _frost_texture(idx: int, size: int) -> np.ndarray:
    """Frost texture (RGB) shrunk so that a size x size crop shows crystal structure, not one blob."""
    tex = cv2.imread(os.path.join(_FROST_DIR, _FROST_FILES[idx]))[..., ::-1]
    scale = FROST_TEXTURE_SCALE * size / REFERENCE_SIZE
    scale = max(scale, (size + 1) / min(tex.shape[:2]))
    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    return np.ascontiguousarray(cv2.resize(tex, (0, 0), fx=scale, fy=scale, interpolation=interp))


def _f(x: np.ndarray) -> np.ndarray:
    return x.astype(np.float32) / 255.0


# --------------------------------------------------------------------------------------
# Corruptions: f(x uint8 HxWx3, c) -> float array in [0, 255] (clipped/cast by `corrupt`)
# --------------------------------------------------------------------------------------
def gaussian_noise(x, c):
    x = _f(x)
    return (x + np.random.normal(size=x.shape, scale=c)) * 255


def shot_noise(x, c):
    return np.random.poisson(_f(x) * c) / float(c) * 255


def impulse_noise(x, c):
    # Salt & pepper on a fraction c of values. Written out by hand because
    # skimage.util.random_noise draws from its own generator and ignores np.random.seed.
    x = _f(x)
    flip = np.random.uniform(size=x.shape) < c
    salt = np.random.uniform(size=x.shape) < 0.5
    x[flip & salt] = 1.0
    x[flip & ~salt] = 0.0
    return x * 255


def speckle_noise(x, c):
    x = _f(x)
    return (x + x * np.random.normal(size=x.shape, scale=c)) * 255


def gaussian_blur(x, c):
    return gaussian(_f(x), sigma=c, channel_axis=-1) * 255


def defocus_blur(x, c):
    kernel = _pkg.disk(radius=c[0], alias_blur=c[1])
    return cv2.filter2D(_f(x), -1, kernel) * 255


def motion_blur(x, c):
    angle = np.random.uniform(-45, 45)
    return _pkg._motion_blur(x.astype(np.float32), radius=c[0], sigma=c[1], angle=angle)


def zoom_blur(x, c):
    factors = np.arange(1, c[0], c[1])
    x = _f(x)
    out = np.zeros_like(x)
    for z in factors:
        out += _clipped_zoom(x, z)
    return (x + out) / (len(factors) + 1) * 255


def snow(x, c):
    x = _f(x)
    layer = np.random.normal(size=x.shape[:2], loc=c[0], scale=c[1])
    layer = _clipped_zoom(layer[..., np.newaxis], c[2])
    layer[layer < c[3]] = 0
    layer = np.clip(layer.squeeze(-1), 0, 1)
    layer = _pkg._motion_blur(layer, radius=c[4], sigma=c[5], angle=np.random.uniform(-135, -45))
    layer = (np.round(layer * 255).astype(np.uint8) / 255.0)[..., np.newaxis]
    gray = cv2.cvtColor(x, cv2.COLOR_RGB2GRAY)[..., np.newaxis]
    x = c[6] * x + (1 - c[6]) * np.maximum(x, gray * 1.5 + 0.5)
    return (x + layer + np.rot90(layer, k=2)) * 255


def fog(x, c):
    x = _f(x)
    h, w = x.shape[:2]
    mapsize = 1 << (max(h, w) - 1).bit_length()
    max_val = x.max()
    x = x + c[0] * _plasma_fractal(mapsize, c[1])[:h, :w, np.newaxis]
    return x * max_val / (max_val + c[0]) * 255


def frost(x, c):
    h, w = x.shape[:2]
    idx = np.random.randint(5)  # upstream draws from the first five of six textures
    tex = _frost_texture(idx, max(h, w))
    top = np.random.randint(0, tex.shape[0] - h)
    left = np.random.randint(0, tex.shape[1] - w)
    return c[0] * x.astype(np.float32) + c[1] * tex[top:top + h, left:left + w]


def brightness(x, c):
    x = skimage.color.rgb2hsv(_f(x))
    x[:, :, 2] = np.clip(x[:, :, 2] + c, 0, 1)
    return skimage.color.hsv2rgb(x) * 255


def contrast(x, c):
    x = _f(x)
    means = np.mean(x, axis=(0, 1), keepdims=True)
    return ((x - means) * c + means) * 255


def saturate(x, c):
    x = skimage.color.rgb2hsv(_f(x))
    x[:, :, 1] = np.clip(x[:, :, 1] * c[0] + c[1], 0, 1)
    return skimage.color.hsv2rgb(x) * 255


def jpeg_compression(x, c):
    buf = BytesIO()
    Image.fromarray(x).save(buf, "JPEG", quality=int(c))
    return np.asarray(Image.open(buf).convert("RGB"))


def pixelate(x, c):
    h, w = x.shape[:2]
    im = Image.fromarray(x).resize((int(w * c), int(h * c)), Image.BOX)
    return np.asarray(im.resize((w, h), Image.BOX))


def elastic_transform(x, c):
    image = _f(x)
    shape = image.shape
    h, w = shape[:2]
    side = min(h, w)
    disp, sigma, affine = c[0] * side, c[1] * side, c[2] * side

    # random affine
    center = np.float32([h, w]) // 2
    sq = side // 3
    pts1 = np.float32([center + sq, [center[0] + sq, center[1] - sq], center - sq])
    pts2 = pts1 + np.random.uniform(-affine, affine, size=pts1.shape).astype(np.float32)
    M = cv2.getAffineTransform(pts1, pts2)
    image = cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)

    # smooth random displacement field, rescaled so that its RMS length is exactly `disp` pixels.
    # (Tiny-ImageNet-C multiplies the smoothed noise by a fixed amplitude instead; the smoothing shrinks
    # it so much that its average warp at 64 px is only 0-1.2 px, which barely changes a sign.)
    rx = np.random.uniform(-1, 1, size=(h, w))
    ry = np.random.uniform(-1, 1, size=(h, w))
    if disp > 0:
        dx = gaussian(rx, sigma, mode="reflect", truncate=3)
        dy = gaussian(ry, sigma, mode="reflect", truncate=3)
        scale = disp / max(float(np.sqrt(np.mean(dx ** 2 + dy ** 2))), 1e-8)
        dx, dy = (dx * scale).astype(np.float32), (dy * scale).astype(np.float32)
        yy, xx, zz = np.meshgrid(np.arange(h), np.arange(w), np.arange(shape[2]), indexing="ij")
        idx = (yy + dy[..., None]).reshape(-1, 1), (xx + dx[..., None]).reshape(-1, 1), zz.reshape(-1, 1)
        image = map_coordinates(image, idx, order=1, mode="reflect").reshape(shape)
    return image * 255


def spatter(x, c):
    x = _f(x)
    liquid = np.random.normal(size=x.shape[:2], loc=c[0], scale=c[1])
    liquid = gaussian(liquid, sigma=c[2])
    liquid[liquid < c[3]] = 0
    if c[5] == 0:  # water
        liquid = (liquid * 255).astype(np.uint8)
        dist = 255 - cv2.Canny(liquid, 50, 150)
        dist = cv2.distanceTransform(dist, cv2.DIST_L2, 5)
        _, dist = cv2.threshold(dist, 20, 20, cv2.THRESH_TRUNC)
        dist = cv2.blur(dist, (3, 3)).astype(np.uint8)
        dist = cv2.equalizeHist(dist)
        ker = np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]])
        dist = cv2.filter2D(dist, cv2.CV_8U, ker)
        dist = cv2.blur(dist, (3, 3)).astype(np.float32)
        m = (liquid * dist)[..., np.newaxis].astype(np.float32)
        m = m / max(float(m.max()), 1e-8) * c[4]
        color = np.array([175, 238, 238], dtype=np.float32) / 255.0  # pale turquoise, RGB
        return (x + m * color) * 255
    # mud
    m = np.where(liquid > c[3], 1, 0)
    m = gaussian(m.astype(np.float32), sigma=c[4])
    m[m < 0.8] = 0
    color = np.array([63, 42, 20], dtype=np.float32) / 255.0
    return (x * (1 - m[..., np.newaxis]) + color * m[..., np.newaxis]) * 255


CORRUPTIONS = {
    "gaussian_noise": gaussian_noise, "shot_noise": shot_noise, "impulse_noise": impulse_noise,
    "defocus_blur": defocus_blur, "motion_blur": motion_blur, "zoom_blur": zoom_blur,
    "snow": snow, "fog": fog, "frost": frost, "brightness": brightness,
    "contrast": contrast, "pixelate": pixelate, "elastic_transform": elastic_transform,
    "jpeg_compression": jpeg_compression,
    "speckle_noise": speckle_noise, "gaussian_blur": gaussian_blur, "spatter": spatter,
    "saturate": saturate,
}
assert set(CORRUPTIONS) == set(PARAMS)


# --------------------------------------------------------------------------------------
# Public API
# --------------------------------------------------------------------------------------
def derive_seed(base_seed: int, name: str, severity: int, index: int) -> int:
    """Per-image seed, independent of processing order and of which other corruptions are generated."""
    ss = np.random.SeedSequence([int(base_seed), CORRUPTION_IDS[name], int(severity), int(index)])
    return int(ss.generate_state(1)[0])


def corrupt(image: np.ndarray, name: str, severity: int, seed: int | None = None) -> np.ndarray:
    """Corrupt one uint8 HxWx3 RGB image. With `seed`, the result is deterministic and the global
    numpy random state is left untouched."""
    if image.dtype != np.uint8 or image.ndim != 3 or image.shape[2] != 3:
        raise ValueError(f"expected uint8 HxWx3, got {image.dtype} {image.shape}")
    if severity not in (1, 2, 3, 4, 5):
        raise ValueError("severity must be in 1..5")
    c = PARAMS[name][severity - 1]
    if seed is None:
        out = CORRUPTIONS[name](image, c)
    else:
        state = np.random.get_state()
        np.random.seed(seed)
        try:
            out = CORRUPTIONS[name](image, c)
        finally:
            np.random.set_state(state)
    return np.clip(out, 0, 255).astype(np.uint8)


def corrupt_batch(x: np.ndarray, name: str, severity: int, base_seed: int,
                  start_index: int = 0) -> np.ndarray:
    """Corrupt x[i] with seed derive_seed(base_seed, name, severity, start_index + i)."""
    out = np.empty_like(x)
    for i in range(len(x)):
        out[i] = corrupt(x[i], name, severity, derive_seed(base_seed, name, severity, start_index + i))
    return out


def package_default(image: np.ndarray, name: str, severity: int, seed: int | None = None) -> np.ndarray:
    """The unmodified `imagecorruptions` corruption (ImageNet-scale constants), for comparison only.

    Patches the two things that crash on current numpy/scikit-image.
    """
    _pkg.plasma_fractal = lambda mapsize=256, wibbledecay=3: _plasma_fractal(mapsize, wibbledecay)

    def _gaussian(img, sigma=1, multichannel=None, **kw):
        if multichannel:
            kw["channel_axis"] = -1
        return gaussian(img, sigma=sigma, **kw)

    _pkg.gaussian = _gaussian
    if seed is not None:
        np.random.seed(seed)
    out = getattr(_pkg, name)(Image.fromarray(image), severity)
    return np.clip(np.asarray(out), 0, 255).astype(np.uint8)


In [ ]:
%%writefile /content/RoadLens/roadlens/data/external.py
"""Cross-dataset test sets: BTSD (BelgiumTSC) and CTSD (Chinese traffic signs, TSRD release).

Only classes that also exist in GTSRB are evaluated; the class maps live in
roadlens/data/mappings/<name>_to_gtsrb.yaml.
"""

from __future__ import annotations

import re
import zipfile
from pathlib import Path

import numpy as np
import yaml

from ..utils import get_logger
from .gtsrb import download, load_resized, read_gt_csv, roi_of

log = get_logger()

MAPPING_DIR = Path(__file__).parent / "mappings"
BTSD_URL = "https://btsd.ethz.ch/shareddata/BelgiumTSC/"
BTSD_FILES = {"Training": "BelgiumTSC_Training.zip", "Testing": "BelgiumTSC_Testing.zip"}
IMG_EXT = {".ppm", ".png", ".jpg", ".jpeg", ".bmp"}

CTSD_HELP = """CTSD needs a manual download (no reliable scripted source).
Get the Chinese Traffic Sign Recognition Database (TSRD, 58 classes) from either
  - http://www.nlpr.ia.ac.cn/pal/trafficdata/recognition.html  (tsrd-train.zip, TSRD-Test.zip + annotation .txt)
  - https://www.kaggle.com/datasets/dmitryyemelyanov/chinese-traffic-signs  (images/ + annotations.csv)
and unzip it anywhere under: {dst}
File names must keep their original form <class:3 digits>_<...>.png."""


def _cache_path(data_root: Path, name: str, size: int, crop_roi: bool) -> Path:
    return data_root / "processed" / f"{name}_s{size}{'_roi' if crop_roi else ''}.npz"


def _index_btsd(zf: zipfile.ZipFile):
    """Members look like Testing/00017/01234_00001.ppm with a GT-00017.csv per class folder."""
    names = zf.namelist()
    rois = {}
    for gt in (n for n in names if n.endswith(".csv")):
        cdir = gt.rsplit("/", 1)[0]
        for r in read_gt_csv(zf, gt):
            rois[f"{cdir}/{r['Filename']}"] = roi_of(r)
    items, labels = [], []
    for n in sorted(names):
        parts = n.split("/")
        if len(parts) >= 2 and parts[-2].isdigit() and Path(n).suffix.lower() in IMG_EXT:
            items.append((n, rois.get(n)))
            labels.append(int(parts[-2]))
    return items, labels


def _index_ctsd(raw: Path):
    """Layout-agnostic: class id = first three digits of the file name; ROIs from any annotation file."""
    rois = {}
    for ann in list(raw.rglob("*.txt")) + list(raw.rglob("*.csv")):
        for line in ann.read_text(errors="ignore").splitlines():
            parts = [p for p in re.split(r"[;,]", line.strip()) if p != ""]
            if len(parts) >= 8 and parts[1].isdigit():
                rois[parts[0]] = tuple(int(v) for v in parts[3:7])
    items, labels = [], []
    for p in sorted(raw.rglob("*")):
        m = re.match(r"^(\d{3})_", p.name)
        if p.suffix.lower() in IMG_EXT and m:
            items.append((p, rois.get(p.name)))
            labels.append(int(m.group(1)))
    return items, labels


def prepare_external(data_root: Path, name: str, size: int = 64, crop_roi: bool = False) -> None:
    """Cache every image of the dataset at the working resolution with its SOURCE class id."""
    path = _cache_path(data_root, name, size, crop_roi)
    if path.exists():
        return
    raw = data_root / "raw" / name
    if name == "btsd":
        xs, labels = [], []
        for split, fname in BTSD_FILES.items():
            download(BTSD_URL + fname, raw / fname)
            with zipfile.ZipFile(raw / fname) as zf:
                items, lab = _index_btsd(zf)
                if not crop_roi:
                    items = [(p, None) for p, _ in items]
                xs.append(load_resized(items, size, desc=f"btsd {split} -> {size}px", zf=zf))
                labels += lab
        x = np.concatenate(xs)
    elif name == "ctsd":
        raw.mkdir(parents=True, exist_ok=True)
        items, labels = _index_ctsd(raw)
        if not items:
            raise FileNotFoundError(CTSD_HELP.format(dst=raw))
        if not crop_roi:
            items = [(p, None) for p, _ in items]
        x = load_resized(items, size, desc=f"ctsd -> {size}px")
    else:
        raise ValueError(name)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez(tmp, x=x, y_src=np.array(labels, dtype=np.int64))
    tmp.replace(path)
    log.info(f"Saved {path} {x.shape}")


def load_mapping(name: str) -> dict:
    with open(MAPPING_DIR / f"{name}_to_gtsrb.yaml", encoding="utf-8") as f:
        return yaml.safe_load(f)


def load_external(data_root: Path, name: str, size: int = 64, crop_roi: bool = False,
                  mapped_only: bool = True, allow_unverified: bool = False):
    """Return (x uint8 NHWC, y in GTSRB ids, y_src). With mapped_only=False, unmapped images
    are kept with y = -1."""
    path = _cache_path(data_root, name, size, crop_roi)
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run: python scripts/prepare_data.py --datasets {name}")
    with np.load(path) as d:
        x, y_src = d["x"], d["y_src"]
    mapping = load_mapping(name)
    if mapped_only and not mapping.get("verified", False) and not allow_unverified:
        raise RuntimeError(
            f"The {name}->GTSRB class map is not marked verified. Inspect it with "
            f"`python scripts/check_mapping.py --dataset {name}`, fix "
            f"{MAPPING_DIR / (name + '_to_gtsrb.yaml')} and set `verified: true`.")
    lut = np.full(int(y_src.max()) + 1, -1, dtype=np.int64)
    for src, dst in mapping["map"].items():
        if int(src) < len(lut):
            lut[int(src)] = int(dst)
    y = lut[y_src]
    if mapped_only:
        keep = y >= 0
        return x[keep], y[keep], y_src[keep]
    return x, y, y_src


In [ ]:
%%writefile /content/RoadLens/roadlens/data/gtsrb.py
"""GTSRB (official 39,209 train / 12,630 test): download, resize-once cache, track-wise val split."""

from __future__ import annotations

import csv
import io
import json
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm import tqdm

from ..utils import get_logger

log = get_logger()

_BASE = "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/"
GTSRB_FILES = {
    "train": "GTSRB_Final_Training_Images.zip",
    "test": "GTSRB_Final_Test_Images.zip",
    "test_gt": "GTSRB_Final_Test_GT.zip",
}
N_TRAIN, N_TEST = 39209, 12630


def download(url: str, dst: Path) -> None:
    """Download with a progress bar; writes to `.part` first so interrupted downloads are not reused."""
    if dst.exists():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_suffix(dst.suffix + ".part")
    log.info(f"Downloading {url}")
    with urllib.request.urlopen(url) as r, open(tmp, "wb") as f:
        total = int(r.headers.get("Content-Length", 0))
        with tqdm(total=total, unit="B", unit_scale=True, desc=dst.name) as bar:
            while chunk := r.read(1 << 20):
                f.write(chunk)
                bar.update(len(chunk))
    tmp.replace(dst)


def load_resized(items: list[tuple[str, tuple[int, int, int, int] | None]], size: int,
                 desc: str = "resize", zf: zipfile.ZipFile | None = None) -> np.ndarray:
    """Read images (members of `zf`, or file paths), optionally crop to ROI (x1, y1, x2, y2) and
    resize to size x size. Returns uint8 NHWC.

    Reading straight from the archive avoids unpacking tens of thousands of small files.
    """
    out = np.empty((len(items), size, size, 3), dtype=np.uint8)
    for i, (path, roi) in enumerate(tqdm(items, desc=desc)):
        src = io.BytesIO(zf.read(path)) if zf is not None else path
        with Image.open(src) as im:
            im = im.convert("RGB")
            if roi is not None:
                im = im.crop((roi[0], roi[1], roi[2] + 1, roi[3] + 1))
            out[i] = np.asarray(im.resize((size, size), Image.BILINEAR))
    return out


def read_gt_csv(zf: zipfile.ZipFile, member: str) -> list[dict]:
    with zf.open(member) as f:
        return list(csv.DictReader(io.TextIOWrapper(f, encoding="utf-8"), delimiter=";"))


def roi_of(row: dict) -> tuple[int, int, int, int]:
    return tuple(int(row[k]) for k in ("Roi.X1", "Roi.Y1", "Roi.X2", "Roi.Y2"))


def _index_train(zf: zipfile.ZipFile):
    items, labels, tracks = [], [], []
    gts = sorted(n for n in zf.namelist() if "Final_Training/Images/" in n and n.endswith(".csv"))
    for gt in gts:
        cdir = gt.rsplit("/", 1)[0]
        c = int(cdir.rsplit("/", 1)[1])
        for row in read_gt_csv(zf, gt):
            # File names are <track>_<frame>.ppm: each track is ~30 frames of one physical sign.
            track = int(row["Filename"].split("_")[0])
            items.append((f"{cdir}/{row['Filename']}", roi_of(row)))
            labels.append(c)
            tracks.append(c * 10000 + track)
    return items, np.array(labels, dtype=np.int64), np.array(tracks, dtype=np.int64)


def _index_test(zf_gt: zipfile.ZipFile):
    rows = read_gt_csv(zf_gt, "GT-final_test.csv")
    items = [(f"GTSRB/Final_Test/Images/{r['Filename']}", roi_of(r)) for r in rows]
    labels = np.array([int(r["ClassId"]) for r in rows], dtype=np.int64)
    return items, labels


def _cache_path(data_root: Path, split: str, size: int, crop_roi: bool) -> Path:
    return data_root / "processed" / f"gtsrb_{split}_s{size}{'_roi' if crop_roi else ''}.npz"


def prepare_gtsrb(data_root: Path, size: int = 64, crop_roi: bool = False) -> None:
    """Download GTSRB and cache both official splits as uint8 arrays at the working resolution."""
    raw = data_root / "raw" / "gtsrb"
    todo = [s for s in ("train", "test") if not _cache_path(data_root, s, size, crop_roi).exists()]
    if not todo:
        return
    for name in GTSRB_FILES.values():
        download(_BASE + name, raw / name)

    for split in todo:
        zf = zipfile.ZipFile(raw / GTSRB_FILES[split])
        if split == "train":
            items, y, tracks = _index_train(zf)
            assert len(items) == N_TRAIN, f"expected {N_TRAIN} train images, found {len(items)}"
        else:
            with zipfile.ZipFile(raw / GTSRB_FILES["test_gt"]) as zf_gt:
                items, y = _index_test(zf_gt)
            tracks = np.full(len(items), -1, dtype=np.int64)
            assert len(items) == N_TEST, f"expected {N_TEST} test images, found {len(items)}"
        if not crop_roi:
            items = [(p, None) for p, _ in items]
        x = load_resized(items, size, desc=f"gtsrb {split} -> {size}px", zf=zf)
        zf.close()
        path = _cache_path(data_root, split, size, crop_roi)
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_name(path.stem + ".tmp.npz")
        np.savez(tmp, x=x, y=y, track=tracks)
        tmp.replace(path)
        log.info(f"Saved {path} {x.shape}")


def trackwise_split(y: np.ndarray, tracks: np.ndarray, val_fraction: float, seed: int):
    """Stratified split that keeps every track (frames of one physical sign) on one side.

    A random per-image split would put near-duplicate frames of the same sign in train and val,
    making validation accuracy meaningless for early stopping.
    """
    rng = np.random.RandomState(seed)
    val_mask = np.zeros(len(y), dtype=bool)
    for c in np.unique(y):
        ctracks = np.unique(tracks[y == c])
        rng.shuffle(ctracks)
        n_val = max(1, int(round(val_fraction * len(ctracks))))
        val_mask |= np.isin(tracks, ctracks[:n_val])
    return np.flatnonzero(~val_mask), np.flatnonzero(val_mask)


def load_gtsrb(data_root: Path, split: str, size: int = 64, crop_roi: bool = False,
               val_fraction: float = 0.1, split_seed: int = 0, with_tracks: bool = False):
    """Return (x uint8 NHWC, y int64) for split in {train, val, trainval, test}.

    with_tracks: also return the track id (class * 10000 + track) of every image; -1 for test images.
    """
    official = "test" if split == "test" else "train"
    path = _cache_path(data_root, official, size, crop_roi)
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run: python scripts/prepare_data.py")
    with np.load(path) as d:
        x, y, tracks = d["x"], d["y"], d["track"]
    if split in ("test", "trainval"):
        return (x, y, tracks) if with_tracks else (x, y)

    split_file = data_root / "processed" / f"gtsrb_split_seed{split_seed}_val{val_fraction}.json"
    if split_file.exists():
        idx = json.loads(split_file.read_text())
    else:
        tr, va = trackwise_split(y, tracks, val_fraction, split_seed)
        idx = {"train": tr.tolist(), "val": va.tolist()}
        split_file.write_text(json.dumps(idx))
    sel = np.asarray(idx[split])
    return (x[sel], y[sel], tracks[sel]) if with_tracks else (x[sel], y[sel])


def halve_by_track(y: np.ndarray, tracks: np.ndarray, seed: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """Indices of two halves that share no track, with the tracks of every class dealt out alternately.

    Used to fit a model on one half of the validation split and select it on the other: a random
    per-image split would put near-duplicate frames of one physical sign on both sides.
    """
    rng = np.random.RandomState(seed)
    first = np.zeros(len(y), dtype=bool)
    turn = 0
    for c in np.unique(y):
        ctracks = np.unique(tracks[y == c])
        rng.shuffle(ctracks)
        first |= np.isin(tracks, ctracks[turn % 2::2]) & (y == c)
        turn += len(ctracks)            # a class with an odd number of tracks hands the extra one over
    return np.flatnonzero(first), np.flatnonzero(~first)


def load_gtsrb_cfg(cfg: dict, data_root: Path, split: str, with_tracks: bool = False):
    d = cfg["data"]
    return load_gtsrb(data_root, split, d["image_size"], d["crop_roi"], d["val_fraction"], d["split_seed"], with_tracks)


In [ ]:
%%writefile /content/RoadLens/roadlens/data/loading.py
"""Config-level access to the cached arrays, plus the GPU batch helpers.

Images stay uint8 NHWC on disk / in RAM and become float NCHW in [0, 1] on the device. The whole
training set lives on the GPU (430 MB), so there are no DataLoader workers: shuffling and all
augmentations run batched on the device and are seeded through torch's global generator.
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import torch

from ..utils import corruption_names, data_root
from .corrupted import load_corrupted, load_val_c
from .external import load_external
from .gtsrb import halve_by_track, load_gtsrb_cfg


def _subset(cfg: dict, split: str, n_total: int, spread: bool = False) -> np.ndarray | None:
    """Indices of the reduced subset for `split`, or None for the full split.

    train: random. val and class-sorted sets (`spread`): evenly spaced, so every class is covered.
    GTSRB test (already in random order): a prefix.
    """
    n = int(cfg["data"].get(f"{split}_subset", 0) or 0)
    if not n or n >= n_total:
        return None
    if split == "train":
        return np.sort(np.random.RandomState(0).permutation(n_total)[:n])
    if split == "val" or spread:
        return np.linspace(0, n_total - 1, n).astype(np.int64)
    return np.arange(n)


def load_split(cfg: dict, split: str):
    """(x uint8 NHWC, y int64) of GTSRB train / val / test, honouring data.<split>_subset."""
    x, y = load_gtsrb_cfg(cfg, data_root(cfg), split)
    idx = _subset(cfg, split, len(x))
    return (x, y) if idx is None else (x[idx], y[idx])


def load_val_c_split(cfg: dict):
    """(x, y, corruption_id, severity) of validation-C, aligned with load_split(cfg, 'val')."""
    x, y, cid, sev = load_val_c(data_root(cfg), cfg["data"]["image_size"])
    idx = _subset(cfg, "val", len(x))
    return (x, y, cid, sev) if idx is None else (x[idx], y[idx], cid[idx], sev[idx])


def stack_split(cfg: dict) -> tuple[np.ndarray, np.ndarray]:
    """Indices (fit, select) into load_split(cfg, 'val'): two halves that share no physical sign.

    The fit half is for training a combiner on images its base models never saw (stacked
    generalisation); the select half is for choosing and calibrating it. The split depends only on
    the data settings, so every run and every evaluation sees the same two halves.
    """
    _, y, tracks = load_gtsrb_cfg(cfg, data_root(cfg), "val", with_tracks=True)
    idx = _subset(cfg, "val", len(y))
    if idx is not None:
        y, tracks = y[idx], tracks[idx]
    return halve_by_track(y, tracks, seed=int(cfg["data"]["split_seed"]))


def load_stack_data(cfg: dict):
    """(x fit, y fit, x select, y select, x select-C, y select-C) for a run with train.fit_split = val.

    Fit set: the clean fit half plus its validation-C copies. Select sets: the clean select half and
    its validation-C copies. No test image and no test corruption is involved.
    """
    x, y = load_split(cfg, "val")
    xc, yc, _, _ = load_val_c_split(cfg)
    fit, sel = stack_split(cfg)
    return (np.concatenate([x[fit], xc[fit]]), np.concatenate([y[fit], yc[fit]]),
            x[sel], y[sel], xc[sel], yc[sel])


def load_selection_data(cfg: dict):
    """(x, y, x-C, y-C): the select half of the validation split, clean and as validation-C."""
    return load_stack_data(cfg)[2:]


def load_test(cfg: dict, dataset: str = "gtsrb"):
    """Clean test set of `dataset` (labels in GTSRB ids), honouring data.test_subset."""
    if dataset == "gtsrb":
        return load_split(cfg, "test")
    d = cfg["data"]
    x, y, _ = load_external(data_root(cfg), dataset, d["image_size"], d["crop_roi"])
    idx = _subset(cfg, "test", len(x), spread=True)
    return (x, y) if idx is None else (x[idx], y[idx])


def corrupted_dir(cfg: dict, dataset: str = "gtsrb") -> Path:
    """Directory of the pre-generated corrupted test sets of `dataset`."""
    d = cfg["data"]
    name = f"{dataset}_c_s{d['image_size']}"
    n = int(d.get("test_subset", 0) or 0)
    return data_root(cfg) / (name + (f"_sub{n}" if n else ""))


def load_corrupted_set(cfg: dict, dataset: str, corruption: str, severity: int):
    """(x uint8 NHWC memory-mapped, y) of one corrupted test set."""
    return load_corrupted(corrupted_dir(cfg, dataset), corruption, severity)


def u8_to_device(x: np.ndarray, device) -> torch.Tensor:
    """uint8 NHWC numpy (possibly memory-mapped) -> uint8 NCHW tensor on `device`."""
    x = np.ascontiguousarray(x)
    if not x.flags.writeable:          # read-only memory maps: torch wants a writable buffer
        x = x.copy()
    return torch.from_numpy(x).to(device).permute(0, 3, 1, 2).contiguous()


def corrupted_sets_missing(cfg: dict, dataset: str = "gtsrb") -> list[str]:
    """Corrupted test sets of `dataset` ('<corruption>/<severity>') that have not been generated yet."""
    needed = [f"{n}/{s}" for n in corruption_names(cfg) for s in cfg["corruptions"]["severities"]]
    manifest = corrupted_dir(cfg, dataset) / "manifest.json"
    done = set(json.loads(manifest.read_text())["done"]) if manifest.exists() else set()
    return [k for k in needed if k not in done]


def corrupted_fingerprint(cfg: dict, dataset: str = "gtsrb") -> str:
    """Fingerprint of the generated corrupted sets (clean data + seed + corruption parameters)."""
    manifest = corrupted_dir(cfg, dataset) / "manifest.json"
    return json.loads(manifest.read_text())["fingerprint"] if manifest.exists() else ""


def make_c_command(cfg: dict, dataset: str = "gtsrb") -> str:
    return ("python scripts/make_gtsrb_c.py" + ("" if dataset == "gtsrb" else f" --dataset {dataset}")
            + (f" --profile {cfg['profile']}" if cfg.get("profile") else ""))


def external_unavailable(cfg: dict, dataset: str) -> str | None:
    """Why a cross-dataset test set cannot be evaluated (None if it is ready)."""
    from .external import _cache_path, load_mapping

    d = cfg["data"]
    if not _cache_path(data_root(cfg), dataset, d["image_size"], d["crop_roi"]).exists():
        return f"not prepared (python scripts/prepare_data.py --datasets {dataset})"
    if not load_mapping(dataset).get("verified", False):
        return (f"its class map is not verified (python scripts/check_mapping.py --dataset {dataset}, "
                f"then set verified: true in roadlens/data/mappings/{dataset}_to_gtsrb.yaml)")
    missing = corrupted_sets_missing(cfg, dataset)
    if missing:
        return f"{len(missing)} corrupted sets are missing or still being generated ({make_c_command(cfg, dataset)})"
    return None


def as_float(x_u8: torch.Tensor) -> torch.Tensor:
    """uint8 NCHW -> float32 in [0, 1]."""
    return x_u8.float().div_(255.0)


In [ ]:
%%writefile /content/RoadLens/roadlens/engine.py
"""Training loop.

AdamW + linear warm-up + cosine decay, label smoothing, mixed precision, AugMix/JSD views,
early stopping on validation-C accuracy, CSV log (optional Weights & Biases), checkpoint every
epoch and exact resume (optimizer, scheduler, grad scaler and all random generators are restored).

RoadLens v3 options (all off unless a run config sets them, so older runs are unaffected):
    train.ema              decay of an exponential moving average of the weights (e.g. 0.999). The
                           average is what is validated, selected and saved as best.pt.
    train.distill          {teacher: <run>, weight: w, temperature: T}: add w * T^2 * KL(teacher || student)
                           on every view of every batch (the teacher sees exactly the same augmented
                           images: "consistent" distillation, Beyer et al. 2022).
    train.channels_last    NHWC memory layout for convolutions (faster tensor-core kernels).
    train.cudnn_benchmark  let cuDNN pick the fastest kernels (gives up bitwise reproducibility).
    train.fused_optimizer  the fused CUDA AdamW kernel (one launch for all parameters).
    train.compile          torch.compile the training forward pass.
"""

from __future__ import annotations

import copy
import csv
import json
import math
import os
import random
import shutil
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

from .augment import TrainAugment, jsd_loss
from .data.loading import as_float, load_split, load_stack_data, load_val_c_split, u8_to_device
from .utils import get_logger, seed_everything

log = get_logger()
LOG_FIELDS = ["epoch", "lr", "loss", "ce", "jsd", "kd", "train_acc", "val_acc", "val_c_acc", "epoch_s", "img_per_s"]


def param_groups(model, weight_decay: float) -> list[dict]:
    """No weight decay on biases, normalisation parameters and position / class tokens."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (no_decay if p.ndim <= 1 or "pos_embed" in name or "cls_token" in name else decay).append(p)
    return [{"params": decay, "weight_decay": weight_decay}, {"params": no_decay, "weight_decay": 0.0}]


def lr_factor(step: int, total: int, warmup: int) -> float:
    if step < warmup:
        return (step + 1) / warmup
    return 0.5 * (1.0 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))


@torch.no_grad()
def accuracy(model, x_u8: torch.Tensor, y: torch.Tensor, batch_size: int, amp: bool) -> float:
    model.eval()
    correct = 0
    for s in range(0, len(x_u8), batch_size):
        with torch.autocast(x_u8.device.type, enabled=amp):
            pred = model(as_float(x_u8[s:s + batch_size])).argmax(dim=1)
        correct += int((pred == y[s:s + batch_size]).sum())
    return correct / len(x_u8)


def kd_loss(student_logits: torch.Tensor, teacher_logits: torch.Tensor, temperature: float) -> torch.Tensor:
    """T^2 * KL(softmax(teacher / T) || softmax(student / T)), averaged over the batch (Hinton et al. 2015)."""
    t = float(temperature)
    return F.kl_div(F.log_softmax(student_logits.float() / t, dim=1), F.log_softmax(teacher_logits.float() / t, dim=1),
                    log_target=True, reduction="batchmean") * t * t


def training_loss(model, views: list[torch.Tensor], y: torch.Tensor, jsd: bool, smoothing: float,
                  jsd_weight: float, amp: bool, ce_all_views: bool = False, teacher=None,
                  kd_weight: float = 0.0, kd_temperature: float = 1.0, stats: dict | None = None):
    """Forward the views of one batch and return (loss, cross-entropy, JSD term, logits of the first view).

    jsd=False: cross-entropy on the single view. jsd=True: cross-entropy on the clean view plus
    jsd_weight * Jensen-Shannon divergence between the clean and the two augmented views.
    ce_all_views: with jsd, average the cross-entropy over all views instead of the clean one only.
    teacher: a frozen model; adds kd_weight * kd_loss on all views (reported as stats["kd"]).
    """
    x = torch.cat(views) if len(views) > 1 else views[0]
    with torch.autocast(y.device.type, enabled=amp):
        logits = model(x)
    kd = torch.zeros((), device=y.device)
    if teacher is not None and kd_weight > 0:
        with torch.no_grad(), torch.autocast(y.device.type, enabled=amp):
            teacher_logits = teacher(x)
        kd = kd_loss(logits, teacher_logits, kd_temperature)
    if stats is not None:
        stats["kd"] = kd.detach()
    if jsd:
        parts = list(logits.chunk(len(views)))
        if ce_all_views:
            ce = sum(F.cross_entropy(p.float(), y, label_smoothing=smoothing) for p in parts) / len(parts)
        else:
            ce = F.cross_entropy(parts[0].float(), y, label_smoothing=smoothing)
        div = jsd_loss(parts)
        return ce + jsd_weight * div + kd_weight * kd, ce, div, parts[0]
    ce = F.cross_entropy(logits.float(), y, label_smoothing=smoothing)
    return ce + kd_weight * kd, ce, torch.zeros((), device=y.device), logits


class ModelEMA:
    """Exponential moving average of a model's weights and buffers (mean-teacher / Polyak averaging).

    decay_t = min(decay, (1 + t) / (10 + t)), so that the average is not dominated by the random
    initial head during the first steps (the usual warm-up of timm / YOLO EMAs).
    """

    def __init__(self, model, decay: float):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.updates = float(decay), 0

    @torch.no_grad()
    def update(self, model) -> None:
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))
        current = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(current[k].detach().to(v.dtype), alpha=1 - d)
            else:
                v.copy_(current[k])


def _save(obj, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def _rng_state(device) -> dict:
    state = {"torch": torch.get_rng_state(), "numpy": np.random.get_state(), "python": random.getstate()}
    if device.type == "cuda":
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def _set_rng_state(state: dict, device) -> None:
    torch.set_rng_state(state["torch"])
    np.random.set_state(state["numpy"])
    random.setstate(state["python"])
    if device.type == "cuda" and "cuda" in state:
        torch.cuda.set_rng_state_all(state["cuda"])


def train_run(cfg: dict, seed: int, run_dir: Path, model, device, use_wandb: bool = False,
              scratch_dir: Path | None = None, fingerprint: str | None = None, teacher=None) -> dict:
    """Train `model` as described by `cfg` and leave best.pt, log.csv and done.json in `run_dir`.

    Returns the contents of done.json. Safe to call again: a finished run returns immediately, an
    interrupted one continues from its last completed epoch.

    scratch_dir: where the per-epoch checkpoints (last.pt and the in-progress best.pt) are kept;
        defaults to run_dir. best.pt is copied to run_dir once, when the run finishes.
    fingerprint: identifies the configuration; a resume checkpoint with another fingerprint is ignored.
    teacher: the frozen teacher model of a train.distill run.
    """
    run_dir.mkdir(parents=True, exist_ok=True)
    work = Path(scratch_dir) if scratch_dir else run_dir
    work.mkdir(parents=True, exist_ok=True)
    done_path, log_path, final_best = run_dir / "done.json", run_dir / "log.csv", run_dir / "best.pt"
    last_path, best_path = work / "last.pt", work / "best.pt"
    if done_path.exists():
        return json.loads(done_path.read_text())

    t = cfg["train"]
    n_params = sum(p.numel() for p in model.parameters())
    trainable = [p for p in model.parameters() if p.requires_grad]
    if t.get("skip") or not trainable:
        if not t.get("skip"):
            raise RuntimeError(f"run {cfg['name']} has nothing to train; set train.skip: true if that is intended")
        info = {"name": cfg["name"], "seed": seed, "skipped": True, "n_params": n_params, "n_trainable": 0,
                "train_time_s": 0.0, "epochs_run": 0}
        done_path.write_text(json.dumps(info, indent=1))
        return info

    seed_everything(seed)
    amp = bool(t.get("amp", True)) and device.type == "cuda"
    cuda = device.type == "cuda"
    if t.get("cudnn_benchmark") and cuda:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
    if t.get("channels_last") and cuda:
        model = model.to(memory_format=torch.channels_last)
    distill = t.get("distill") or {}
    if distill and teacher is None:
        raise ValueError(f"run {cfg['name']} distils from {distill.get('teacher')!r}: pass the teacher model")
    kd_weight = float(distill.get("weight", 1.0)) if distill else 0.0
    kd_temperature = float(distill.get("temperature", 1.0)) if distill else 1.0
    if teacher is not None:
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
    fit_split = t.get("fit_split", "train")
    if fit_split == "val":
        # Stacked generalisation: fit on one half of the validation split (images the frozen base
        # models never trained on), select on the other half. See data.loading.load_stack_data.
        xtr, ytr, xva, yva, xvc, yvc = load_stack_data(cfg)
    elif fit_split == "train":
        xtr, ytr = load_split(cfg, "train")
        xva, yva = load_split(cfg, "val")
        xvc, yvc, _, _ = load_val_c_split(cfg)
    else:
        raise ValueError(f"train.fit_split must be train | val, got {fit_split!r}")
    Xtr, Ytr = u8_to_device(xtr, device), torch.from_numpy(ytr).to(device)
    Xva, Yva = u8_to_device(xva, device), torch.from_numpy(yva).to(device)
    Xvc, Yvc = u8_to_device(xvc, device), torch.from_numpy(yvc).to(device)

    bs, epochs = int(t["batch_size"]), int(t["epochs"])
    steps_per_epoch = len(Xtr) // bs
    if steps_per_epoch == 0:
        raise ValueError(f"batch size {bs} is larger than the training set ({len(Xtr)})")
    total_steps = epochs * steps_per_epoch
    warmup_steps = int(t.get("warmup_epochs", 0) * steps_per_epoch)
    opt = torch.optim.AdamW(param_groups(model, float(t["weight_decay"])), lr=float(t["lr"]),
                            **({"fused": True} if t.get("fused_optimizer") and cuda else {}))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: lr_factor(s, total_steps, warmup_steps))
    scaler = torch.amp.GradScaler(device.type, enabled=amp)
    aug = TrainAugment(cfg["augment"])
    jsd_weight, smoothing = float(cfg["augment"].get("jsd_weight", 12.0)), float(t.get("label_smoothing", 0.0))
    grad_clip, patience = float(t.get("grad_clip", 0) or 0), int(t.get("patience", 0) or 0)
    eval_bs = int(t.get("eval_batch_size", 512))
    ce_all_views = bool(t.get("ce_all_views", False))
    ema = ModelEMA(model, float(t["ema"])) if t.get("ema") else None
    forward_model = torch.compile(model) if t.get("compile") and cuda else model

    state = {"epoch": 0, "best": -1.0, "best_epoch": 0, "best_val_acc": 0.0, "bad_epochs": 0, "elapsed": 0.0}
    ckpt = torch.load(last_path, map_location=device, weights_only=False) if last_path.exists() else None
    if ckpt is not None and ckpt.get("fingerprint") != fingerprint:
        log.warning(f"[{cfg['name']} seed {seed}] ignoring a resume checkpoint from a different configuration")
        ckpt = None
    if ckpt is not None:
        model.load_checkpoint_state(ckpt["model"])
        opt.load_state_dict(ckpt["opt"])
        sched.load_state_dict(ckpt["sched"])
        scaler.load_state_dict(ckpt["scaler"])
        if ema is not None and "ema" in ckpt:
            ema.module.load_state_dict(ckpt["ema"])
            ema.updates = int(ckpt.get("ema_updates", 0))
        _set_rng_state(ckpt["rng"], device)
        state = ckpt["state"]
        log.info(f"[{cfg['name']} seed {seed}] resuming after epoch {state['epoch']}")
    rows = []
    if log_path.exists():        # keep only epochs that the checkpoint covers
        with open(log_path, newline="") as f:
            rows = [r for r in csv.DictReader(f) if int(r["epoch"]) <= state["epoch"]]
    with open(log_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        w.writeheader()
        w.writerows(rows)

    wb = None
    if use_wandb:
        try:
            import wandb
            wb = wandb.init(project="roadlens", name=f"{cfg['name']}-seed{seed}", config=cfg, reinit=True)
        except ImportError:
            log.warning("wandb is not installed; continuing with CSV logging only")

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    log.info(f"[{cfg['name']} seed {seed}] {n_params / 1e6:.2f} M params ({sum(p.numel() for p in trainable) / 1e6:.2f} M "
             f"trainable), {len(Xtr)} train images, {steps_per_epoch} steps/epoch, views={aug.n_views}, amp={amp}")

    if t.get("eval_at_init") and state["epoch"] == 0 and state["best"] < 0:
        # The untrained model is a candidate too ("epoch 0"). For a gate that starts as the plain
        # average this makes selection do no harm: training is kept only if it beats the average.
        val_acc = accuracy(model, Xva, Yva, eval_bs, amp)
        val_c_acc = accuracy(model, Xvc, Yvc, eval_bs, amp)
        state.update(best=val_c_acc, best_epoch=0, best_val_acc=val_acc)
        _save({"model": model.checkpoint_state(), "epoch": 0, "val_c_acc": val_c_acc, "val_acc": val_acc}, best_path)
        log.info(f"[{cfg['name']} seed {seed}] epoch   0  (untrained)  val {val_acc:.4f}  val-C {val_c_acc:.4f}  *")

    stop = patience > 0 and state["bad_epochs"] >= patience
    for epoch in range(state["epoch"] + 1, epochs + 1):
        if stop:
            break
        model.train()
        t0 = time.time()
        perm = torch.randperm(len(Xtr), device=device)
        tot = {"loss": 0.0, "ce": 0.0, "jsd": 0.0, "kd": 0.0, "correct": 0}
        step_stats = {}
        bar = tqdm(range(steps_per_epoch), desc=f"{cfg['name']} s{seed} ep {epoch}/{epochs}", leave=False, mininterval=2.0)
        for i in bar:
            idx = perm[i * bs:(i + 1) * bs]
            y = Ytr[idx]
            views = aug(as_float(Xtr[idx]))
            if t.get("channels_last") and cuda:
                views = [v.contiguous(memory_format=torch.channels_last) for v in views]
            loss, ce, jsd, first = training_loss(forward_model, views, y, aug.mode == "jsd", smoothing, jsd_weight,
                                                 amp, ce_all_views, teacher, kd_weight, kd_temperature, step_stats)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"non-finite loss at epoch {epoch}, step {i}")
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if grad_clip:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainable, grad_clip)
            scaler.step(opt)
            scaler.update()
            sched.step()
            if ema is not None:
                ema.update(model)
            tot["kd"] += float(step_stats.get("kd", 0.0))
            tot["loss"] += loss.item()
            tot["ce"] += ce.item()
            tot["jsd"] += jsd.item()
            tot["correct"] += int((first.argmax(dim=1) == y).sum())
            if i % 20 == 0:
                bar.set_postfix(loss=f"{loss.item():.3f}")
        epoch_s = time.time() - t0
        eval_model = ema.module if ema is not None else model
        val_acc = accuracy(eval_model, Xva, Yva, eval_bs, amp)
        val_c_acc = accuracy(eval_model, Xvc, Yvc, eval_bs, amp)
        row = {"epoch": epoch, "lr": f"{opt.param_groups[0]['lr']:.3e}",
               "loss": f"{tot['loss'] / steps_per_epoch:.4f}", "ce": f"{tot['ce'] / steps_per_epoch:.4f}",
               "jsd": f"{tot['jsd'] / steps_per_epoch:.4f}", "kd": f"{tot['kd'] / steps_per_epoch:.4f}",
               "train_acc": f"{tot['correct'] / (steps_per_epoch * bs):.4f}",
               "val_acc": f"{val_acc:.4f}", "val_c_acc": f"{val_c_acc:.4f}",
               "epoch_s": f"{epoch_s:.1f}", "img_per_s": f"{steps_per_epoch * bs / epoch_s:.0f}"}
        with open(log_path, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=LOG_FIELDS).writerow(row)
        if wb is not None:
            wb.log({k: float(v) for k, v in row.items()})

        state["epoch"], state["elapsed"] = epoch, state["elapsed"] + epoch_s
        if val_c_acc > state["best"]:
            state.update(best=val_c_acc, best_epoch=epoch, best_val_acc=val_acc, bad_epochs=0)
            _save({"model": eval_model.checkpoint_state(), "epoch": epoch, "val_c_acc": val_c_acc, "val_acc": val_acc},
                  best_path)
        else:
            state["bad_epochs"] += 1
        _save({"model": model.checkpoint_state(), "opt": opt.state_dict(), "sched": sched.state_dict(),
               "scaler": scaler.state_dict(), "rng": _rng_state(device), "state": state,
               "fingerprint": fingerprint,
               **({"ema": ema.module.state_dict(), "ema_updates": ema.updates} if ema is not None else {})}, last_path)
        log.info(f"[{cfg['name']} seed {seed}] epoch {epoch:3d}  loss {row['loss']}  train {row['train_acc']}  "
                 f"val {row['val_acc']}  val-C {row['val_c_acc']}  ({epoch_s:.0f}s, {row['img_per_s']} img/s)"
                 + ("  *" if state["bad_epochs"] == 0 else ""))
        stop = patience > 0 and state["bad_epochs"] >= patience

    info = {"name": cfg["name"], "seed": seed, "skipped": False, "n_params": n_params,
            "n_trainable": sum(p.numel() for p in trainable), "epochs_run": state["epoch"],
            "best_epoch": state["best_epoch"], "best_val_c_acc": state["best"], "val_acc_at_best": state["best_val_acc"],
            "train_time_s": round(state["elapsed"], 1), "early_stopped": bool(stop),
            "peak_gpu_mem_mb": round(torch.cuda.max_memory_allocated(device) / 2 ** 20) if device.type == "cuda" else None}
    if wb is not None:
        wb.summary.update(info)
        wb.finish()
    if best_path != final_best:             # the one write of the weights to the persistent directory
        tmp = final_best.with_suffix(".pt.tmp")
        shutil.copyfile(best_path, tmp)
        os.replace(tmp, final_best)
        best_path.unlink(missing_ok=True)
    done_path.write_text(json.dumps(info, indent=1))
    last_path.unlink(missing_ok=True)       # only needed for resuming
    return info


In [ ]:
%%writefile /content/RoadLens/roadlens/eval/calibration.py
"""Post-hoc temperature scaling (Guo et al. 2017).

One scalar T > 0 per trained model: predictions become softmax(logits / T). The arg-max, and with
it accuracy and macro-F1, never changes; only the confidence does. T is fitted by minimising the
negative log-likelihood on held-out validation data, never on test data.

The models here are trained with label smoothing, which caps their confidence near 0.9 while
their accuracy is far higher, so they need T < 1 (sharpening), the opposite of the usual case.
"""

from __future__ import annotations

import math

import torch
import torch.nn.functional as F


def nll_at(logits: torch.Tensor, y: torch.Tensor, temperature: float) -> float:
    return float(F.cross_entropy(logits.double() / temperature, y))


def fit_temperature(logits: torch.Tensor, y: torch.Tensor, lo: float = 0.02, hi: float = 50.0, iters: int = 60) -> float:
    """T in [lo, hi] that minimises the NLL of softmax(logits / T) on (logits, y).

    The NLL is convex in 1 / T, hence unimodal in log T: a golden-section search on log T finds
    the minimum to about 1e-6 relative precision without any learning rate to tune.
    """
    logits, y = logits.detach().double().cpu(), y.detach().long().cpu()
    if len(y) == 0:
        raise ValueError("cannot fit a temperature on an empty set")
    g = (math.sqrt(5.0) - 1.0) / 2.0
    a, b = math.log(lo), math.log(hi)
    c, d = b - g * (b - a), a + g * (b - a)
    fc, fd = nll_at(logits, y, math.exp(c)), nll_at(logits, y, math.exp(d))
    for _ in range(iters):
        if fc < fd:
            b, d, fd = d, c, fc
            c = b - g * (b - a)
            fc = nll_at(logits, y, math.exp(c))
        else:
            a, c, fc = c, d, fd
            d = a + g * (b - a)
            fd = nll_at(logits, y, math.exp(d))
    return math.exp((a + b) / 2.0)


In [ ]:
%%writefile /content/RoadLens/roadlens/eval/evaluate.py
"""Evaluate a model (optionally with test-time adaptation) on suites of test sets.

A suite is a list of test sets:
    gtsrb_clean   GTSRB train / val / test, clean
    gtsrb_sel     the select half of the validation split, clean and as validation-C (model selection)
    gtsrb_c       GTSRB test: clean + 14 corruptions x 5 severities
    btsd_c, ctsd_c  the same for a cross-dataset test set (labels mapped to GTSRB ids)
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Callable

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from ..data.loading import as_float, load_corrupted_set, load_selection_data, load_split, load_test, u8_to_device
from ..utils import corruption_names
from .metrics import summarize_logits


@dataclass
class EvalSet:
    dataset: str
    split: str
    corruption: str            # "clean" or a corruption name
    severity: int              # 0 for clean
    load: Callable[[], tuple[np.ndarray, np.ndarray]]

    @property
    def key(self) -> str:
        return f"{self.dataset}/{self.split}/{self.corruption}/{self.severity}"


def build_sets(cfg: dict, suite: str, severities=None, corruptions=None) -> list[EvalSet]:
    if suite == "gtsrb_clean":
        return [EvalSet("gtsrb", s, "clean", 0, lambda s=s: load_split(cfg, s)) for s in ("train", "val", "test")]
    if suite == "gtsrb_sel":       # the select half of the validation split: the model-selection data
        return [EvalSet("gtsrb", "val_sel", "clean", 0, lambda: load_selection_data(cfg)[:2]),
                EvalSet("gtsrb", "val_sel", "val_c", 0, lambda: load_selection_data(cfg)[2:])]
    if not suite.endswith("_c"):
        raise ValueError(f"unknown evaluation suite: {suite!r}")
    dataset = suite[:-2]
    sets = [EvalSet(dataset, "test", "clean", 0, lambda: load_test(cfg, dataset))]
    for name in corruptions or corruption_names(cfg):
        for sev in severities or cfg["corruptions"]["severities"]:
            sets.append(EvalSet(dataset, "test", name, int(sev),
                                lambda name=name, sev=sev: load_corrupted_set(cfg, dataset, name, sev)))
    return sets


@torch.no_grad()
def predict_logits(model, x: np.ndarray, device, batch_size: int = 512, amp: bool = True) -> torch.Tensor:
    """Logits [N, K] (float32, CPU) of a uint8 NHWC array, in order, without adaptation."""
    model.eval()
    out = []
    for s in range(0, len(x), batch_size):
        xb = as_float(u8_to_device(x[s:s + batch_size], device))
        with torch.autocast(device.type, enabled=amp and device.type == "cuda"):
            out.append(model(xb).float().cpu())
    return torch.cat(out)


def predict_stream(tta, x: np.ndarray, device, batch_size: int, seed: int = 0) -> torch.Tensor:
    """Run a test-time-adaptation method over `x` as an online stream of batches.

    The stream is shuffled with a fixed seed (a class-sorted stream would make batch statistics
    meaningless) and the method is reset first. Returns logits in the original order of `x`.
    """
    X = u8_to_device(np.asarray(x), device)
    perm = torch.from_numpy(np.random.RandomState(seed).permutation(len(X))).to(device)
    tta.reset()
    out = None
    for s in range(0, len(X), batch_size):
        idx = perm[s:s + batch_size]
        logits = tta.predict(as_float(X[idx]))
        if out is None:
            out = torch.empty(len(X), logits.shape[1], device=device)
        out[idx] = logits
    return out.cpu()


def evaluate_sets(predict: Callable[[np.ndarray], torch.Tensor], sets: list[EvalSet], num_classes: int,
                  n_per_set: int = 0, subset_seed: int = 0, desc: str = "eval"):
    """Evaluate every set. Returns (DataFrame with one row per set, confusions [S, K, K], bins [S, 15, 3]).

    `n_per_set` > 0 evaluates a fixed random subset of each set (same images for every corruption).
    """
    rows, confs, bins = [], [], []
    for es in tqdm(sets, desc=desc, leave=False):
        x, y = es.load()
        if n_per_set and n_per_set < len(x):
            idx = np.sort(np.random.RandomState(subset_seed).permutation(len(x))[:n_per_set])
            x, y = np.asarray(x[idx]), y[idx]
        yt = torch.from_numpy(np.asarray(y)).long()
        shared = np.unique(y) if es.dataset != "gtsrb" else None
        stats, cm, b = summarize_logits(predict(x), yt, num_classes, shared)
        rows.append({"dataset": es.dataset, "split": es.split, "corruption": es.corruption,
                     "severity": es.severity, **stats})
        confs.append(cm)
        bins.append(b)
    return pd.DataFrame(rows), np.stack(confs), np.stack(bins)


In [ ]:
%%writefile /content/RoadLens/roadlens/eval/fourier.py
"""Fourier heat map of model error (Yin et al., "A Fourier Perspective on Model Robustness", 2019).

For every 2-D Fourier basis U_{u,v} (a real planar wave with unit L2 norm) each test image is
perturbed to clip(x + r * eps * U_{u,v}), with an independent random sign r per image and channel,
and the error rate is recorded at position (u, v). Low frequencies sit in the centre of the map.
"""

from __future__ import annotations

import math

import numpy as np
import torch
from tqdm import tqdm

from ..data.loading import as_float, u8_to_device


@torch.no_grad()
def fourier_heatmaps(models: dict, x: np.ndarray, y: np.ndarray, device, eps: float = 4.0, stride: int = 1,
                     batch_size: int = 1000, amp: bool = True, seed: int = 0):
    """Returns ({name: [H/stride, W/stride] error map}, {name: clean error}) for several models at once
    (all models see exactly the same perturbed images)."""
    X = as_float(u8_to_device(x, device))
    Y = torch.from_numpy(np.asarray(y)).to(device)
    n, c, h, w = X.shape
    use_amp = amp and device.type == "cuda"
    for m in models.values():
        m.eval()

    def errors(images):
        out = {}
        with torch.autocast(device.type, enabled=use_amp):
            for name, m in models.items():
                wrong = 0
                for s in range(0, n, batch_size):
                    wrong += int((m(images[s:s + batch_size]).argmax(dim=1) != Y[s:s + batch_size]).sum())
                out[name] = wrong / n
        return out

    clean = errors(X)
    us = list(range(-h // 2, h // 2, stride))
    vs = list(range(-w // 2, w // 2, stride))
    maps = {name: np.full((len(us), len(vs)), np.nan) for name in models}
    gen = torch.Generator(device="cpu").manual_seed(seed)
    ii = torch.arange(h, device=device, dtype=torch.float32).view(h, 1)
    jj = torch.arange(w, device=device, dtype=torch.float32).view(1, w)
    index = {(u % h, v % w): (a, b) for a, u in enumerate(us) for b, v in enumerate(vs)}
    todo = [(u, v) for u in us for v in vs]
    for u, v in tqdm(todo, desc="fourier heat map", leave=False):
        a, b = index[(u % h, v % w)]
        if not np.isnan(maps[next(iter(maps))][a, b]):
            continue                                   # filled by its mirror frequency
        basis = torch.cos(2 * math.pi * (u * ii / h + v * jj / w))
        basis = basis / basis.norm()
        sign = (torch.randint(0, 2, (n, c, 1, 1), generator=gen) * 2 - 1).to(device=device, dtype=torch.float32)
        err = errors((X + eps * sign * basis).clamp_(0, 1))
        mirror = index.get(((-u) % h, (-v) % w))       # U_{-u,-v} is the same real basis
        for name in models:
            maps[name][a, b] = err[name]
            if mirror is not None:
                maps[name][mirror] = err[name]
    return maps, clean


In [ ]:
%%writefile /content/RoadLens/roadlens/eval/gradcam.py
"""Grad-CAM (Selvaraju et al. 2017) for CNN feature maps and for ViT patch tokens."""

from __future__ import annotations

import math

import torch
import torch.nn.functional as F

from ..models.base import CamTarget


def grad_cam(model, targets: list[CamTarget], x: torch.Tensor, class_idx: torch.Tensor | None = None):
    """Class-activation maps of `model` at each target layer.

    x: float images [B, 3, H, W] in [0, 1]. class_idx: classes to explain (default: the predictions).
    Returns (cams: list of [B, H, W] tensors in [0, 1], one per target; logits [B, K]).
    """
    model.eval()
    acts = {}
    hooks = [t.module.register_forward_hook(lambda m, i, o, k=k: acts.__setitem__(k, o))
             for k, t in enumerate(targets)]
    try:
        with torch.enable_grad():
            # the input must require grad: frozen models (the fusion's branches) build no graph otherwise
            logits = model(x.detach().clone().requires_grad_(True)).float()
            cls = logits.argmax(dim=1) if class_idx is None else class_idx
            score = logits.gather(1, cls.view(-1, 1)).sum()
            grads = torch.autograd.grad(score, [acts[k] for k in range(len(targets))], allow_unused=True)
    finally:
        for hook in hooks:
            hook.remove()

    cams = []
    for k, t in enumerate(targets):
        a = acts[k].detach().float()
        g = torch.zeros_like(a) if grads[k] is None else grads[k].detach().float()
        if t.kind == "tokens":                        # [B, prefix + h*w, C] -> [B, C, h, w]
            a, g = a[:, t.prefix_tokens:], g[:, t.prefix_tokens:]
            side = int(math.isqrt(a.shape[1]))
            a = a.transpose(1, 2).reshape(a.shape[0], -1, side, side)
            g = g.transpose(1, 2).reshape(g.shape[0], -1, side, side)
        weights = g.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * a).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False).squeeze(1)
        cams.append(cam / cam.amax(dim=(1, 2), keepdim=True).clamp_min(1e-8))
    return cams, logits.detach()


In [ ]:
%%writefile /content/RoadLens/roadlens/eval/metrics.py
"""Classification, calibration and corruption-robustness metrics."""

from __future__ import annotations

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

N_BINS = 15


def confusion_matrix(pred: torch.Tensor, y: torch.Tensor, k: int) -> torch.Tensor:
    """[k, k] counts, rows = true class, columns = predicted class."""
    return torch.bincount(y * k + pred, minlength=k * k).view(k, k)


def macro_f1(conf) -> float:
    """Unweighted mean F1 over the classes that occur in the ground truth.

    (On GTSRB that is all 43 classes; on a cross-dataset test set it is the shared classes, and
    predictions of an absent class simply count as errors of the true class.)
    """
    conf = np.asarray(conf, dtype=np.float64)
    tp = np.diag(conf)
    support, predicted = conf.sum(1), conf.sum(0)
    present = support > 0
    f1 = 2 * tp[present] / np.maximum(support[present] + predicted[present], 1e-12)
    return float(f1.mean()) if present.any() else float("nan")


def reliability_bins(confidence: torch.Tensor, correct: torch.Tensor, n_bins: int = N_BINS) -> torch.Tensor:
    """[n_bins, 3] = (count, sum of confidence, number correct) over equal-width bins (a, b]."""
    idx = torch.clamp(torch.ceil(confidence * n_bins).long() - 1, 0, n_bins - 1)
    out = torch.zeros(n_bins, 3, dtype=torch.float64)
    out[:, 0] = torch.bincount(idx, minlength=n_bins).double()
    out[:, 1] = torch.bincount(idx, weights=confidence.double(), minlength=n_bins)
    out[:, 2] = torch.bincount(idx, weights=correct.double(), minlength=n_bins)
    return out


def ece_from_bins(bins) -> float:
    """Expected calibration error: sum_b (n_b / N) * |accuracy_b - confidence_b|."""
    bins = np.asarray(bins, dtype=np.float64)
    n = bins[:, 0].sum()
    if n == 0:
        return float("nan")
    return float(np.abs(bins[:, 2] - bins[:, 1]).sum() / n)


def summarize_logits(logits: torch.Tensor, y: torch.Tensor, num_classes: int, shared=None) -> tuple[dict, np.ndarray, np.ndarray]:
    """Metrics of one test set from its logits (CPU tensors).

    Returns (scalars, confusion [K, K], reliability bins [15, 3]). With `shared` (class ids present
    in a cross-dataset test set), `acc_shared` is the accuracy when the arg-max is restricted to them.
    """
    logits = logits.float()
    probs = F.softmax(logits, dim=1)
    conf, pred = probs.max(dim=1)
    correct = pred == y
    cm = confusion_matrix(pred, y, num_classes).numpy()
    bins = reliability_bins(conf, correct).numpy()
    out = {
        "n": int(len(y)),
        "acc": float(correct.float().mean()),
        "macro_f1": macro_f1(cm),
        "ece": ece_from_bins(bins),
        "nll": float(F.cross_entropy(logits, y)),
    }
    if shared is not None:
        mask = torch.full((num_classes,), float("-inf"))
        mask[torch.as_tensor(shared)] = 0.0
        out["acc_shared"] = float(((logits + mask).argmax(dim=1) == y).float().mean())
    return out, cm, bins


# ----------------------------------------------------------------------------------------
# Corruption robustness summaries
# ----------------------------------------------------------------------------------------
def corruption_summary(df: pd.DataFrame, normalizer: pd.DataFrame | None = None) -> dict:
    """Summary of one model/seed.

    df: rows of one evaluation with columns corruption, severity, acc, macro_f1, ece; the clean test
        set is the row with corruption == 'clean'.
    normalizer: the same kind of table for the reference model (SmallCNN), averaged over its seeds.
        mCE     = 100 * mean_c [ sum_s E(c, s) / sum_s E_ref(c, s) ]
        rel_mce = 100 * mean_c [ sum_s (E(c, s) - E_clean) / sum_s (E_ref(c, s) - E_ref_clean) ]
    (Hendrycks & Dietterich 2019, with SmallCNN in the role of AlexNet.)
    """
    clean = df[df.corruption == "clean"].iloc[0]
    corr = df[df.corruption != "clean"]
    out = {
        "clean_acc": 100 * clean.acc, "clean_f1": 100 * clean.macro_f1, "clean_ece": 100 * clean.ece,
        "corr_acc": 100 * corr.acc.mean(), "corr_f1": 100 * corr.macro_f1.mean(), "corr_ece": 100 * corr.ece.mean(),
    }
    if normalizer is not None:
        ref_clean_err = 1 - float(normalizer[normalizer.corruption == "clean"].acc.iloc[0])
        ref = normalizer[normalizer.corruption != "clean"].groupby("corruption").acc.mean()
        err = 1 - corr.groupby("corruption").acc.mean()
        ref_err = (1 - ref).reindex(err.index)
        ce = err / ref_err.clip(lower=1e-4)
        rel = (err - (1 - clean.acc)) / (ref_err - ref_clean_err).clip(lower=1e-4)
        out["mce"] = 100 * float(ce.mean())
        out["rel_mce"] = 100 * float(rel.mean())
    return out


def mean_std(df: pd.DataFrame, by, cols=None) -> pd.DataFrame:
    """Mean and standard deviation over seeds: one row per group, columns <col>_mean / <col>_std
    (sample std, ddof=1; NaN when there is a single seed) plus n_seeds."""
    by = [by] if isinstance(by, str) else list(by)
    cols = cols or [c for c in df.columns if c not in by and c != "seed" and pd.api.types.is_numeric_dtype(df[c])]
    g = df.groupby(by, sort=False)
    out = g.size().rename("n_seeds").to_frame()
    mean, std = g[cols].mean(), g[cols].std(ddof=1)
    for c in cols:
        out[f"{c}_mean"], out[f"{c}_std"] = mean[c], std[c]
    return out.reset_index()


In [ ]:
%%writefile /content/RoadLens/roadlens/eval/plots.py
"""Figures (matplotlib, vector PDF).

Colour rules: model / method identity uses a fixed categorical palette that was checked for
colour-vision-deficiency separation, and every series also gets its own marker, so identity never
depends on colour alone. Magnitudes (confusion, Fourier error, Grad-CAM) use one hue from light to
dark. Every figure has its numbers in a CSV next to it.
"""

from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

from .metrics import ece_from_bins

# categorical slots, in fixed order (never cycled)
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SEQUENTIAL = LinearSegmentedColormap.from_list(
    "roadlens_blue", ["#ffffff", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])


def style() -> None:
    plt.rcParams.update({
        "pdf.fonttype": 42, "ps.fonttype": 42, "font.size": 8, "font.family": "sans-serif",
        "axes.titlesize": 8, "axes.labelsize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
        "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
        "axes.edgecolor": AXIS, "axes.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.5, "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 1.5, "lines.markersize": 4.5, "legend.frameon": False,
        "figure.dpi": 120, "savefig.bbox": "tight", "savefig.pad_inches": 0.03,
    })


def series_style(names: list[str]) -> dict[str, dict]:
    """Fixed colour + marker per entity, assigned in order."""
    if len(names) > len(CATEGORICAL):
        raise ValueError(f"at most {len(CATEGORICAL)} series per figure (got {len(names)}); split the figure instead")
    return {n: {"color": CATEGORICAL[i], "marker": MARKERS[i], "markeredgecolor": "white", "markeredgewidth": 0.6}
            for i, n in enumerate(names)}


def _save(fig, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path)
    plt.close(fig)


# ----------------------------------------------------------------------------------------
def plot_severity_curves(per_seed: pd.DataFrame, runs: list[str], labels: dict, corruptions: list[str], path: Path) -> None:
    """Accuracy vs. severity, one panel per corruption plus the mean; mean over seeds with a +-1 std band.

    per_seed: columns run, seed, corruption, severity, acc (clean test set = corruption 'clean', severity 0).
    """
    style()
    st = series_style(runs)
    panels = corruptions + ["mean of all corruptions"]
    ncols = 5
    nrows = int(np.ceil(len(panels) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7.2, 1.55 * nrows + 0.5), sharex=True, sharey=True, squeeze=False)
    sev = sorted(per_seed.loc[per_seed.corruption != "clean", "severity"].unique())
    for ax, name in zip(axes.ravel(), panels):
        for run in runs:
            d = per_seed[per_seed.run == run]
            clean = 100 * d[d.corruption == "clean"].set_index("seed").acc
            if name in corruptions:
                c = 100 * d[d.corruption == name].pivot(index="seed", columns="severity", values="acc")
            else:
                c = 100 * d[d.corruption != "clean"].groupby(["seed", "severity"]).acc.mean().unstack("severity")
            vals = pd.concat([clean.rename(0), c[sev]], axis=1)
            mean, std = vals.mean(0).values, vals.std(0, ddof=1).fillna(0).values
            xs = [0] + list(sev)
            ax.fill_between(xs, mean - std, mean + std, color=st[run]["color"], alpha=0.12, linewidth=0)
            ax.plot(xs, mean, label=labels.get(run, run), **st[run])
        ax.set_title(name.replace("_", " "), color=INK)
        ax.set_xticks([0] + list(sev))
    for ax in axes.ravel()[len(panels):]:
        ax.axis("off")
    for ax in axes[-1]:
        ax.set_xlabel("severity (0 = clean)")
    for ax in axes[:, 0]:
        ax.set_ylabel("accuracy (%)")
    handles, lab = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, lab, loc="upper center", ncol=min(len(runs), 3), bbox_to_anchor=(0.5, 1.0))
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.035 * int(np.ceil(len(runs) / 3)) - 0.02))
    _save(fig, path)


def plot_confusion(conf: np.ndarray, class_labels: list[str], title: str, path: Path, annotate_min: float = 0.10,
                   max_annotations: int = 30) -> None:
    """Row-normalised confusion matrix (rows = true class). Off-diagonal cells holding at least
    `annotate_min` of a class are labelled with their percentage (largest first, capped)."""
    style()
    conf = np.asarray(conf, dtype=np.float64)
    k = conf.shape[0]
    frac = conf / np.maximum(conf.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(7.2, 6.6))
    mesh = ax.pcolormesh(np.arange(k + 1), np.arange(k + 1), frac, cmap=SEQUENTIAL, vmin=0, vmax=1,
                         edgecolors="white", linewidth=0.15)
    ax.set_xlim(0, k)
    ax.set_ylim(k, 0)
    ax.set_aspect("equal")
    ax.grid(False)
    ax.set_xticks(np.arange(k) + 0.5)
    ax.set_yticks(np.arange(k) + 0.5)
    ax.set_xticklabels(class_labels, rotation=90, fontsize=5.5)
    ax.set_yticklabels(class_labels, fontsize=5.5)
    ax.tick_params(length=0)
    ax.set_xlabel("predicted class")
    ax.set_ylabel("true class")
    acc = np.trace(conf) / max(conf.sum(), 1)
    ax.set_title(f"{title}  (accuracy {100 * acc:.2f}%)", color=INK)
    off = frac.copy()
    np.fill_diagonal(off, 0)
    cells = np.argwhere(off >= annotate_min)
    cells = sorted(cells, key=lambda rc: -off[rc[0], rc[1]])[:max_annotations]
    for r, c in cells:
        ax.text(c + 0.5, r + 0.5, f"{100 * off[r, c]:.0f}", ha="center", va="center", fontsize=4.5,
                color="white" if off[r, c] > 0.5 else INK)
    cb = fig.colorbar(mesh, ax=ax, fraction=0.035, pad=0.02)
    cb.set_label("fraction of the true class")
    cb.outline.set_visible(False)
    _save(fig, path)


def plot_reliability(bins: dict, conditions: list[str], runs: list[str], labels: dict, path: Path) -> None:
    """Reliability diagrams. bins[(run, condition)] = [15, 3] array (count, sum confidence, n correct).
    Bars: accuracy of each confidence bin. Diagonal: perfect calibration."""
    style()
    fig, axes = plt.subplots(len(conditions), len(runs), figsize=(1.75 * len(runs) + 0.4, 1.8 * len(conditions) + 0.3),
                             sharex=True, sharey=True, squeeze=False)
    for i, cond in enumerate(conditions):
        for j, run in enumerate(runs):
            ax, b = axes[i, j], np.asarray(bins[(run, cond)], dtype=np.float64)
            n_bins = len(b)
            edges = np.linspace(0, 1, n_bins + 1)
            used = b[:, 0] > 0
            acc = np.divide(b[:, 2], b[:, 0], out=np.zeros(n_bins), where=used)
            ax.plot([0, 1], [0, 1], color=MUTED, linewidth=0.8, zorder=1)
            ax.bar(edges[:-1][used] + 0.5 / n_bins, acc[used], width=0.82 / n_bins, color=CATEGORICAL[0], zorder=2)
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.set_aspect("equal")
            ax.set_xticks([0, 0.5, 1])
            ax.set_yticks([0, 0.5, 1])
            ax.text(0.04, 0.95, f"ECE {100 * ece_from_bins(b):.2f}%", transform=ax.transAxes, va="top", fontsize=7,
                    color=INK, zorder=3, bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.85, "pad": 1.5})
            if i == 0:
                ax.set_title(labels.get(run, run), color=INK)
            if i == len(conditions) - 1:
                ax.set_xlabel("confidence")
            if j == 0:
                ax.set_ylabel(f"{cond}\naccuracy")
    fig.tight_layout()
    _save(fig, path)


def plot_fourier(maps: dict, clean_err: dict, labels: dict, eps: float, path: Path) -> None:
    """Fourier heat maps (Yin et al. 2019): error rate under a perturbation along each Fourier basis.
    maps[run] = [h, w] array, zero frequency in the centre."""
    style()
    runs = list(maps)
    fig, axes = plt.subplots(1, len(runs), figsize=(2.3 * len(runs) + 0.6, 2.95), sharey=True, squeeze=False)
    mesh = None
    for ax, run in zip(axes[0], runs):
        m = np.asarray(maps[run])
        h, w = m.shape
        mesh = ax.pcolormesh(np.linspace(-0.5, 0.5, w + 1), np.linspace(-0.5, 0.5, h + 1), m, cmap=SEQUENTIAL,
                             vmin=0, vmax=1, rasterized=False)
        ax.set_aspect("equal")
        ax.grid(False)
        ax.set_xticks([-0.5, 0, 0.5])
        ax.set_yticks([-0.5, 0, 0.5])
        ax.set_xlabel("horizontal frequency\n(cycles / pixel)")
        ax.set_title(f"{labels.get(run, run)}\nclean error {100 * clean_err[run]:.1f}%, "
                     f"mean {100 * np.nanmean(m):.1f}%", color=INK)
    axes[0, 0].set_ylabel("vertical frequency\n(cycles / pixel)")
    cb = fig.colorbar(mesh, ax=list(axes[0]), fraction=0.025, pad=0.02)
    cb.set_label("error rate")
    cb.outline.set_visible(False)
    fig.suptitle(f"Error under a Fourier-basis perturbation of L2 norm {eps:g} per channel", y=1.04, fontsize=8, color=INK)
    _save(fig, path)


def cam_overlay(image: np.ndarray, cam: np.ndarray) -> np.ndarray:
    """Blend a single-hue heat layer (opacity = Grad-CAM value) over the desaturated image."""
    gray = image.mean(axis=2, keepdims=True).repeat(3, axis=2) * 0.85 + 0.1
    hue = np.array(matplotlib.colors.to_rgb(CATEGORICAL[1]))
    a = (0.85 * np.clip(cam, 0, 1))[..., None]
    return np.clip((1 - a) * gray + a * hue, 0, 1)


def plot_gradcam(columns: list[dict], row_labels: list[str], path: Path) -> None:
    """columns: one dict per sample with 'title' and 'cells' = list (one per row) of (image HxWx3 in [0,1], caption)."""
    style()
    n_rows, n_cols = len(row_labels), len(columns)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.12 * n_cols + 1.1, 1.22 * n_rows + 0.2), squeeze=False)
    for j, col in enumerate(columns):
        for i, (img, caption) in enumerate(col["cells"]):
            ax = axes[i, j]
            ax.imshow(img, interpolation="nearest")
            ax.set_xticks([])
            ax.set_yticks([])
            ax.grid(False)
            for s in ax.spines.values():
                s.set_visible(False)
            ax.set_xlabel(caption, fontsize=5.5, labelpad=1.5, color=INK2)
            if i == 0:
                ax.set_title(col["title"], fontsize=6.5, color=INK)
            if j == 0:
                ax.set_ylabel(row_labels[i], rotation=0, ha="right", va="center", fontsize=6.5, color=INK)
    fig.tight_layout(h_pad=0.25, w_pad=0.2)
    _save(fig, path)


def plot_tta_sweep(summary: pd.DataFrame, sizes: list[int], method_labels: dict, path: Path, title: str = "") -> None:
    """Accuracy vs. test batch size, one panel per model, one line per adaptation method.
    summary: columns model, method, batch_size, acc_mean, acc_std."""
    style()
    models = list(dict.fromkeys(summary.model))
    methods = list(dict.fromkeys(summary.method))
    st = series_style(methods)
    fig, axes = plt.subplots(1, len(models), figsize=(1.9 * len(models) + 0.5, 2.3), sharey=True, squeeze=False)
    xs = np.arange(len(sizes))
    for ax, model in zip(axes[0], models):
        for method in methods:
            d = summary[(summary.model == model) & (summary.method == method)].set_index("batch_size").reindex(sizes)
            if d.acc_mean.isna().all():
                continue
            ax.errorbar(xs, d.acc_mean, yerr=d.acc_std.fillna(0), label=method_labels.get(method, method),
                        capsize=2, elinewidth=0.8, **st[method])
        ax.set_xticks(xs)
        ax.set_xticklabels([str(s) for s in sizes])
        ax.set_xlabel("test batch size")
        ax.set_title(model, color=INK)
    axes[0, 0].set_ylabel("mean corruption accuracy (%)")
    handles, lab = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, lab, loc="upper center", ncol=len(methods), bbox_to_anchor=(0.5, 1.0))
    if title:
        fig.suptitle(title, y=1.1, fontsize=8, color=INK)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    _save(fig, path)


# ----------------------------------------------------------------------------------------
# accuracy heatmap: corruption x severity (one model per figure)
# ----------------------------------------------------------------------------------------
HEATMAP_ORDER = ["gaussian_noise", "shot_noise", "impulse_noise", "defocus_blur", "motion_blur", "zoom_blur", "snow",
                 "frost", "fog", "brightness", "contrast", "pixelate", "elastic_transform", "jpeg_compression"]


def slug(text: str) -> str:
    return "".join(ch if ch.isalnum() else "_" for ch in text).strip("_").lower()


def plot_corruption_heatmaps(long: pd.DataFrame, key: str, out_dir: Path, dataset: str = "GTSRB") -> list[Path]:
    """One heatmap PNG per value of `key` (e.g. "step" or "model"), averaged over any other columns (seeds)."""
    out = []
    for name, g in long.groupby(key, sort=False):
        d = g.groupby(["corruption", "severity"], as_index=False).acc.mean()
        out.append(plot_corruption_heatmap(d, f"{dataset} - {name} accuracy (%)", Path(out_dir) / f"{slug(str(name))}.png"))
    return out


def plot_corruption_heatmap(df: pd.DataFrame, title: str, path: Path, order: list[str] | None = None,
                            dpi: int = 200) -> Path:
    """Accuracy of one model on every corruption (rows) at every severity (columns), annotated in %.

    df: one evaluation table with columns corruption, severity, acc (the clean row, corruption == 'clean',
    gives the subtitle). Viridis from 0 to 1, so figures of different models are directly comparable.
    Written as PNG (dpi) or whatever suffix `path` has.
    """
    order = [c for c in (order or HEATMAP_ORDER) if c in set(df.corruption)]
    corr = df[df.corruption != "clean"]
    grid = corr.pivot_table(index="corruption", columns="severity", values="acc").reindex(order)
    sev = list(grid.columns)
    clean = df[df.corruption == "clean"]
    subtitle = f"clean test accuracy {100 * float(clean.acc.iloc[0]):.1f}%" if len(clean) else ""

    fig, ax = plt.subplots(figsize=(5.4, 5.6))
    mesh = ax.imshow(grid.values, cmap="viridis", vmin=0.0, vmax=1.0, aspect="auto")
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            v = float(grid.values[i, j])
            ax.text(j, i, f"{100 * v:.1f}", ha="center", va="center", fontsize=6.5,
                    color="white" if v < 0.62 else "black")
    ax.set_xticks(range(len(sev)), [str(int(s)) for s in sev], fontsize=7)
    ax.set_yticks(range(len(order)), [c.replace("_", " ") for c in order], fontsize=7)
    ax.set_xlabel("severity", fontsize=8)
    ax.grid(False)
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=0)
    cbar = fig.colorbar(mesh, ax=ax, fraction=0.05, pad=0.03)
    cbar.set_label("accuracy", fontsize=8)
    cbar.ax.tick_params(labelsize=7, length=2)
    cbar.outline.set_visible(False)
    ax.set_title(f"{title}\n{subtitle}" if subtitle else title, fontsize=8.5, color=INK)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return path


In [ ]:
%%writefile /content/RoadLens/roadlens/eval/tables.py
"""Result tables: 'mean +- std' formatting, CSV and LaTeX (booktabs) writers."""

from __future__ import annotations

import math
from pathlib import Path

import pandas as pd

_ESCAPE = {"_": r"\_", "%": r"\%", "&": r"\&", "#": r"\#"}


def tex_escape(text: str) -> str:
    return "".join(_ESCAPE.get(ch, ch) for ch in str(text))


def pm(mean: float, std: float | None = None, digits: int = 2, tex: bool = False) -> str:
    """'98.12 +- 0.05'. The +- part is dropped when there is no std (single seed)."""
    if mean is None or (isinstance(mean, float) and math.isnan(mean)):
        return "--" if tex else "n/a"
    s = f"{mean:.{digits}f}"
    if std is not None and not math.isnan(std):
        s += (r" $\pm$ " if tex else " +- ") + f"{std:.{digits}f}"
    return s


def over_seeds(n: int) -> str:
    """Caption fragment: how the numbers were aggregated."""
    return f"mean $\\pm$ std over {n} seeds" if n > 1 else "one seed"


def format_summary(summary: pd.DataFrame, columns: dict, label_col: str = "model") -> tuple[pd.DataFrame, list[list[str]]]:
    """Turn a mean_std() table into display cells.

    columns: {metric: (header, digits, best)} with best in {"max", "min", None}; the metric's values are
    read from <metric>_mean / <metric>_std. Returns (plain-text DataFrame, LaTeX cell rows with the best
    mean of each column in bold).
    """
    plain = {label_col: summary[label_col].tolist()}
    tex_rows = [[tex_escape(v)] for v in summary[label_col]]
    for metric, (header, digits, best) in columns.items():
        means = summary[f"{metric}_mean"] if f"{metric}_mean" in summary else pd.Series([float("nan")] * len(summary))
        stds = summary[f"{metric}_std"] if f"{metric}_std" in summary else pd.Series([float("nan")] * len(summary))
        target = None
        if best and means.notna().any():
            target = round(means.max() if best == "max" else means.min(), digits)
        plain[header] = [pm(m, s, digits) for m, s in zip(means, stds)]
        for row, m, s in zip(tex_rows, means, stds):
            cell = pm(m, s, digits, tex=True)
            if target is not None and not math.isnan(m) and round(m, digits) == target:
                cell = r"\textbf{" + f"{m:.{digits}f}" + "}" + cell[len(f"{m:.{digits}f}"):]
            row.append(cell)
    return pd.DataFrame(plain), tex_rows


def write_latex(path: Path, headers: list[str], rows: list[list[str]], caption: str, label: str,
                align: str | None = None, note: str | None = None) -> None:
    """Write a booktabs table. `rows` are already-escaped LaTeX cells (see format_summary)."""
    align = align or ("l" + "c" * (len(headers) - 1))
    lines = [r"% Requires \usepackage{booktabs}. Generated by RoadLens; do not edit by hand.",
             r"\begin{table}[t]", r"\centering", r"\small",
             r"\caption{" + caption + "}", r"\label{" + label + "}",
             r"\begin{tabular}{" + align + "}", r"\toprule",
             " & ".join(tex_escape(h) for h in headers) + r" \\", r"\midrule"]
    lines += [" & ".join(r) + r" \\" for r in rows]
    lines += [r"\bottomrule", r"\end{tabular}"]
    if note:
        lines.append(r"\\[2pt] {\footnotesize " + note + "}")
    lines.append(r"\end{table}")
    Path(path).write_text("\n".join(lines) + "\n", encoding="utf-8")


def write_table(out_dir: Path, stem: str, summary: pd.DataFrame, columns: dict, caption: str, label: str,
                label_col: str = "model", label_header: str = "Model", note: str | None = None) -> pd.DataFrame:
    """Write <stem>.csv (raw means / stds), <stem>.txt (readable) and <stem>.tex (booktabs)."""
    out_dir.mkdir(parents=True, exist_ok=True)
    summary.to_csv(out_dir / f"{stem}.csv", index=False, float_format="%.4f")
    plain, tex_rows = format_summary(summary, columns, label_col)
    plain = plain.rename(columns={label_col: label_header})
    (out_dir / f"{stem}.txt").write_text(plain.to_string(index=False) + "\n", encoding="utf-8")
    write_latex(out_dir / f"{stem}.tex", list(plain.columns), tex_rows, caption, label, note=note)
    return plain


In [ ]:
%%writefile /content/RoadLens/roadlens/experiment.py
"""Shared plumbing of the experiment scripts: CLI, config loading, train-if-needed + cached evaluation."""

from __future__ import annotations

import argparse
from pathlib import Path

import pandas as pd

from .eval.metrics import corruption_summary, mean_std
from .runs import ensure_trained, evaluate_run
from .utils import REPO_ROOT, apply_profile, get_device, get_logger, load_config, results_root

log = get_logger()


def make_parser(description: str, default_config: str) -> argparse.ArgumentParser:
    ap = argparse.ArgumentParser(description=description, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--config", default=default_config)
    ap.add_argument("--profile", default=None, help="smoke | pilot (configs/profiles/); default: full scale")
    ap.add_argument("--seeds", type=int, nargs="+", default=None, help="override the seeds of the config")
    ap.add_argument("--device", default=None)
    ap.add_argument("--wandb", action="store_true", help="also log training to Weights & Biases")
    ap.add_argument("-o", "--override", dest="overrides", action="append", default=[],
                    metavar="KEY=VALUE", help="experiment-config override, repeatable (e.g. -o n_per_set=500)")
    return ap


class Experiment:
    """One experiment = an experiment config + a results directory + access to runs."""

    def __init__(self, args, name: str):
        cfg = apply_profile(load_config(REPO_ROOT / args.config if not Path(args.config).is_absolute() else args.config,
                                        args.overrides), args.profile)
        if args.seeds:
            cfg["seeds"] = args.seeds
        self.cfg, self.profile, self.wandb = cfg, args.profile, args.wandb
        self.device = get_device(args.device)
        self.seeds = list(cfg["seeds"])
        self.out = results_root(cfg) / name
        self.out.mkdir(parents=True, exist_ok=True)
        log.info(f"{name}: seeds {self.seeds}, device {self.device}, results -> {self.out}"
                 + (f"  [profile: {self.profile}]" if self.profile else ""))

    def label(self, run: str) -> str:
        return self.cfg.get("labels", {}).get(run, run)

    def train(self, runs: list[str]) -> pd.DataFrame:
        """Make sure every (run, seed) is trained; one seed at a time so partial results arrive early."""
        rows = []
        for seed in self.seeds:
            for run in runs:
                info = ensure_trained(run, seed, self.profile, self.device, self.wandb)
                rows.append({"run": run, "seed": seed, **info})
        return pd.DataFrame(rows)

    def evaluate(self, runs: list[str], suite: str, **kw) -> tuple[pd.DataFrame, dict]:
        """Long table over runs x seeds (columns run, seed + one row per test set) and the cached arrays."""
        frames, arrays = [], {}
        for seed in self.seeds:
            for run in runs:
                ensure_trained(run, seed, self.profile, self.device, self.wandb)
                df, arr = evaluate_run(run, seed, suite, self.profile, self.device, **kw)
                frames.append(df.assign(run=run, seed=seed))
                arrays[(run, seed)] = arr
        return pd.concat(frames, ignore_index=True), arrays

    def normalizer_table(self, suite: str) -> pd.DataFrame:
        """Per-set accuracy of the mCE reference model (SmallCNN), averaged over seeds."""
        ref, _ = self.evaluate([self.cfg["normalizer"]], suite)
        return ref.groupby(["corruption", "severity"], as_index=False).acc.mean()

    def summarize(self, long: pd.DataFrame, suite: str, key: str = "run") -> tuple[pd.DataFrame, pd.DataFrame]:
        """Per-seed and mean+-std corruption summaries (clean acc, corruption acc, mCE, relative mCE, F1, ECE)."""
        norm = self.normalizer_table(suite)
        rows = []
        for (k, seed), g in long.groupby([key, "seed"], sort=False):
            rows.append({key: k, "seed": seed, **corruption_summary(g, norm)})
        per_seed = pd.DataFrame(rows)
        return per_seed, mean_std(per_seed, key)


In [ ]:
%%writefile /content/RoadLens/roadlens/models/__init__.py
from __future__ import annotations

from .base import CamTarget, Classifier  # noqa: F401
from .fusion import Fusion
from .smallcnn import SmallCNN
from .timm_models import TimmClassifier


def build_model(mcfg: dict, num_classes: int = 43, branches: list | None = None,
                pretrained: bool | None = None) -> Classifier:
    """Build a model from the `model:` section of a run config.

    name: smallcnn | fusion | <any timm model>.  For `fusion`, pass the already-built branch
    models. `pretrained=False` skips the ImageNet download (used when weights come from a checkpoint).
    """
    name = mcfg["name"]
    input_norm = mcfg.get("input_norm", "dataset")
    if name == "smallcnn":
        return SmallCNN(num_classes, mcfg.get("width", 32), mcfg.get("dropout", 0.3), input_norm)
    if name == "fusion":
        if not branches:
            raise ValueError("a fusion model needs its branch models")
        f = mcfg.get("fusion", {})
        return Fusion(branches, num_classes, f.get("mode", "gate"), f.get("gate_inputs", ("features", "probs")),
                      f.get("hidden", 256), f.get("dropout", 0.1), f.get("freeze_branches", True),
                      f.get("views", ("raw",)))
    use_pretrained = mcfg.get("pretrained", True) if pretrained is None else pretrained
    return TimmClassifier(name, num_classes, use_pretrained, mcfg.get("input_size", 128), mcfg.get("drop_path", 0.0),
                          input_norm)


def count_parameters(model) -> int:
    return sum(p.numel() for p in model.parameters())


In [ ]:
%%writefile /content/RoadLens/roadlens/models/base.py
"""Common interface of every RoadLens classifier.

All models take float images [B, 3, H, W] in [0, 1] at the working resolution (64 px).
Normalisation and any upsampling to the backbone's input size happen inside the model, so
augmentations, corruptions, Fourier perturbations and Grad-CAM all work in plain pixel space.
"""

from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class Preprocess(nn.Module):
    """Normalise and (optionally) bilinearly resize to the backbone's input size.

    norm 'dataset'   subtract the dataset mean and divide by its std, per channel (the default).
    norm 'instance'  standardise every image with its own per-channel mean and std. The network then
                     cannot see a per-channel affine change of intensity (gain and offset), which is
                     what exposure, haze and a loss of contrast mostly are. Contrast normalisation of
                     this kind is classical for traffic signs (Sermanet & LeCun 2011, Ciresan et al. 2012).
    norm 'local'     local-global contrast normalisation (RoadLens v3). Instance normalisation removes
                     one gain and offset per image; haze that varies across the image (fog is a
                     low-frequency veil) survives it. Here every pixel is also standardised by the
                     mean and std of its Gaussian neighbourhood (sigma = LOCAL_SIGMA px at 64 px):

                         local(x) = (x - G*x) / max(sqrt(G*x^2 - (G*x)^2), LOCAL_FLOOR * std(x))

                     The floor is relative to the image's own std, so local(a*x + b) = local(x) still
                     holds exactly, while flat regions (a sign's interior) are not amplified into
                     noise. The output is a learnable per-channel blend of the two,
                     (1 - s) * instance(x) + s * local(x) with s = sigmoid(mix), starting at s = 0.5,
                     so training decides how much local detail to use; absolute colour, which local
                     normalisation discards, stays available through the instance part.
    """

    EPS = 1e-4
    LOCAL_SIGMA = 4.0
    LOCAL_FLOOR = 0.5
    LOCAL_CLIP = 6.0

    def __init__(self, input_size: int | None = None, mean=IMAGENET_MEAN, std=IMAGENET_STD, norm: str = "dataset"):
        super().__init__()
        if norm not in ("dataset", "instance", "local"):
            raise ValueError(f"input_norm must be dataset | instance | local, got {norm!r}")
        self.input_size, self.norm = input_size, norm
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1), persistent=False)
        if norm == "local":
            self.mix = nn.Parameter(torch.zeros(1, 3, 1, 1))

    @classmethod
    def local_norm(cls, x: torch.Tensor) -> torch.Tensor:
        """Per-pixel standardisation by the Gaussian-weighted neighbourhood (see the class docstring)."""
        from .views import gauss

        x = x.float()
        mu = gauss(x, cls.LOCAL_SIGMA)
        sd = (gauss(x * x, cls.LOCAL_SIGMA) - mu * mu).clamp_min(0).sqrt()
        floor = cls.LOCAL_FLOOR * x.std(dim=(2, 3), keepdim=True, correction=0) + cls.EPS
        return ((x - mu) / torch.maximum(sd, floor)).clamp(-cls.LOCAL_CLIP, cls.LOCAL_CLIP)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.norm in ("instance", "local"):
            x = x.float()
            inst = (x - x.mean(dim=(2, 3), keepdim=True)) / (x.std(dim=(2, 3), keepdim=True, correction=0) + self.EPS)
            if self.norm == "local":
                s = torch.sigmoid(self.mix)
                x = (1 - s) * inst + s * self.local_norm(x)
            else:
                x = inst
        else:
            x = (x - self.mean) / self.std
        if self.input_size and x.shape[-1] != self.input_size:
            x = F.interpolate(x, size=(self.input_size, self.input_size), mode="bilinear", align_corners=False)
        return x


@dataclass
class CamTarget:
    """A layer whose activations Grad-CAM can use."""
    name: str
    module: nn.Module
    kind: str                 # "conv": [B, C, h, w]   "tokens": [B, prefix + h*w, C]
    prefix_tokens: int = 0


class Classifier(nn.Module):
    feature_dim: int = 0

    def features_logits(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Return (pooled features [B, D], logits [B, K])."""
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.features_logits(x)[1]

    def cam_targets(self) -> list[CamTarget]:
        return []

    # Checkpoints store only what a run actually trained (see Fusion).
    def checkpoint_state(self) -> dict:
        return self.state_dict()

    def load_checkpoint_state(self, state: dict) -> None:
        self.load_state_dict(state)


In [ ]:
%%writefile /content/RoadLens/roadlens/models/fusion.py
"""Fusion: a mixture of expert predictions with per-image weights.

    p(y | x) = sum_e w_e(x) * softmax(z_e(x))

An expert is one branch evaluated on one input view (roadlens/models/views.py). With the default
single view 'raw' the experts are simply the branches; with views (raw, median3, gauss) every
branch also sees two low-pass copies of the image, and the experts are ordered view by view
(all branches on the first view, then all branches on the second, ...).

mode 'gate'       w = softmax(MLP(gate inputs))                          (learned)
mode 'confidence' w_e proportional to the expert's max softmax probability (fallback comparison)
mode 'average'    w_e = 1 / E                                             (plain ensemble)

Gate inputs (any combination):
    features  the branches' pooled features (large; needs a lot of data)
    probs     the experts' class probabilities
    stats     three confidence numbers per expert plus their agreement (views.expert_stats)
    image     five statistics of the input image (views.image_stats), so that the gate can tell
              a noisy or hazy image from a clean one

The output is log p, which is a valid logit vector (softmax(log p) = p), so losses, metrics,
calibration and TENT treat the fusion like any other classifier. Because the result is a convex
combination, the gate can only re-weight the experts; it cannot invent a class no expert supports.
A gate starts as the plain average (zero-initialised output layer).
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

from .base import Classifier
from .views import N_IMAGE_STATS, VIEWS, expert_stats, image_stats

GATE_INPUTS = ("features", "probs", "stats", "image")


class Fusion(Classifier):
    def __init__(self, branches: list[Classifier], num_classes: int = 43, mode: str = "gate",
                 gate_inputs=("features", "probs"), hidden: int = 256, dropout: float = 0.1,
                 freeze_branches: bool = True, views=("raw",)):
        super().__init__()
        if mode not in ("gate", "confidence", "average"):
            raise ValueError(f"fusion mode must be gate | confidence | average, got {mode!r}")
        self.views = tuple(views)
        unknown = [v for v in self.views if v not in VIEWS]
        if unknown or not self.views:
            raise ValueError(f"fusion views must be a non-empty subset of {sorted(VIEWS)}, got {list(views)}")
        self.branches = nn.ModuleList(branches)
        self.mode = mode
        self.gate_inputs = tuple(gate_inputs)
        self.freeze_branches = bool(freeze_branches) or mode != "gate"
        self.feature_dim = sum(b.feature_dim for b in branches)
        self.n_experts = len(branches) * len(self.views)
        if self.freeze_branches:
            for p in self.branches.parameters():
                p.requires_grad_(False)
        if mode == "gate":
            bad = [g for g in self.gate_inputs if g not in GATE_INPUTS]
            if bad or not self.gate_inputs:
                raise ValueError(f"gate_inputs must be a non-empty subset of {list(GATE_INPUTS)}, got {list(gate_inputs)}")
            in_dim = 0
            if "features" in self.gate_inputs:
                self.feat_norms = nn.ModuleList([nn.LayerNorm(b.feature_dim) for b in branches])
                in_dim += self.feature_dim * len(self.views)
            if "probs" in self.gate_inputs:
                in_dim += num_classes * self.n_experts
            if "stats" in self.gate_inputs:
                in_dim += 3 * self.n_experts + 1
            if "image" in self.gate_inputs:
                in_dim += N_IMAGE_STATS
            self.gate = nn.Sequential(nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(dropout),
                                      nn.Linear(hidden, self.n_experts))
            nn.init.zeros_(self.gate[-1].weight)        # start as the plain average ensemble
            nn.init.zeros_(self.gate[-1].bias)

    def train(self, mode: bool = True):
        super().train(mode)
        if self.freeze_branches:                        # frozen experts keep their BN statistics / no dropout
            self.branches.eval()
        return self

    def branch_weights(self, feats, logits, x=None) -> torch.Tensor:
        """[B, E] mixture weights. `x` (the input batch) is needed only for the 'image' gate input."""
        probs = [F.softmax(z.float(), dim=1) for z in logits]
        if self.mode == "average":
            return torch.full((probs[0].shape[0], len(probs)), 1.0 / len(probs), device=probs[0].device)
        if self.mode == "confidence":
            conf = torch.stack([p.amax(dim=1) for p in probs], dim=1)
            return conf / conf.sum(dim=1, keepdim=True)
        parts = []
        if "features" in self.gate_inputs:
            k = len(self.branches)
            parts += [self.feat_norms[i % k](f.float()) for i, f in enumerate(feats)]
        if "probs" in self.gate_inputs:
            parts += probs
        if "stats" in self.gate_inputs:
            parts.append(expert_stats(probs))
        if "image" in self.gate_inputs:
            if x is None:
                raise ValueError("this gate uses image statistics: pass the input batch to branch_weights")
            parts.append(image_stats(x))
        return F.softmax(self.gate(torch.cat(parts, dim=1)), dim=1)

    def forward_detailed(self, x) -> dict:
        """logits: fused log p. weights: [B, E]. branch_logits: the E expert logits (view by view).
        features: the branches' pooled features on the first view."""
        outs = []
        for name in self.views:
            xv = VIEWS[name](x)
            outs += [b.features_logits(xv) for b in self.branches]
        feats, logits = [o[0] for o in outs], [o[1] for o in outs]
        w = self.branch_weights(feats, logits, x)
        logp = torch.stack([F.log_softmax(z.float(), dim=1) for z in logits], dim=1)      # [B, E, C]
        fused = torch.logsumexp(logp + torch.log(w.clamp_min(1e-8)).unsqueeze(-1), dim=1)
        return {"logits": fused, "weights": w, "branch_logits": logits, "features": torch.cat(
            [f.float() for f in feats[:len(self.branches)]], dim=1)}

    def features_logits(self, x):
        out = self.forward_detailed(x)
        return out["features"], out["logits"]

    def cam_targets(self):
        return [t for b in self.branches for t in b.cam_targets()]

    # Frozen branches are restored from their own runs, so a fusion checkpoint holds only the gate.
    def checkpoint_state(self) -> dict:
        state = self.state_dict()
        if self.freeze_branches:
            state = {k: v for k, v in state.items() if not k.startswith("branches.")}
        return state

    def load_checkpoint_state(self, state: dict) -> None:
        missing, unexpected = self.load_state_dict(state, strict=False)
        bad = [k for k in missing if not (self.freeze_branches and k.startswith("branches."))]
        if bad or unexpected:
            raise RuntimeError(f"fusion checkpoint mismatch: missing={bad} unexpected={list(unexpected)}")


In [ ]:
%%writefile /content/RoadLens/roadlens/models/smallcnn.py
"""SmallCNN: the from-scratch baseline, and the model whose errors normalise mCE."""

from __future__ import annotations

import torch.nn as nn

from .base import CamTarget, Classifier, Preprocess


def _block(cin: int, cout: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
    )


class SmallCNN(Classifier):
    """Four conv-BN-ReLU x2 stages (width, 2w, 4w, 8w) with max-pooling, global average pooling,
    dropout and a linear head. 1.2 M parameters at width 32."""

    def __init__(self, num_classes: int = 43, width: int = 32, dropout: float = 0.3, input_norm: str = "dataset"):
        super().__init__()
        chans = [width, width * 2, width * 4, width * 8]
        self.pre = Preprocess(None, norm=input_norm)
        stages, cin = [], 3
        for i, c in enumerate(chans):
            stages.append(_block(cin, c))
            if i < len(chans) - 1:
                stages.append(nn.MaxPool2d(2))
            cin = c
        self.features = nn.Sequential(*stages)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(cin, num_classes)
        self.feature_dim = cin

    def features_logits(self, x):
        feat = self.pool(self.features(self.pre(x))).flatten(1)
        return feat, self.fc(self.drop(feat))

    def cam_targets(self):
        return [CamTarget("smallcnn", self.features[-1], "conv")]


In [ ]:
%%writefile /content/RoadLens/roadlens/models/timm_models.py
"""ImageNet-pretrained backbones from timm (ResNet-18, DeiT-S, Swin-T, ...), fine-tuned end to end."""

from __future__ import annotations

import timm

from .base import IMAGENET_MEAN, IMAGENET_STD, CamTarget, Classifier, Preprocess


class TimmClassifier(Classifier):
    """`name` is any timm model. The 64 px input is upsampled to `input_size` before the backbone:
    pretrained filters and patch embeddings expect objects at roughly ImageNet scale, and a ViT with
    16 px patches would otherwise see only 4 x 4 tokens."""

    def __init__(self, name: str, num_classes: int = 43, pretrained: bool = True,
                 input_size: int = 128, drop_path: float = 0.0, input_norm: str = "dataset"):
        super().__init__()
        kwargs = dict(pretrained=pretrained, num_classes=num_classes)
        if drop_path:
            kwargs["drop_path_rate"] = drop_path
        try:        # transformers need the input size up front (position embeddings are resampled)
            self.backbone = timm.create_model(name, img_size=input_size, **kwargs)
        except TypeError:   # CNNs accept any size
            self.backbone = timm.create_model(name, **kwargs)
        pcfg = getattr(self.backbone, "pretrained_cfg", None) or {}
        self.pre = Preprocess(input_size, pcfg.get("mean", IMAGENET_MEAN), pcfg.get("std", IMAGENET_STD), input_norm)
        self.feature_dim = self.backbone.num_features
        self.arch = name

    def features_logits(self, x):
        feats = self.backbone.forward_features(self.pre(x))
        pooled = self.backbone.forward_head(feats, pre_logits=True)
        return pooled, self.backbone.get_classifier()(pooled)

    def cam_targets(self):
        b = self.backbone
        if hasattr(b, "layer4"):                                    # ResNet family
            return [CamTarget(self.arch, b.layer4, "conv")]
        if hasattr(b, "blocks") and hasattr(b, "num_prefix_tokens") and hasattr(b.blocks[-1], "norm1"):
            return [CamTarget(self.arch, b.blocks[-1].norm1, "tokens", b.num_prefix_tokens)]   # ViT / DeiT
        return []


In [ ]:
%%writefile /content/RoadLens/roadlens/models/views.py
"""Deterministic input views and per-image statistics for view-expert fusion.

A view is a fixed, parameter-free transform of the input image. The fusion evaluates every branch on
every view and mixes the resulting "experts". The two low-pass views trade pixel noise for a little
blur: the models here are measurably robust to blur and fragile to noise, so a smoothed copy of a
noisy image is an easier input than the original.

Nothing here is learned and no view is a training augmentation. Which views a model uses is a
hyper-parameter, to be chosen on validation-C like any other.
"""

from __future__ import annotations

import math

import torch
import torch.nn.functional as F


def raw(x: torch.Tensor) -> torch.Tensor:
    return x


def median3(x: torch.Tensor) -> torch.Tensor:
    """3 x 3 median filter per channel (replicated border). Removes isolated outliers, keeps edges."""
    b, c, h, w = x.shape
    patches = F.unfold(F.pad(x, (1, 1, 1, 1), mode="replicate"), kernel_size=3)       # [B, C * 9, H * W]
    return patches.view(b, c, 9, h * w).median(dim=2).values.view(b, c, h, w)


def gauss(x: torch.Tensor, sigma: float = 1.0) -> torch.Tensor:
    """Separable Gaussian blur with standard deviation `sigma` pixels (reflected border)."""
    radius = max(1, int(math.ceil(2.0 * sigma)))
    t = torch.arange(-radius, radius + 1, device=x.device, dtype=x.dtype)
    k = torch.exp(-0.5 * (t / sigma) ** 2)
    k = k / k.sum()
    c = x.shape[1]
    x = F.conv2d(F.pad(x, (radius, radius, 0, 0), mode="reflect"), k.view(1, 1, 1, -1).repeat(c, 1, 1, 1), groups=c)
    return F.conv2d(F.pad(x, (0, 0, radius, radius), mode="reflect"), k.view(1, 1, -1, 1).repeat(c, 1, 1, 1), groups=c)


VIEWS = {"raw": raw, "median3": median3, "gauss": gauss}

N_IMAGE_STATS = 5


def image_stats(x: torch.Tensor) -> torch.Tensor:
    """[B, 5] cheap descriptors of how degraded an image looks, all of order 1:

    mean luminance, luminance contrast (std x 4), high-frequency energy (mean |L - blur(L)| x 10),
    the same energy relative to the contrast, and mean chroma (max - min over channels, x 2).
    Pixel noise raises the high-frequency terms, blur lowers them, haze lowers the contrast.
    """
    x = x.float()
    lum = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
    mean = lum.mean(dim=(1, 2, 3))
    std = lum.std(dim=(1, 2, 3), correction=0)
    smooth = F.avg_pool2d(F.pad(lum, (1, 1, 1, 1), mode="replicate"), kernel_size=3, stride=1)
    hf = (lum - smooth).abs().mean(dim=(1, 2, 3))
    chroma = (x.amax(dim=1) - x.amin(dim=1)).mean(dim=(1, 2))
    return torch.stack([mean, 4.0 * std, 10.0 * hf, hf / (std + 0.02), 2.0 * chroma], dim=1)


def expert_stats(probs: list[torch.Tensor]) -> torch.Tensor:
    """[B, 3 E + 1] confidence descriptors of E expert distributions, all in [0, 1]:

    per expert its top probability, top-1 minus top-2 margin and normalised entropy, then the share
    of expert pairs that predict the same class.
    """
    out = []
    for p in probs:
        top2 = p.topk(2, dim=1).values
        ent = -(p * p.clamp_min(1e-8).log()).sum(dim=1) / math.log(p.shape[1])
        out += [top2[:, 0], top2[:, 0] - top2[:, 1], ent]
    pred = torch.stack([p.argmax(dim=1) for p in probs], dim=1)                       # [B, E]
    out.append((pred[:, :, None] == pred[:, None, :]).float().mean(dim=(1, 2)))
    return torch.stack(out, dim=1)


In [ ]:
%%writefile /content/RoadLens/roadlens/runs.py
"""Named training runs: configs/runs/<name>.yaml -> <runs_root>/<name>/seed<k>/.

Experiments never train or evaluate directly; they ask for (run, seed) and this module trains it
if needed (resumable), loads it, and caches every evaluation next to the checkpoint. That is what
lets the six experiments share models: e.g. `resnet18_full` is trained once per seed and reused by
the corruption table, the ablation, the leave-category-out study, the fusion and the TTA sweep.
"""

from __future__ import annotations

import hashlib
import json
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

from .data.loading import corrupted_fingerprint, corrupted_sets_missing, load_selection_data, make_c_command
from .engine import train_run
from .eval.calibration import fit_temperature, nll_at
from .eval.evaluate import build_sets, evaluate_sets, predict_logits, predict_stream
from .models import build_model
from .tta import build_tta
from .utils import REPO_ROOT, apply_profile, get_device, get_logger, load_config, runs_root, scratch_root

log = get_logger()
RUN_CONFIG_DIR = REPO_ROOT / "configs" / "runs"


def load_run_config(name: str, profile: str | None = None) -> dict:
    path = RUN_CONFIG_DIR / f"{name}.yaml"
    if not path.exists():
        raise FileNotFoundError(f"no run config {path}")
    cfg = load_config(path)
    cfg["name"] = name
    return apply_profile(cfg, profile)


def run_dir(cfg: dict, seed: int) -> Path:
    return runs_root(cfg) / cfg["name"] / f"seed{seed}"


def fingerprint(cfg: dict) -> str:
    """Hash of everything that determines the trained weights (including the branch runs of a fusion)."""
    train = {k: v for k, v in cfg["train"].items() if k != "eval_batch_size"}
    data = {k: v for k, v in cfg["data"].items() if k != "test_subset"}
    spec = {"model": cfg["model"], "train": train, "augment": cfg["augment"], "data": data,
            "val_corruptions": [cfg["corruptions"]["val"], cfg["corruptions"]["val_severities"], cfg["corruptions"]["seed"]],
            "branches": [fingerprint(load_run_config(b, cfg.get("profile"))) for b in cfg["model"].get("branches", [])]}
    # v3 dependencies enter the hash only when used, so the fingerprints of older runs do not change
    teacher = (cfg["train"].get("distill") or {}).get("teacher")
    if teacher:
        spec["teacher"] = fingerprint(load_run_config(teacher, cfg.get("profile")))
    if cfg.get("ssl"):
        spec["ssl"] = cfg["ssl"]
    if cfg["model"].get("init_from"):
        spec["init_from"] = fingerprint(load_run_config(cfg["model"]["init_from"], cfg.get("profile")))
    return hashlib.sha1(json.dumps(spec, sort_keys=True, default=str).encode()).hexdigest()[:16]


def _check_fingerprint(cfg: dict, rd: Path) -> None:
    fp_file = rd / "fingerprint.txt"
    fp = fingerprint(cfg)
    if fp_file.exists() and fp_file.read_text().strip() != fp:
        raise RuntimeError(
            f"{rd} was trained with a different configuration than configs/runs/{cfg['name']}.yaml "
            f"(or its branches / base configs) now describe. Delete that directory to retrain, or give the "
            f"changed configuration a new run name.")
    rd.mkdir(parents=True, exist_ok=True)
    fp_file.write_text(fp)


def _build(cfg: dict, seed: int, device, for_training: bool):
    branches = [load_model(b, seed, device, cfg.get("profile"))[0] for b in cfg["model"].get("branches", [])] or None
    model = build_model(cfg["model"], cfg["data"]["num_classes"], branches, pretrained=None if for_training else False)
    return model.to(device)


def ensure_trained(name: str, seed: int, profile: str | None = None, device=None, use_wandb: bool = False) -> dict:
    """Train run `name` with `seed` unless it is already finished. Returns its done.json."""
    cfg = load_run_config(name, profile)
    device = device or get_device()
    rd = run_dir(cfg, seed)
    _check_fingerprint(cfg, rd)
    done = rd / "done.json"
    if done.exists():
        return json.loads(done.read_text())
    for b in cfg["model"].get("branches", []):
        ensure_trained(b, seed, profile, device, use_wandb)
    teacher_name = (cfg["train"].get("distill") or {}).get("teacher")
    init_from = cfg["model"].get("init_from")
    for dep in (teacher_name, init_from):
        if dep:
            ensure_trained(dep, seed, profile, device, use_wandb)
    (rd / "config.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
    model = _build(cfg, seed, device, for_training=True)
    if init_from:
        _init_from(model, init_from, seed, profile, device)
    scratch = scratch_root(cfg)
    scratch_dir = scratch / cfg["name"] / f"seed{seed}" if scratch else None
    if cfg["train"].get("objective", "supervised") == "simclr":
        from .ssl import train_simclr
        info = train_simclr(cfg, seed, rd, model, device, scratch_dir=scratch_dir, fingerprint=fingerprint(cfg))
    else:
        teacher = load_model(teacher_name, seed, device, profile)[0] if teacher_name else None
        info = train_run(cfg, seed, rd, model, device, use_wandb, scratch_dir=scratch_dir,
                         fingerprint=fingerprint(cfg), teacher=teacher)
        del teacher
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return info


def _classifier_prefixes(model) -> list[str]:
    """Parameter-name prefixes of the classification head(s) of a model (not transferred by init_from)."""
    heads = []
    backbone = getattr(model, "backbone", None)
    if backbone is not None and hasattr(backbone, "get_classifier"):
        heads.append(backbone.get_classifier())
    if hasattr(model, "head"):
        heads.append(model.head)
    return [name + "." for name, m in model.named_modules() if any(m is h for h in heads) and name]


def _init_from(model, run: str, seed: int, profile: str | None, device) -> None:
    """Initialise `model` with the weights of a finished run (e.g. self-supervised pre-training), except
    the classification head and any parameter whose shape differs."""
    cfg = load_run_config(run, profile)
    state = torch.load(run_dir(cfg, seed) / "best.pt", map_location=device, weights_only=False)["model"]
    own = model.state_dict()
    skip = tuple(_classifier_prefixes(model))
    keep = {k: v for k, v in state.items() if k in own and v.shape == own[k].shape and not k.startswith(skip)}
    model.load_state_dict(keep, strict=False)
    log.info(f"initialised {len(keep)} of {len(own)} tensors from {run} seed {seed}")


def load_model(name: str, seed: int, device=None, profile: str | None = None):
    """Return (model in eval mode with its best weights, config). The run must be trained."""
    cfg = load_run_config(name, profile)
    device = device or get_device()
    rd = run_dir(cfg, seed)
    if not (rd / "done.json").exists():
        raise FileNotFoundError(f"run {name} seed {seed} is not trained yet ({rd}). "
                                f"Run: python scripts/train.py --run {name} --seeds {seed}")
    _check_fingerprint(cfg, rd)
    model = _build(cfg, seed, device, for_training=False)
    best = rd / "best.pt"
    if best.exists():
        model.load_checkpoint_state(torch.load(best, map_location=device, weights_only=False)["model"])
    return model.eval(), cfg


def eval_tag(suite: str, tta: dict | None = None, severities=None, n_per_set: int = 0, calibrate: bool = False) -> str:
    tag = suite
    if tta and tta.get("method", "none") != "none":
        tag += f"__{tta['method']}_bs{tta['batch_size']}"
        if tta["method"] == "tent":
            tag += f"_lr{float(tta.get('lr', 1e-3)):g}" + ("_fused" if tta.get("objective") == "fused" else "")
        if float(tta.get("prior", 0) or 0) > 0:
            tag += f"_p{float(tta['prior']):g}"
    if severities:
        tag += "__sev" + "".join(str(s) for s in severities)
    if n_per_set:
        tag += f"__n{n_per_set}"
    if calibrate:
        tag += "__cal"
    return tag


def temperature(name: str, seed: int, profile: str | None = None, device=None, force: bool = False) -> float:
    """Post-hoc temperature of (run, seed): fitted once and cached in <run dir>/temperature.json.

    It is fitted on the select half of the validation split, clean and as validation-C in equal
    parts (the half that no stacked gate was trained on), so every model is calibrated on the same
    held-out images and no test image is involved.
    """
    cfg = load_run_config(name, profile)
    device = device or get_device()
    path = run_dir(cfg, seed) / "temperature.json"
    if path.exists() and not force:
        return float(json.loads(path.read_text())["temperature"])
    model, cfg = load_model(name, seed, device, profile)
    kw = dict(device=device, amp=bool(cfg["train"].get("amp", True)), batch_size=int(cfg["train"].get("eval_batch_size", 512)))
    xv, yv, xc, yc = load_selection_data(cfg)
    logits = torch.cat([predict_logits(model, xv, **kw), predict_logits(model, xc, **kw)])
    y = torch.from_numpy(np.concatenate([yv, yc])).long()
    t = fit_temperature(logits, y)
    path.write_text(json.dumps({"temperature": t, "n_images": int(len(y)), "nll_before": nll_at(logits, y, 1.0),
                                "nll_after": nll_at(logits, y, t),
                                "fitted_on": "select half of the validation split: clean + validation-C"}, indent=1))
    log.info(f"[{name} seed {seed}] temperature {t:.3f} (NLL {nll_at(logits, y, 1.0):.4f} -> {nll_at(logits, y, t):.4f} "
             f"on {len(y)} held-out images)")
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return t


def evaluate_run(name: str, seed: int, suite: str, profile: str | None = None, device=None, tta: dict | None = None,
                 severities=None, n_per_set: int = 0, force: bool = False, calibrate: bool = False):
    """Evaluate (run, seed) on a suite, with caching.

    tta: None, or {"method": "bn" | "tent", "batch_size": int, "lr": float, "objective": "branch" | "fused",
         "prior": float}.
    calibrate: divide the logits by the run's post-hoc temperature (see `temperature`).
    Returns (DataFrame, {"confusion": [S, K, K], "bins": [S, 15, 3]}); rows and arrays share their order.
    """
    cfg = load_run_config(name, profile)
    device = device or get_device()
    out_dir = run_dir(cfg, seed) / "eval"
    tag = eval_tag(suite, tta, severities, n_per_set, calibrate)
    csv_path, npz_path = out_dir / f"{tag}.csv", out_dir / f"{tag}.npz"
    # A cached evaluation of corrupted sets is only valid for the sets it was computed on: if the
    # corruption parameters change, GTSRB-C is regenerated with a new fingerprint and old caches are redone.
    data_fp = corrupted_fingerprint(cfg, suite[:-2]) if suite.endswith("_c") else ""
    if csv_path.exists() and npz_path.exists() and not force:
        with np.load(npz_path) as z:
            cached_fp = str(z["data_fp"]) if "data_fp" in z.files else ""
            if cached_fp == data_fp:
                return pd.read_csv(csv_path), {"confusion": z["confusion"], "bins": z["bins"]}
        log.info(f"{name} seed {seed}: {tag} was computed on other corrupted sets; evaluating again")

    if suite.endswith("_c"):
        missing = corrupted_sets_missing(cfg, suite[:-2])
        if missing:
            raise FileNotFoundError(
                f"{len(missing)} of the corrupted {suite[:-2]} test sets are missing or still being generated "
                f"(first: {missing[0]}). Run, or wait for: {make_c_command(cfg, suite[:-2])}")
    temp = temperature(name, seed, profile, device) if calibrate else 1.0
    model, cfg = load_model(name, seed, device, profile)
    amp = bool(cfg["train"].get("amp", True))
    if tta and tta.get("method", "none") != "none":
        adapter = build_tta(tta["method"], model, amp=amp, lr=float(tta.get("lr", 1e-3)),
                            objective=tta.get("objective", "branch"), prior=float(tta.get("prior", 0) or 0))
        predict = partial(predict_stream, adapter, device=device, batch_size=int(tta["batch_size"]))
    else:
        predict = partial(predict_logits, model, device=device, amp=amp,
                          batch_size=int(cfg["train"].get("eval_batch_size", 512)))
    if calibrate:
        uncalibrated = predict

        def predict(x):
            return uncalibrated(x) / temp
    sets = build_sets(cfg, suite, severities)
    df, confs, bins = evaluate_sets(predict, sets, cfg["data"]["num_classes"], n_per_set,
                                    desc=f"{name} seed {seed}: {tag}")
    out_dir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(npz_path, confusion=confs, bins=bins, data_fp=data_fp)
    df.to_csv(csv_path, index=False)
    del model, predict
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return df, {"confusion": confs, "bins": bins}


In [ ]:
%%writefile /content/RoadLens/roadlens/ssl.py
"""Self-supervised pre-training (RoadLens v3): SimCLR on the GTSRB training images, labels unused.

Two augmented views of every image (the same AugMix / PRIME / APR ops as the supervised runs, none
of them a test corruption) are pulled together and pushed away from the other images of the batch
with the NT-Xent loss (Chen et al. 2020), through a small projection head on the pooled features.
The result is a backbone whose features are invariant to the training perturbations; a supervised
run then starts from it (`model.init_from: <this run>`).

No label is used for training or for choosing the checkpoint (the last epoch is kept). Labels are
used only to monitor the representation: a weighted k-nearest-neighbour classifier on the features
(k = 20, cosine similarity, Wu et al. 2018) scores the clean validation split and validation-C each
epoch, and the score is logged.
"""

from __future__ import annotations

import csv
import json
import os
import shutil
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

from .augment import TrainAugment
from .data.loading import as_float, load_split, load_val_c_split, u8_to_device
from .engine import lr_factor, param_groups
from .utils import get_logger, seed_everything

log = get_logger()
SSL_LOG_FIELDS = ["epoch", "lr", "loss", "knn_val_acc", "knn_val_c_acc", "epoch_s", "img_per_s"]


def nt_xent(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.2) -> torch.Tensor:
    """NT-Xent loss of two batches of embeddings [B, D]; row i of z1 and row i of z2 are the positive pair."""
    z = F.normalize(torch.cat([z1, z2]).float(), dim=1)
    n = z1.shape[0]
    sim = z @ z.t() / temperature
    sim.fill_diagonal_(float("-inf"))
    target = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, target)


@torch.no_grad()
def embed(model, x_u8: torch.Tensor, batch_size: int = 512, amp: bool = True) -> torch.Tensor:
    """L2-normalised pooled features of a uint8 NCHW tensor on the model's device."""
    model.eval()
    out = []
    for s in range(0, len(x_u8), batch_size):
        with torch.autocast(x_u8.device.type, enabled=amp):
            out.append(model.features_logits(as_float(x_u8[s:s + batch_size]))[0].float())
    return F.normalize(torch.cat(out), dim=1)


@torch.no_grad()
def knn_accuracy(bank: torch.Tensor, bank_y: torch.Tensor, query: torch.Tensor, query_y: torch.Tensor,
                 num_classes: int, k: int = 20, tau: float = 0.07) -> float:
    """Weighted kNN accuracy: each of the k most similar bank features votes exp(sim / tau) for its class."""
    correct = 0
    for s in range(0, len(query), 1024):
        sim = query[s:s + 1024] @ bank.t()
        top, idx = sim.topk(k, dim=1)
        votes = torch.zeros(len(top), num_classes, device=bank.device)
        votes.scatter_add_(1, bank_y[idx], torch.exp(top / tau))
        correct += int((votes.argmax(dim=1) == query_y[s:s + 1024]).sum())
    return correct / len(query)


def train_simclr(cfg: dict, seed: int, run_dir: Path, model, device, scratch_dir: Path | None = None,
                 fingerprint: str | None = None) -> dict:
    """Pre-train `model` (a Classifier) with SimCLR. Leaves best.pt (= last epoch), log.csv, done.json."""
    run_dir.mkdir(parents=True, exist_ok=True)
    work = Path(scratch_dir) if scratch_dir else run_dir
    work.mkdir(parents=True, exist_ok=True)
    done_path, log_path, final_best, last_path = run_dir / "done.json", run_dir / "log.csv", run_dir / "best.pt", work / "last.pt"
    if done_path.exists():
        return json.loads(done_path.read_text())

    t, s = cfg["train"], cfg.get("ssl", {})
    seed_everything(seed)
    cuda = device.type == "cuda"
    amp = bool(t.get("amp", True)) and cuda
    if t.get("cudnn_benchmark") and cuda:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
    xtr, ytr = load_split(cfg, "train")
    xva, yva = load_split(cfg, "val")
    xvc, yvc, _, _ = load_val_c_split(cfg)
    Xtr, Ytr = u8_to_device(xtr, device), torch.from_numpy(ytr).to(device)
    Xva, Yva = u8_to_device(xva, device), torch.from_numpy(yva).to(device)
    Xvc, Yvc = u8_to_device(xvc, device), torch.from_numpy(yvc).to(device)
    bank_idx = torch.randperm(len(Xtr), generator=torch.Generator().manual_seed(seed))[:int(s.get("knn_bank", 10000))]
    bank_idx = bank_idx.to(device)

    proj = nn.Sequential(nn.Linear(model.feature_dim, int(s.get("proj_hidden", 512))), nn.ReLU(inplace=True),
                         nn.Linear(int(s.get("proj_hidden", 512)), int(s.get("proj_dim", 128)))).to(device)
    params = param_groups(model, float(t["weight_decay"])) + param_groups(proj, float(t["weight_decay"]))
    opt = torch.optim.AdamW(params, lr=float(t["lr"]), **({"fused": True} if t.get("fused_optimizer") and cuda else {}))
    bs, epochs = int(t["batch_size"]), int(t["epochs"])
    steps = len(Xtr) // bs
    total, warmup = epochs * steps, int(t.get("warmup_epochs", 0) * steps)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda i: lr_factor(i, total, warmup))
    scaler = torch.amp.GradScaler(device.type, enabled=amp)
    aug = TrainAugment({**cfg["augment"], "mode": "jsd"})          # [jitter(x), augmix(.), augmix(.)]
    temperature = float(s.get("temperature", 0.2))

    start, elapsed, knn = 1, 0.0, (0.0, 0.0)
    if last_path.exists():
        ck = torch.load(last_path, map_location=device, weights_only=False)
        if ck.get("fingerprint") == fingerprint:
            model.load_checkpoint_state(ck["model"])
            proj.load_state_dict(ck["proj"])
            opt.load_state_dict(ck["opt"])
            sched.load_state_dict(ck["sched"])
            scaler.load_state_dict(ck["scaler"])
            start, elapsed = ck["epoch"] + 1, ck["elapsed"]
            log.info(f"[{cfg['name']} seed {seed}] resuming SimCLR after epoch {ck['epoch']}")
    rows = []
    if log_path.exists() and start > 1:
        with open(log_path, newline="") as f:
            rows = [r for r in csv.DictReader(f) if int(r["epoch"]) < start]
    with open(log_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=SSL_LOG_FIELDS)
        w.writeheader()
        w.writerows(rows)
    if cuda:
        torch.cuda.reset_peak_memory_stats(device)
    log.info(f"[{cfg['name']} seed {seed}] SimCLR: {len(Xtr)} unlabelled images, {steps} steps/epoch, "
             f"batch {bs} x 2 views, temperature {temperature}")

    for epoch in range(start, epochs + 1):
        model.train()
        proj.train()
        t0, tot = time.time(), 0.0
        perm = torch.randperm(len(Xtr), device=device)
        for i in tqdm(range(steps), desc=f"{cfg['name']} s{seed} ep {epoch}/{epochs}", leave=False, mininterval=2.0):
            views = aug(as_float(Xtr[perm[i * bs:(i + 1) * bs]]))[1:]
            with torch.autocast(device.type, enabled=amp):
                feats = model.features_logits(torch.cat(views))[0]
                z = proj(feats)
            loss = nt_xent(*z.float().chunk(2), temperature=temperature)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"non-finite SimCLR loss at epoch {epoch}, step {i}")
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(proj.parameters()), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item()
        epoch_s = time.time() - t0
        elapsed += epoch_s
        bank = embed(model, Xtr[bank_idx], amp=amp)
        k = int(cfg["data"]["num_classes"])
        knn = (knn_accuracy(bank, Ytr[bank_idx], embed(model, Xva, amp=amp), Yva, k),
               knn_accuracy(bank, Ytr[bank_idx], embed(model, Xvc, amp=amp), Yvc, k))
        row = {"epoch": epoch, "lr": f"{opt.param_groups[0]['lr']:.3e}", "loss": f"{tot / steps:.4f}",
               "knn_val_acc": f"{knn[0]:.4f}", "knn_val_c_acc": f"{knn[1]:.4f}", "epoch_s": f"{epoch_s:.1f}",
               "img_per_s": f"{steps * bs / epoch_s:.0f}"}
        with open(log_path, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=SSL_LOG_FIELDS).writerow(row)
        log.info(f"[{cfg['name']} seed {seed}] epoch {epoch:3d}  NT-Xent {row['loss']}  kNN val {row['knn_val_acc']}  "
                 f"kNN val-C {row['knn_val_c_acc']}  ({epoch_s:.0f}s, {row['img_per_s']} img/s)")
        tmp = last_path.with_suffix(".tmp")
        torch.save({"model": model.checkpoint_state(), "proj": proj.state_dict(), "opt": opt.state_dict(),
                    "sched": sched.state_dict(), "scaler": scaler.state_dict(), "epoch": epoch, "elapsed": elapsed,
                    "fingerprint": fingerprint}, tmp)
        os.replace(tmp, last_path)

    tmp = final_best.with_suffix(".pt.tmp")
    torch.save({"model": model.checkpoint_state(), "epoch": epochs, "knn_val_c_acc": knn[1]}, tmp)
    os.replace(tmp, final_best)
    info = {"name": cfg["name"], "seed": seed, "skipped": False, "objective": "simclr",
            "n_params": sum(p.numel() for p in model.parameters()), "n_trainable": sum(p.numel() for p in model.parameters()),
            "epochs_run": epochs, "best_epoch": epochs, "best_val_c_acc": knn[1], "val_acc_at_best": knn[0],
            "train_time_s": round(elapsed, 1), "early_stopped": False,
            "peak_gpu_mem_mb": round(torch.cuda.max_memory_allocated(device) / 2 ** 20) if cuda else None,
            "note": "best_val_c_acc / val_acc_at_best are kNN monitoring scores; the last epoch is kept"}
    done_path.write_text(json.dumps(info, indent=1))
    last_path.unlink(missing_ok=True)
    if work != run_dir:
        shutil.rmtree(work, ignore_errors=True)
    return info


In [ ]:
%%writefile /content/RoadLens/roadlens/tta/__init__.py
from .methods import BNAdapt, NoAdapt, PriorBatchNorm, Tent, batchnorm_layers, build_tta, entropy  # noqa: F401


In [ ]:
%%writefile /content/RoadLens/roadlens/tta/methods.py
"""Test-time adaptation on an unlabeled test stream.

  NoAdapt  the source model, unchanged.
  BNAdapt  plain test-time BatchNorm: every batch is normalised with its own statistics instead of
           the training-set running statistics (Nado et al. 2020).
  Tent     BNAdapt + one entropy-minimisation step per batch on the BatchNorm scale/shift
           parameters only (Wang et al. 2021).

Both adaptive methods take `prior` (default 0): with prior = N > 0 the batch statistics are mixed
with the source statistics, weighted n : N for a batch of n images (Schneider et al. 2020). That
keeps the methods usable at batch sizes where the statistics of one batch alone are too noisy.

Both adaptive methods touch BatchNorm layers only, i.e. the CNN parts of a model: all of SmallCNN
and ResNet-18, and the ResNet branch of the fusion. The DeiT branch has no BatchNorm and stays
fixed. For the fusion, Tent minimises the entropy of the CNN branch's own prediction by default
(`objective="branch"`); `objective="fused"` uses the entropy of the fused prediction instead.

The adaptive methods work on a private copy of the model, so the caller's model is never modified.
`predict` returns the (fused) logits computed *before* the update of that batch, as in the Tent
reference code, and `reset` restores the source state (call it between test distributions).
"""

from __future__ import annotations

from copy import deepcopy

import torch
import torch.nn as nn
import torch.nn.functional as F


def batchnorm_layers(model: nn.Module) -> list[nn.modules.batchnorm._BatchNorm]:
    return [m for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)]


class PriorBatchNorm(nn.Module):
    """A BatchNorm layer that normalises with a mixture of its source statistics and the batch's:

        statistic = (N * source + n * batch) / (N + n)

    with N = `prior` pseudo-samples and n = the batch size (Schneider et al. 2020). A batch of one
    image then moves the statistics by 1 / (N + 1) instead of replacing them, which is what makes
    plain test-time BatchNorm collapse at small batch sizes. The wrapped layer keeps its weight,
    bias and stored statistics.
    """

    def __init__(self, bn: nn.modules.batchnorm._BatchNorm, prior: float):
        super().__init__()
        self.bn, self.prior = bn, float(prior)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Written out instead of calling F.batch_norm: the mixed statistics depend on the batch, and
        # Tent has to back-propagate through them (as it does through plain batch statistics).
        bn = self.bn
        dims = [d for d in range(x.dim()) if d != 1]
        shape = [1, -1] + [1] * (x.dim() - 2)
        xf = x.float()
        a = x.shape[0] / (x.shape[0] + self.prior)
        mean = a * xf.mean(dim=dims) + (1 - a) * bn.running_mean
        var = a * xf.var(dim=dims, correction=0) + (1 - a) * bn.running_var
        out = (xf - mean.view(shape)) / torch.sqrt(var.view(shape) + bn.eps)
        if bn.weight is not None:
            out = out * bn.weight.view(shape) + bn.bias.view(shape)
        return out.to(x.dtype)


def _use_batch_statistics(model: nn.Module, prior: float = 0.0) -> list:
    """Make the BatchNorm layers normalise with test-batch statistics; returns those layers.

    prior = 0: the layers go to training mode with momentum 0, so they use the statistics of the
    current batch alone and leave the stored source statistics untouched.
    prior > 0: every layer is wrapped in PriorBatchNorm (source statistics as a prior of that weight).
    """
    layers = batchnorm_layers(model)
    if prior > 0:
        plain = (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)
        odd = sorted({type(m).__name__ for m in layers if type(m) not in plain})
        if odd:
            raise NotImplementedError(f"a statistics prior is implemented for plain BatchNorm layers only, found {odd}")
        for parent in list(model.modules()):
            for name, child in list(parent.named_children()):
                if type(child) in plain:
                    setattr(parent, name, PriorBatchNorm(child, prior))
        return layers
    for m in layers:
        m.train()
        m.momentum = 0.0
    return layers


def entropy(logits: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits.float(), dim=1)
    return -(logp.exp() * logp).sum(dim=1)


class NoAdapt:
    name = "none"

    def __init__(self, model: nn.Module, amp: bool = True):
        self.model = model.eval()
        self.amp = amp

    def reset(self) -> None:
        pass

    @torch.no_grad()
    def predict(self, x: torch.Tensor) -> torch.Tensor:
        with torch.autocast(x.device.type, enabled=self.amp and x.device.type == "cuda"):
            return self.model(x).float()


class BNAdapt(NoAdapt):
    name = "bn"

    def __init__(self, model: nn.Module, amp: bool = True, prior: float = 0.0):
        super().__init__(deepcopy(model), amp)
        if not _use_batch_statistics(self.model, prior):
            raise ValueError("model has no BatchNorm layers: test-time BN adaptation does not apply")


class Tent:
    name = "tent"

    def __init__(self, model: nn.Module, lr: float = 1e-3, steps: int = 1, amp: bool = True,
                 objective: str = "branch", prior: float = 0.0):
        if objective not in ("branch", "fused"):
            raise ValueError(f"tent objective must be branch | fused, got {objective!r}")
        self.model = deepcopy(model).eval()
        self.amp, self.steps = amp, int(steps)
        for p in self.model.parameters():
            p.requires_grad_(False)
        layers = _use_batch_statistics(self.model, prior)
        if not layers:
            raise ValueError("model has no BatchNorm layers: Tent does not apply")
        # For a fusion model, 'branch' minimises the entropy of each CNN branch's own prediction
        # (plain Tent on that branch); 'fused' minimises the entropy of the fused prediction, whose
        # gradient vanishes whenever the gate gives the CNN branch no weight.
        self.branch_ids = None
        if objective == "branch" and hasattr(self.model, "branches"):
            self.branch_ids = [k for k, b in enumerate(self.model.branches) if batchnorm_layers(b)]
        self.params = [p for m in layers for p in (m.weight, m.bias) if p is not None]
        for p in self.params:
            p.requires_grad_(True)
        self.opt = torch.optim.Adam(self.params, lr=lr, betas=(0.9, 0.999))
        self._source = [p.detach().clone() for p in self.params]
        self._opt_state = deepcopy(self.opt.state_dict())
        self._scaler = None

    def reset(self) -> None:
        with torch.no_grad():
            for p, src in zip(self.params, self._source):
                p.copy_(src)
        self.opt.load_state_dict(deepcopy(self._opt_state))
        self._scaler = None

    def predict(self, x: torch.Tensor) -> torch.Tensor:
        use_amp = self.amp and x.device.type == "cuda"
        if self._scaler is None:
            self._scaler = torch.amp.GradScaler(x.device.type, enabled=use_amp)
        first = None
        for _ in range(self.steps):
            with torch.enable_grad():
                with torch.autocast(x.device.type, enabled=use_amp):
                    if self.branch_ids is None:
                        logits = self.model(x)
                        loss = entropy(logits).mean()
                    else:
                        out = self.model.forward_detailed(x)
                        logits = out["logits"]
                        loss = sum(entropy(out["branch_logits"][k]).mean() for k in self.branch_ids) / len(self.branch_ids)
                self.opt.zero_grad(set_to_none=True)
                self._scaler.scale(loss).backward()
                self._scaler.step(self.opt)
                self._scaler.update()
            if first is None:
                first = logits.detach().float()
        return first


def build_tta(method: str, model: nn.Module, amp: bool = True, lr: float = 1e-3, steps: int = 1,
              objective: str = "branch", prior: float = 0.0):
    """method: none | bn | tent. prior: pseudo-sample weight of the source BatchNorm statistics (0 = batch only)."""
    if method == "none":
        return NoAdapt(model, amp)
    if method == "bn":
        return BNAdapt(model, amp, prior)
    if method == "tent":
        return Tent(model, lr=lr, steps=steps, amp=amp, objective=objective, prior=prior)
    raise ValueError(f"unknown test-time adaptation method: {method!r}")


In [ ]:
%%writefile /content/RoadLens/roadlens/utils.py
"""Config loading, paths, seeding and logging helpers."""

from __future__ import annotations

import copy
import logging
import os
import random
from pathlib import Path
from typing import Iterable

import numpy as np
import yaml

REPO_ROOT = Path(__file__).resolve().parents[1]


def _deep_merge(base: dict, update: dict) -> dict:
    out = copy.deepcopy(base)
    for k, v in update.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = _deep_merge(out[k], v)
        else:
            out[k] = copy.deepcopy(v)
    return out


def load_config(path: str | os.PathLike, overrides: Iterable[str] | None = None) -> dict:
    """Load a YAML config.

    `_base_: other.yaml` (path relative to the including file) is loaded first and deep-merged.
    `overrides` are `dotted.key=value` strings (value parsed as YAML), e.g. `data.image_size=32`.
    """
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    base = cfg.pop("_base_", None)
    if base is not None:
        bases = [base] if isinstance(base, str) else list(base)
        merged: dict = {}
        for b in bases:
            merged = _deep_merge(merged, load_config(path.parent / b))
        cfg = _deep_merge(merged, cfg)
    for item in overrides or []:
        if "=" not in item:
            raise ValueError(f"Override must look like key.sub=value, got: {item!r}")
        key, value = item.split("=", 1)
        node = cfg
        parts = key.split(".")
        for p in parts[:-1]:
            node = node.setdefault(p, {})
        node[parts[-1]] = yaml.safe_load(value)
    return cfg


def deep_merge(base: dict, update: dict) -> dict:
    return _deep_merge(base, update)


def _resolve_dir(value: str | os.PathLike) -> Path:
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = REPO_ROOT / path
    path.mkdir(parents=True, exist_ok=True)
    return path


def data_root(cfg: dict) -> Path:
    """Data directory: $ROADLENS_DATA if set, else cfg['data_root'] (relative to the repo root)."""
    return _resolve_dir(os.environ.get("ROADLENS_DATA") or cfg.get("data_root", "./datasets"))


def _profile_suffix(cfg: dict) -> str:
    return f"_{cfg['profile']}" if cfg.get("profile") else ""


def runs_root(cfg: dict) -> Path:
    """Checkpoints, logs and cached evaluations: $ROADLENS_RUNS if set, else cfg['runs_root'].
    A profile (smoke / pilot) gets its own directory, e.g. runs_pilot/."""
    root = os.environ.get("ROADLENS_RUNS") or cfg.get("runs_root", "./runs")
    return _resolve_dir(str(Path(root)) + _profile_suffix(cfg))


def scratch_root(cfg: dict) -> Path | None:
    """Optional folder for per-epoch training checkpoints: $ROADLENS_SCRATCH (None if unset).

    Set it to a local-disk folder when the runs directory is on a mounted Google Drive: every
    overwrite of a file there sends the previous version to the Drive trash, which keeps counting
    against the quota, so per-epoch checkpoints would silently fill the Drive. With a scratch folder
    only the final best weights are written to the runs directory, once per run.
    """
    root = os.environ.get("ROADLENS_SCRATCH")
    return _resolve_dir(str(Path(root)) + _profile_suffix(cfg)) if root else None


def results_root(cfg: dict) -> Path:
    """Tables and figures: cfg['results_root'] (relative to the repo root); results_pilot/ etc. for a profile."""
    return _resolve_dir(str(Path(cfg.get("results_root", "./results"))) + _profile_suffix(cfg))


PROFILE_DIR = REPO_ROOT / "configs" / "profiles"


def apply_profile(cfg: dict, profile: str | None) -> dict:
    """Overlay configs/profiles/<profile>.yaml (reduced-scale settings) on a config.

    smoke: tiny subsets, 2 epochs; checks that every script runs (minutes).
    pilot: full training set, one seed, shortened schedules, 2,000 test images; real but rough numbers.
    Runs and results of a profile live in their own *_<profile> directories, so they can never be
    mistaken for (or overwrite) the full experiments.
    """
    if not profile:
        return cfg
    path = PROFILE_DIR / f"{profile}.yaml"
    if not path.exists():
        raise FileNotFoundError(f"unknown profile {profile!r}: {path} does not exist")
    overlay = load_config(path)
    scale = overlay.get("train", {}).pop("epoch_scale", None)
    out = _deep_merge(cfg, overlay)
    if scale and "epochs" in cfg.get("train", {}):
        out["train"]["epochs"] = max(2, int(round(cfg["train"]["epochs"] * float(scale))))
    out["profile"] = profile
    return out


def corruption_names(cfg: dict) -> list[str]:
    """Flat list of the test corruptions, in category order."""
    return [name for names in cfg["corruptions"]["test"].values() for name in names]


def corruption_categories(cfg: dict) -> dict[str, str]:
    """corruption name -> category (noise / blur / weather / digital)."""
    return {name: cat for cat, names in cfg["corruptions"]["test"].items() for name in names}


def get_device(name: str | None = None):
    import torch

    if name:
        return torch.device(name)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed: int, deterministic: bool = True) -> None:
    """Seed python, numpy and torch (CPU + CUDA). With `deterministic`, cuDNN is forced onto
    deterministic kernels; runs then repeat exactly on the same hardware and library versions."""
    import torch

    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_logger(name: str = "roadlens") -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter("%(asctime)s | %(message)s", "%H:%M:%S"))
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
        logger.propagate = False
    return logger


In [ ]:
%%writefile /content/RoadLens/roadlens/data/mappings/btsd_to_gtsrb.yaml
# BTSD (BelgiumTSC, 62 classes) -> GTSRB (43 classes).
#
# Built by looking at every BTSD class (scripts/check_mapping.py --dataset btsd) and keeping only
# classes whose pictogram is the same sign as a GTSRB class. Deliberately NOT mapped:
#   32  all speed limits pooled in one class (50, 70, ...): no single GTSRB class
#   35  blue arrow, mixed directions (keep right / keep left / turn) in one class
#   1   single speed hump            (GTSRB 22 is the double-bump "uneven road" = BTSD 0)
#   6   double curve, first right    (GTSRB 21 is first left = BTSD 5)
#   9   cattle                       (GTSRB 31 is a deer)
#   14, 15  road narrows both sides / left   (GTSRB 24 is narrows on the right = BTSD 16)
#   18  crossroads with priority from the right (X); GTSRB 11 is the arrow pictogram = BTSD 17
# and everything with no GTSRB counterpart (parking, no stopping, cycle paths, limits by weight/size, ...).
dataset: btsd
verified: true   # every pair checked visually against GTSRB examples
map:
  0: 22    # uneven road                      -> Bumpy road
  2: 23    # slippery road                    -> Slippery road
  3: 19    # bend to the left                 -> Dangerous curve left
  4: 20    # bend to the right                -> Dangerous curve right
  5: 21    # double bend, first left          -> Double curve
  7: 28    # children                         -> Children crossing
  8: 29    # cyclists                         -> Bicycles crossing
  10: 25   # road works                       -> Road work
  11: 26   # traffic signals                  -> Traffic signals
  13: 18   # other danger (!)                 -> General caution
  16: 24   # road narrows on the right        -> Road narrows on the right
  17: 11   # priority at next intersection    -> Right-of-way at next intersection
  19: 13   # give way                         -> Yield
  21: 14   # stop                             -> Stop
  22: 17   # no entry                         -> No entry
  25: 16   # no lorries                       -> No trucks
  28: 15   # closed to all vehicles           -> No vehicles
  31: 9    # no overtaking                    -> No passing
  34: 35   # ahead only                       -> Ahead only
  36: 36   # ahead or right                   -> Go straight or right
  37: 40   # roundabout                       -> Roundabout mandatory
  61: 12   # priority road                    -> Priority road


In [ ]:
%%writefile /content/RoadLens/roadlens/data/mappings/ctsd_to_gtsrb.yaml
# CTSD (Chinese traffic signs, TSRD release, 58 classes) -> GTSRB (43 classes).
#
# !! DRAFT, NOT VERIFIED !!  The dataset could not be downloaded when this was written, so the
# source ids below come from the published TSRD class list, not from looking at the images.
# Before using it:  python scripts/check_mapping.py --dataset ctsd
# compare every row with its GTSRB example, fix/remove wrong rows, then set `verified: true`.
# The loader refuses to use this file until then.
#
# Only signs that share the GTSRB design are candidates (red-ring prohibitions, blue mandatory arrows).
# Chinese warning signs are yellow with a black border and are intentionally left unmapped.
# Stop / give way exist in TSRD but their ids are not known with confidence: add them after inspection.
dataset: ctsd
verified: false
map:
  2: 1     # speed limit 30
  4: 2     # speed limit 50
  5: 3     # speed limit 60
  6: 4     # speed limit 70
  7: 5     # speed limit 80
  14: 9    # no overtaking
  20: 36   # go straight or right
  21: 35   # go straight
  22: 34   # turn left
  24: 33   # turn right
  25: 39   # keep left
  26: 38   # keep right
  27: 40   # roundabout
  55: 17   # no entry


### `configs/`

In [ ]:
%%writefile /content/RoadLens/configs/base.yaml
# Training defaults. Every file in configs/runs/ inherits from this (directly or through another run).
_base_: data.yaml

runs_root: ./runs            # checkpoints, logs, cached evaluations ($ROADLENS_RUNS overrides)
results_root: ./results      # tables and figures

model:
  name: resnet18             # smallcnn | fusion | any timm model name
  pretrained: true           # ImageNet weights from timm
  input_size: 128            # the 64 px image is upsampled to this size on the GPU before the backbone
  drop_path: 0.0

train:
  epochs: 15
  batch_size: 128
  lr: 5.0e-4
  weight_decay: 0.05
  warmup_epochs: 1
  label_smoothing: 0.1
  grad_clip: 1.0
  patience: 4                # early stopping: epochs without a better validation-C accuracy
  amp: true                  # mixed precision (GPU only)
  eval_batch_size: 512

augment:
  # Baseline augmentation: mild colour jitter only. No horizontal flip (turn-left <-> turn-right).
  color_jitter: {brightness: 0.2, contrast: 0.2, saturation: 0.2, hue: 0.02}
  mode: none                 # none | mix (train on one AugMix view) | jsd (clean + 2 AugMix views, JSD loss)
  ops: [augmix, prime]       # op names or sets (augmix, prime, geometric); see roadlens/augment/ops.py
  exclude_ops: []
  width: 3                   # chains per mixed image
  depth: [1, 3]              # ops per chain, sampled uniformly
  alpha: 1.0                 # Dirichlet / Beta concentration of the mixing weights
  severity: 3                # op magnitude, 0-10 (AugMix scale)
  jsd_weight: 12.0


In [ ]:
%%writefile /content/RoadLens/configs/data.yaml
# Data + corruption settings shared by every experiment (other configs set `_base_: data.yaml`).

# Where raw/processed/corrupted data lives. Relative paths are resolved against the repo root.
# The ROADLENS_DATA environment variable overrides this (use it on Colab: /content/roadlens_data).
data_root: ./datasets

data:
  image_size: 64          # working resolution; images are resized FIRST, then corrupted
  crop_roi: false         # false = full annotated image (GTSRB/BTSD ship with ~10% context border)
  val_fraction: 0.1       # held out from train, split by physical sign track (no frame leakage)
  split_seed: 0           # seed of the train/val split
  num_classes: 43
  # Debug subsets (0 = everything). test_subset = N evaluates the first N test images and expects
  # corrupted sets generated with the same N (`make_gtsrb_c.py` reads it too).
  train_subset: 0
  val_subset: 0
  test_subset: 0

corruptions:
  seed: 0                 # seed of GTSRB-C / validation-C generation
  severities: [1, 2, 3, 4, 5]
  # Test corruptions. NEVER used for training, augmentation design or model selection.
  test:
    noise:   [gaussian_noise, impulse_noise, shot_noise]
    blur:    [motion_blur, zoom_blur, defocus_blur]
    weather: [snow, fog, frost, brightness]
    digital: [contrast, pixelate, elastic_transform, jpeg_compression]
  # Held-out corruptions used only for validation-C (early stopping / model selection).
  val: [speckle_noise, gaussian_blur, spatter, saturate]
  val_severities: [1, 2, 3, 4, 5]


In [ ]:
%%writefile /content/RoadLens/configs/experiments/common.yaml
# Shared by all experiment configs.
_base_: ../base.yaml
seeds: [0, 1, 2]
normalizer: smallcnn_base      # mCE and relative mCE are normalised by this run's errors (mean over seeds)
labels:
  smallcnn_base: SmallCNN
  resnet18_base: ResNet-18
  deit_s_base: DeiT-S
  resnet18_mix: ResNet-18 + mixing
  resnet18_mix_jsd: ResNet-18 + mixing + JSD
  resnet18_full: ResNet-18 (proposed training)
  deit_s_full: DeiT-S (proposed training)
  ensemble_avg: Average ensemble
  ensemble_conf: Confidence-weighted ensemble
  fusion_full: Gated fusion (proposed)
  resnet18_jsd_prime: PRIME ops only
  resnet18_jsd_nogeom: AugMix + PRIME, no geometric ops
  resnet18_in: ResNet-18 + instance input norm
  deit_s_in: DeiT-S + instance input norm
  ensemble_avg_in: Average ensemble (instance norm)
  ensemble_views: Average over view experts
  ensemble_views_in: Average over view experts (instance norm)
  fusion_v2: Stacked view-expert gate (v2)


In [ ]:
%%writefile /content/RoadLens/configs/experiments/exp1_clean.yaml
# Experiment 1: clean train / test accuracy and macro-F1.
_base_: common.yaml
runs: [smallcnn_base, resnet18_base, deit_s_base, resnet18_full, deit_s_full, ensemble_conf, fusion_full]


In [ ]:
%%writefile /content/RoadLens/configs/experiments/exp2_corruption.yaml
# Experiment 2: accuracy and macro-F1 on every corruption x severity of GTSRB-C.
_base_: common.yaml
runs: [smallcnn_base, resnet18_base, deit_s_base, resnet18_full, deit_s_full, ensemble_avg, ensemble_conf, fusion_full]


In [ ]:
%%writefile /content/RoadLens/configs/experiments/exp3_ablation.yaml
# Experiment 3: cumulative ablation. Each step adds one component to the previous one.
_base_: common.yaml
steps:
  - {label: "ResNet-18 baseline", run: resnet18_base}
  - {label: "+ AugMix mixing", run: resnet18_mix}
  - {label: "+ JSD consistency", run: resnet18_mix_jsd}
  - {label: "+ Fourier / PRIME ops", run: resnet18_full}
  - {label: "+ gated fusion with DeiT-S", run: fusion_full}
  - {label: "+ TENT", run: fusion_full, tta: {method: tent, batch_size: 64, lr: 1.0e-3}}


In [ ]:
%%writefile /content/RoadLens/configs/experiments/exp4_leave_category_out.yaml
# Experiment 4: choose the augmentation op set using only some corruption categories, test on the others.
_base_: common.yaml
baseline: resnet18_base
candidates: [resnet18_mix_jsd, resnet18_jsd_prime, resnet18_full, resnet18_jsd_nogeom]
directions:
  - {tune: [noise, blur], test: [weather, digital]}
  - {tune: [weather, digital], test: [noise, blur]}


In [ ]:
%%writefile /content/RoadLens/configs/experiments/exp5_cross_dataset.yaml
# Experiment 5: train on GTSRB, test on signs from other countries (shared classes), clean and corrupted.
_base_: common.yaml
datasets: [btsd, ctsd]         # a dataset that is missing, or whose class map is unverified, is skipped
runs: [smallcnn_base, resnet18_base, deit_s_base, resnet18_full, deit_s_full, ensemble_conf, fusion_full]


In [ ]:
%%writefile /content/RoadLens/configs/experiments/exp6_tent_batchsize.yaml
# Experiment 6: test-time adaptation vs. test batch size.
_base_: common.yaml
runs: [smallcnn_base, resnet18_base, resnet18_full, fusion_full]   # models with BatchNorm layers
methods: [none, bn, tent]      # source model / test-time BN statistics / TENT
batch_sizes: [1, 16, 64, 200]
severities: [5]                # GTSRB-C severities in the stream (each corruption is its own stream)
n_per_set: 1000                # images per corruption stream (fixed random subset; 0 = all 12,630)
n_per_set_bs1: 250             # shorter stream for batch size 1 (one optimisation step per image is slow,
                               # and adaptation with single-image statistics fails long before 250 steps)
lr: 1.0e-3                     # TENT: Adam, one step per batch


In [ ]:
%%writefile /content/RoadLens/configs/experiments/exp7_improved.yaml
# Experiment 7: the v2 approach, one mechanism per row.
#
# Every step is evaluated on GTSRB-C and, when prepared, on the cross-dataset sets below.
# "Val-C (sel.)" is the model-selection metric: accuracy on the select half of the validation split
# under the four validation corruptions, none of which is a test corruption. Decide between rows
# with that column, not with the test columns.
_base_: common.yaml
steps:
  - {label: "ResNet-18 (proposed training)", run: resnet18_full}
  - {label: "ResNet-18 + instance input norm", run: resnet18_in}
  - {label: "DeiT-S (proposed training)", run: deit_s_full}
  - {label: "DeiT-S + instance input norm", run: deit_s_in}
  - {label: "v1 gated fusion", run: fusion_full}
  - {label: "v1 average ensemble", run: ensemble_avg}
  - {label: "v1 average + view experts", run: ensemble_views}
  - {label: "Average + instance input norm", run: ensemble_avg_in}
  - {label: "Average + instance norm + view experts", run: ensemble_views_in}
  - {label: "v2: + stacked degradation-aware gate", run: fusion_v2}
  - {label: "v2 + temperature scaling", run: fusion_v2, calibrate: true}
datasets: [btsd]               # cross-dataset columns; a dataset that is not prepared is skipped

# Second table: test-time BatchNorm at small batch sizes, with and without a source-statistics prior.
adaptation_runs: [resnet18_base, resnet18_full, resnet18_in, fusion_v2]
batch_sizes: [1, 16, 64]
prior: 16                      # pseudo-samples of source statistics (Schneider et al. 2020)
severities: [5]
n_per_set: 1000                # as in experiment 6, so that shared evaluations are reused
n_per_set_bs1: 250


In [ ]:
%%writefile /content/RoadLens/configs/experiments/exp8_v3.yaml
# Experiment 8: the v3 approach (one mechanism per row), other architectures, self-supervised
# pre-training, and distillation into a single fast network.
#
# Read it like experiment 7: decide between rows with "Val-C (sel.)" (validation corruptions, no test
# image), not with the test columns. The v3 mechanisms were designed after looking at the pilot's
# per-corruption results (fog, noise), so GTSRB-C is not a blind test of them.
_base_: common.yaml
labels:
  resnet18_ln: ResNet-18 + local-global norm
  resnet18_ln_apr: ResNet-18 + local-global norm + APR
  resnet18_v3: ResNet-18 v3 (+ EMA, GPU speed-ups)
  resnet18_v3_64: ResNet-18 v3, native 64 px
  resnet18_v3_ssl: ResNet-18 v3, SimCLR pre-trained
  convnext_atto_v3: ConvNeXt-Atto v3
  efficientnet_b0_v3: EfficientNet-B0 v3
  mobilenetv3_v3: MobileNetV3-L v3
  deit_s_v3: DeiT-S v3
  ensemble_v3: v3 view-expert ensemble
  student_v3: v3 student (ResNet-18, distilled)
  student_mbv3: v3 student (MobileNetV3, distilled)
steps:
  - {label: "ResNet-18 + instance norm (v2 branch)", run: resnet18_in}
  - {label: "+ local-global contrast norm", run: resnet18_ln}
  - {label: "+ amplitude-phase recombination", run: resnet18_ln_apr}
  - {label: "+ EMA weights = ResNet-18 v3", run: resnet18_v3}
  - {label: "ResNet-18 v3, native 64 px (fast)", run: resnet18_v3_64}
  - {label: "ResNet-18 v3, SimCLR pre-trained", run: resnet18_v3_ssl}
  - {label: "ConvNeXt-Atto v3", run: convnext_atto_v3}
  - {label: "EfficientNet-B0 v3", run: efficientnet_b0_v3}
  - {label: "MobileNetV3-L v3", run: mobilenetv3_v3}
  - {label: "DeiT-S v3", run: deit_s_v3}
  - {label: "v2 teacher: view experts (instance norm)", run: ensemble_views_in}
  - {label: "v3 view-expert ensemble", run: ensemble_v3}
  - {label: "v3 student: ResNet-18 distilled", run: student_v3}
  - {label: "v3 student: MobileNetV3 distilled", run: student_mbv3}
  - {label: "v3 student (ResNet-18) + temperature", run: student_v3, calibrate: true}
datasets: [btsd]
speed:
  batch_size: 256              # throughput batch
  n_images: 2048
  repeats: 3


In [ ]:
%%writefile /content/RoadLens/configs/experiments/exp9_classical_unsupervised.yaml
# Experiment 9: other learning algorithms on the same test sets.
#   A. classical supervised learners on hand-crafted HOG features, and classifiers on the
#      self-supervised (label-free) SimCLR representation;
#   B. an unsupervised analysis: k-means clusters of each network's features, and how often a
#      corrupted image stays in the cluster of its clean original (no labels involved).
_base_: common.yaml
classical:
  hog: {win: 64, block: 16, stride: 8, cell: 8, bins: 9}     # 1,764 features per image
  svm_c: 0.05                 # LinearSVC regularisation (standardised HOG)
  rf_trees: 200
  knn_k: 5
  pca_dim: 64
  logreg_c: 1.0
ssl_run: resnet18_simclr
knn_bank: 0                   # 0 = the whole training set as the kNN memory bank
cluster:
  runs: [resnet18_base, resnet18_full, resnet18_in, resnet18_v3, resnet18_v3_ssl, resnet18_simclr, deit_s_full,
         convnext_atto_v3, student_v3]
  k: 43
  n_init: 3
  severities: [3, 5]


In [ ]:
%%writefile /content/RoadLens/configs/experiments/figures.yaml
# Figures. All of them are written as vector PDF to results/figures/.
_base_: common.yaml
baseline: resnet18_base
proposed: fusion_full
curve_runs: [smallcnn_base, resnet18_base, deit_s_base, resnet18_full, deit_s_full, fusion_full]
reliability_runs: [smallcnn_base, resnet18_base, deit_s_base, fusion_full]
fourier:
  runs: [resnet18_base, deit_s_base, fusion_full]
  seed: 0                      # model seed
  n_images: 500                # test images per Fourier basis
  eps: 4.0                     # L2 norm of the perturbation per channel (pixel range [0, 1])
  stride: 1                    # evaluate every `stride`-th frequency in each direction
gradcam:
  runs: [resnet18_base, fusion_full]
  seed: 0
  n_samples: 6
  corruptions: [gaussian_noise, fog, motion_blur]
  severity: 3


In [ ]:
%%writefile /content/RoadLens/configs/profiles/pilot.yaml
# --profile pilot: real but rough results in roughly an hour on one GPU.
# Full training set, ONE seed, schedules shortened to 40%, the first 2,000 test images.
# Use it to preview trends before spending GPU time on the full three-seed experiments.
# Outputs go to runs_pilot/ and results_pilot/.
# Needs: python scripts/make_gtsrb_c.py --profile pilot   (and --dataset btsd --profile pilot)
data:
  test_subset: 2000
train:
  epoch_scale: 0.4           # epochs = max(2, round(0.4 * epochs of the run))
seeds: [0]
n_per_set: 500               # experiment 6
n_per_set_bs1: 100
fourier: {n_images: 200, stride: 2}


In [ ]:
%%writefile /content/RoadLens/configs/profiles/pilot_4gb.yaml
# --profile pilot_4gb: the pilot profile for a GPU with about 4 GB of memory (e.g. a laptop RTX 3050).
# Same as pilot.yaml, but with batch size 64 so that three JSD views of DeiT-S fit in memory.
# Outputs go to runs_pilot_4gb/ and results_pilot_4gb/; it shares the 2,000-image GTSRB-C subset with pilot.
data:
  test_subset: 2000
train:
  epoch_scale: 0.4
  batch_size: 64
seeds: [0]
n_per_set: 500
n_per_set_bs1: 100
fourier: {n_images: 200, stride: 2}


In [ ]:
%%writefile /content/RoadLens/configs/profiles/smoke.yaml
# --profile smoke: a tiny end-to-end run of every script (minutes). Numbers are meaningless.
# Outputs go to runs_smoke/ and results_smoke/.
# Needs: python scripts/make_gtsrb_c.py --profile smoke   (and --dataset btsd --profile smoke)
data:
  train_subset: 3000
  val_subset: 400
  test_subset: 200
train:
  epochs: 2
  batch_size: 64             # small enough for a 4 GB GPU
seeds: [0, 1]
n_per_set: 64                # experiment 6
n_per_set_bs1: 16
fourier: {n_images: 32, stride: 8}
gradcam: {n_samples: 3}


In [ ]:
%%writefile /content/RoadLens/configs/runs/convnext_atto_v3.yaml
# Architecture study: ConvNeXt-Atto (3.4 M parameters with the 43-class head; LayerNorm, 7x7 depthwise kernels, GELU), v3 recipe.
_base_: resnet18_v3.yaml
model: {name: convnext_atto, drop_path: 0.05}


In [ ]:
%%writefile /content/RoadLens/configs/runs/deit_s_base.yaml
# DeiT-S (ImageNet-pretrained), fine-tuned with standard training.
# For Swin-T use: model: {name: swin_tiny_patch4_window7_224, input_size: 224}
_base_: ../base.yaml
model: {name: deit_small_patch16_224, drop_path: 0.1}
train: {lr: 1.0e-4, warmup_epochs: 2}


In [ ]:
%%writefile /content/RoadLens/configs/runs/deit_s_full.yaml
# DeiT-S with the proposed training method. Transformer branch of the fusion.
_base_: deit_s_base.yaml
augment: {mode: jsd, ops: [augmix, prime]}


In [ ]:
%%writefile /content/RoadLens/configs/runs/deit_s_in.yaml
# v2 transformer branch: the proposed training, with per-image standardised input.
_base_: deit_s_full.yaml
model: {input_norm: instance}


In [ ]:
%%writefile /content/RoadLens/configs/runs/deit_s_v3.yaml
# v3 transformer branch: DeiT-S with the local-global stem, + APR, + EMA, + speed-ups.
_base_: deit_s_full.yaml
model: {input_norm: local}
augment: {ops: [augmix, prime, apr]}
train: {ema: 0.998, cudnn_benchmark: true, fused_optimizer: true}


In [ ]:
%%writefile /content/RoadLens/configs/runs/efficientnet_b0_v3.yaml
# Architecture study: EfficientNet-B0 (4.1 M parameters; MBConv + squeeze-excitation), v3 recipe.
_base_: resnet18_v3.yaml
model: {name: efficientnet_b0}
train: {lr: 1.0e-3}


In [ ]:
%%writefile /content/RoadLens/configs/runs/ensemble_avg.yaml
# Reference: plain average of the two branch distributions (nothing to train).
_base_: ../base.yaml
model:
  name: fusion
  branches: [resnet18_full, deit_s_full]
  fusion: {mode: average}
train: {skip: true}


In [ ]:
%%writefile /content/RoadLens/configs/runs/ensemble_avg_in.yaml
# Control for v2: plain average of the two instance-normalised branches (nothing to train).
_base_: ../base.yaml
model:
  name: fusion
  branches: [resnet18_in, deit_s_in]
  fusion: {mode: average}
train: {skip: true}


In [ ]:
%%writefile /content/RoadLens/configs/runs/ensemble_conf.yaml
# Fallback comparison: the same two branches combined with confidence weights (nothing to train).
_base_: ../base.yaml
model:
  name: fusion
  branches: [resnet18_full, deit_s_full]
  fusion: {mode: confidence}
train: {skip: true}


In [ ]:
%%writefile /content/RoadLens/configs/runs/ensemble_v3.yaml
# v3 view-expert ensemble: the two v3 branches on three views (raw, 3x3 median, Gaussian blur), averaged.
_base_: ../base.yaml
model:
  name: fusion
  branches: [resnet18_v3, deit_s_v3]
  fusion: {mode: average, views: [raw, median3, gauss]}
train: {skip: true}


In [ ]:
%%writefile /content/RoadLens/configs/runs/ensemble_views.yaml
# View experts on the v1 branches, no retraining: each branch also sees a median-filtered and a
# Gaussian-blurred copy of the image, and the six predictions are averaged (nothing to train).
_base_: ../base.yaml
model:
  name: fusion
  branches: [resnet18_full, deit_s_full]
  fusion: {mode: average, views: [raw, median3, gauss]}
train: {skip: true}


In [ ]:
%%writefile /content/RoadLens/configs/runs/ensemble_views_in.yaml
# Control for the v2 gate: plain average over the six view experts of the instance-normalised
# branches (2 branches x 3 views; nothing to train). The v2 gate starts as exactly this model.
_base_: ../base.yaml
model:
  name: fusion
  branches: [resnet18_in, deit_s_in]
  fusion: {mode: average, views: [raw, median3, gauss]}
train: {skip: true}


In [ ]:
%%writefile /content/RoadLens/configs/runs/fusion_full.yaml
# Proposed model: ResNet-18 + DeiT-S (both trained with the proposed method, same seed) fused by a
# learned gating network. The branches are frozen; only the gate is trained, on the same augmented views.
_base_: ../base.yaml
model:
  name: fusion
  branches: [resnet18_full, deit_s_full]
  fusion: {mode: gate, gate_inputs: [features, probs], hidden: 256, dropout: 0.1, freeze_branches: true}
train: {epochs: 5, lr: 1.0e-3, weight_decay: 0.01, warmup_epochs: 0, patience: 2}
augment: {mode: jsd, ops: [augmix, prime]}


In [ ]:
%%writefile /content/RoadLens/configs/runs/fusion_v2.yaml
# RoadLens v2: degradation-aware gate over six view experts, fitted by stacked generalisation.
#
# Experts: the two instance-normalised branches (frozen), each on three views of the image
# (raw, 3x3 median, Gaussian blur). The gate is a small MLP on the experts' confidence statistics
# and five statistics of the input image; it starts as the plain average of the experts.
#
# Unlike fusion_full, the gate is NOT trained on the branches' training images, which the branches
# fit almost perfectly and which therefore say nothing about when a branch fails. It is fitted on
# one half of the validation split (clean + validation-C, with strong AugMix/PRIME views on top)
# and selected on the other half; the two halves share no physical sign. The untrained gate, i.e.
# the plain average, is a candidate too (eval_at_init), so selection cannot make the model worse
# than averaging on the selection data. The loss is the plain negative log-likelihood of all views.
_base_: ../base.yaml
model:
  name: fusion
  branches: [resnet18_in, deit_s_in]
  fusion:
    mode: gate
    gate_inputs: [stats, image]
    hidden: 32
    dropout: 0.0
    freeze_branches: true
    views: [raw, median3, gauss]
train:
  fit_split: val
  epochs: 30
  lr: 2.0e-3
  weight_decay: 0.0
  warmup_epochs: 1
  patience: 8
  label_smoothing: 0.0
  ce_all_views: true
  eval_at_init: true
augment: {mode: jsd, ops: [augmix, prime], severity: 6, jsd_weight: 0.0}


In [ ]:
%%writefile /content/RoadLens/configs/runs/mobilenetv3_v3.yaml
# Architecture study: MobileNetV3-Large (4.3 M parameters, built for mobile / embedded inference), v3 recipe.
_base_: resnet18_v3.yaml
model: {name: mobilenetv3_large_100}
train: {lr: 1.0e-3}


In [ ]:
%%writefile /content/RoadLens/configs/runs/resnet18_base.yaml
# ResNet-18 (ImageNet-pretrained), fine-tuned with standard training.
_base_: ../base.yaml
model: {name: resnet18}


In [ ]:
%%writefile /content/RoadLens/configs/runs/resnet18_full.yaml
# Ablation step "+ Fourier ops": the proposed training method (AugMix ops + PRIME-style primitives, JSD).
# Also the CNN branch of the fusion.
_base_: resnet18_base.yaml
augment: {mode: jsd, ops: [augmix, prime]}


In [ ]:
%%writefile /content/RoadLens/configs/runs/resnet18_in.yaml
# v2 CNN branch: the proposed training, with every input image standardised by its own
# per-channel mean and std (model.input_norm: instance) instead of the dataset statistics.
_base_: resnet18_full.yaml
model: {input_norm: instance}


In [ ]:
%%writefile /content/RoadLens/configs/runs/resnet18_jsd_nogeom.yaml
# Leave-category-out candidate: full op set without anything that moves pixels
# (rotate, shear, translate, spatial_warp), i.e. without the ops related to elastic_transform.
_base_: resnet18_base.yaml
augment: {mode: jsd, ops: [augmix, prime], exclude_ops: [geometric]}


In [ ]:
%%writefile /content/RoadLens/configs/runs/resnet18_jsd_prime.yaml
# Leave-category-out candidate: PRIME-style primitives only.
_base_: resnet18_base.yaml
augment: {mode: jsd, ops: [prime]}


In [ ]:
%%writefile /content/RoadLens/configs/runs/resnet18_ln.yaml
# RoadLens v3, step 1: the v2 ResNet-18 recipe with the local-global contrast-normalisation stem
# (model.input_norm: local, see roadlens/models/base.py) instead of plain instance normalisation.
_base_: resnet18_full.yaml
model: {input_norm: local}


In [ ]:
%%writefile /content/RoadLens/configs/runs/resnet18_ln_apr.yaml
# RoadLens v3, step 2: + amplitude-phase recombination (op set 'apr', roadlens/augment/ops.py) in the
# AugMix chains, next to the AugMix and PRIME ops.
_base_: resnet18_ln.yaml
augment: {ops: [augmix, prime, apr]}


In [ ]:
%%writefile /content/RoadLens/configs/runs/resnet18_mix.yaml
# Ablation step "+ mixing": AugMix-style mixing with the AugMix ops, plain cross-entropy on the mixed view.
_base_: resnet18_base.yaml
augment: {mode: mix, ops: [augmix]}


In [ ]:
%%writefile /content/RoadLens/configs/runs/resnet18_mix_jsd.yaml
# Ablation step "+ JSD": clean + two mixed views with the Jensen-Shannon consistency loss (= AugMix).
_base_: resnet18_base.yaml
augment: {mode: jsd, ops: [augmix]}


In [ ]:
%%writefile /content/RoadLens/configs/runs/resnet18_simclr.yaml
# Self-supervised pre-training (no labels): SimCLR on the GTSRB training images, starting from the
# ImageNet weights, with the v3 stem and the v3 augmentation ops as the two views. See roadlens/ssl.py.
_base_: resnet18_base.yaml
model: {input_norm: local}
train: {objective: simclr, epochs: 15, batch_size: 256, lr: 5.0e-4, weight_decay: 0.05, warmup_epochs: 1,
        label_smoothing: 0.0, cudnn_benchmark: true, fused_optimizer: true}
augment: {mode: jsd, ops: [augmix, prime, apr], severity: 6}
ssl: {temperature: 0.2, proj_hidden: 512, proj_dim: 128, knn_bank: 10000}


In [ ]:
%%writefile /content/RoadLens/configs/runs/resnet18_v3.yaml
# RoadLens v3, step 3 = the v3 ResNet-18: + an exponential moving average of the weights (validated,
# selected and saved instead of the raw weights) and the GPU speed-ups (they change speed, not the maths).
_base_: resnet18_ln_apr.yaml
train: {ema: 0.998, cudnn_benchmark: true, fused_optimizer: true, channels_last: true}


In [ ]:
%%writefile /content/RoadLens/configs/runs/resnet18_v3_64.yaml
# v3 ResNet-18 at the native 64 px instead of upsampling to 128 px: 4x fewer pixels through the
# backbone. Measures the speed / accuracy trade-off.
_base_: resnet18_v3.yaml
model: {input_size: 64}


In [ ]:
%%writefile /content/RoadLens/configs/runs/resnet18_v3_ssl.yaml
# v3 ResNet-18 fine-tuned from the self-supervised SimCLR backbone instead of the ImageNet weights.
_base_: resnet18_v3.yaml
model: {init_from: resnet18_simclr}


In [ ]:
%%writefile /content/RoadLens/configs/runs/smallcnn_base.yaml
# SmallCNN from scratch, standard training. Baseline and mCE normaliser.
_base_: ../base.yaml
model: {name: smallcnn, pretrained: false, width: 32, dropout: 0.3}
train: {epochs: 30, lr: 2.0e-3, weight_decay: 0.01, patience: 6}


In [ ]:
%%writefile /content/RoadLens/configs/runs/student_mbv3.yaml
# v3 student, mobile size: MobileNetV3-Large distilled from the same six-expert teacher.
_base_: mobilenetv3_v3.yaml
train:
  distill: {teacher: ensemble_views_in, weight: 1.0, temperature: 2.0}


In [ ]:
%%writefile /content/RoadLens/configs/runs/student_v3.yaml
# v3 student: ONE ResNet-18 (v3 recipe) distilled from the six-expert v2 teacher (ensemble_views_in,
# the best model of the 15 Oct pilot by validation-C). The teacher sees exactly the same augmented views
# as the student. At test time the student needs 1 forward pass instead of the teacher's 6.
_base_: resnet18_v3.yaml
train:
  distill: {teacher: ensemble_views_in, weight: 1.0, temperature: 2.0}


### `scripts/`

In [ ]:
%%writefile /content/RoadLens/scripts/_bootstrap.py
"""Make `import roadlens` work when scripts are run as `python scripts/<name>.py`."""

import sys
import warnings
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# imagecorruptions imports the deprecated pkg_resources
warnings.filterwarnings("ignore", message=".*pkg_resources.*")


In [ ]:
%%writefile /content/RoadLens/scripts/benchmark.py
"""Measure training / inference speed of every run on this GPU and estimate experiment run times.

    python scripts/benchmark.py                    # all runs used by the experiments (about 3 minutes)
    python scripts/benchmark.py --batch-size 48    # on a GPU with less than ~6 GB memory

No dataset or checkpoint is needed: the models are built with random weights and fed random images
through the real training step (augmentation, mixed precision, loss, backward, optimizer). The
estimates assume every run trains for all of its epochs (early stopping makes real runs shorter)
and are written to results/benchmark.csv.
"""

import argparse
import time

import _bootstrap  # noqa: F401
import pandas as pd
import torch

from roadlens.augment import TrainAugment
from roadlens.engine import param_groups, training_loss
from roadlens.models import build_model
from roadlens.runs import load_run_config
from roadlens.utils import REPO_ROOT, apply_profile, get_device, load_config, results_root

N_TRAIN, N_TEST, N_SETS = 35219, 12630, 71          # GTSRB train split, test set, clean + 70 corrupted sets
EXPERIMENTS = ["exp1_clean", "exp2_corruption", "exp3_ablation", "exp4_leave_category_out",
               "exp5_cross_dataset", "exp6_tent_batchsize"]


def build(cfg):
    branches = [build(load_run_config(b)) for b in cfg["model"].get("branches", [])] or None
    return build_model(cfg["model"], cfg["data"]["num_classes"], branches, pretrained=False)


def sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize()


def measure(name, device, batch_size, steps, profile):
    cfg = load_run_config(name, profile)
    t, size = cfg["train"], cfg["data"]["image_size"]
    bs = batch_size or int(t["batch_size"])
    amp = bool(t.get("amp", True)) and device.type == "cuda"
    model = build(cfg).to(device)
    row = {"run": name, "epochs": int(t["epochs"]), "batch_size": bs, "train_img_s": float("nan"), "peak_mem_gb": float("nan")}
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    x8 = torch.randint(0, 256, (bs, 3, size, size), device=device, dtype=torch.uint8)
    y = torch.randint(0, cfg["data"]["num_classes"], (bs,), device=device)
    if not t.get("skip"):
        aug = TrainAugment(cfg["augment"])
        opt = torch.optim.AdamW(param_groups(model, float(t["weight_decay"])), lr=float(t["lr"]))
        scaler = torch.amp.GradScaler(device.type, enabled=amp)
        model.train()
        for i in range(steps + 5):
            if i == 5:                              # the first steps include cuDNN / allocator warm-up
                sync(device)
                t0 = time.perf_counter()
            loss = training_loss(model, aug(x8.float().div(255)), y, aug.mode == "jsd",
                                 float(t.get("label_smoothing", 0)), float(cfg["augment"].get("jsd_weight", 12)), amp)[0]
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        sync(device)
        row["train_img_s"] = steps * bs / (time.perf_counter() - t0)
        row["views"] = aug.n_views
    model.eval()
    eb = min(int(t.get("eval_batch_size", 512)), 4 * bs) if batch_size else int(t.get("eval_batch_size", 512))
    xe = torch.randint(0, 256, (eb, 3, size, size), device=device, dtype=torch.uint8)
    with torch.no_grad():
        for i in range(8):
            if i == 3:
                sync(device)
                t0 = time.perf_counter()
            with torch.autocast(device.type, enabled=amp):
                model(xe.float().div(255))
        sync(device)
    row["infer_img_s"] = 5 * eb / (time.perf_counter() - t0)
    if device.type == "cuda":
        row["peak_mem_gb"] = torch.cuda.max_memory_allocated(device) / 2 ** 30
    row["train_min"] = 0.0 if t.get("skip") else row["epochs"] * N_TRAIN / row["train_img_s"] / 60
    row["eval_gtsrb_c_min"] = N_SETS * N_TEST / row["infer_img_s"] / 60
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return row


def experiment_runs(profile):
    """Runs each experiment needs (branches of fusions and the mCE normaliser included)."""
    out = {}
    for name in EXPERIMENTS:
        cfg = apply_profile(load_config(REPO_ROOT / "configs" / "experiments" / f"{name}.yaml"), profile)
        runs = list(cfg.get("runs", [])) + [s["run"] for s in cfg.get("steps", [])] + list(cfg.get("candidates", []))
        runs += [cfg[k] for k in ("baseline", "normalizer") if k in cfg]
        todo, seen = list(dict.fromkeys(runs)), []
        while todo:
            r = todo.pop(0)
            if r not in seen:
                seen.append(r)
                todo += load_run_config(r)["model"].get("branches", [])
        out[name] = (seen, len(cfg["seeds"]))
    return out


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--runs", nargs="+", default=None, help="default: every run used by the experiments")
    ap.add_argument("--batch-size", type=int, default=0, help="override the training batch size (small GPUs)")
    ap.add_argument("--steps", type=int, default=20)
    ap.add_argument("--profile", default=None)
    ap.add_argument("--device", default=None)
    args = ap.parse_args()
    device = get_device(args.device)
    exps = experiment_runs(args.profile)
    runs = args.runs or list(dict.fromkeys(r for rs, _ in exps.values() for r in rs))
    print(f"device: {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu'}")
    rows = []
    for name in runs:
        try:
            rows.append(measure(name, device, args.batch_size, args.steps, args.profile))
        except RuntimeError as err:                 # most likely out of memory
            print(f"{name}: {str(err).splitlines()[0]}")
            if device.type == "cuda":
                torch.cuda.empty_cache()
    df = pd.DataFrame(rows).set_index("run")
    pd.set_option("display.width", 200)
    print(df.round(1).to_string())
    out = results_root(apply_profile(load_config(REPO_ROOT / "configs" / "base.yaml"), args.profile)) / "benchmark.csv"
    df.round(2).to_csv(out)

    if not args.runs:
        print("\nEstimated GPU time (all epochs, no early stopping; training shared between experiments is counted once):")
        trained, total = set(), 0.0
        for name, (needed, n_seeds) in exps.items():
            new = [r for r in needed if r not in trained and r in df.index]
            trained |= set(new)
            train = n_seeds * sum(df.train_min[r] for r in new)
            evals = n_seeds * sum(df.eval_gtsrb_c_min[r] for r in needed if r in df.index)
            total += train + evals
            print(f"  {name:28s} new training {train / 60:5.1f} h   evaluation {evals / 60:5.1f} h   ({n_seeds} seed(s))")
        print(f"  {'total':28s} {total / 60:5.1f} h   (+ GTSRB-C generation on the CPU, + test-time-adaptation streams)")
    print(f"\nwritten: {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/RoadLens/scripts/check_mapping.py
"""Contact sheet for building / auditing a cross-dataset class map.

    python scripts/check_mapping.py --dataset btsd

One row per source class: its mapped GTSRB class (one GTSRB example, blank if unmapped) followed by
random examples of the source class. Writes results/mappings/<dataset>_mapping.pdf (+ .png pages).
"""

import argparse

import _bootstrap  # noqa: F401
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from roadlens.data.classes import GTSRB_SHORT
from roadlens.data.external import load_external
from roadlens.data.gtsrb import load_gtsrb_cfg
from roadlens.utils import REPO_ROOT, data_root, load_config


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/data.yaml")
    ap.add_argument("--dataset", required=True, choices=["btsd", "ctsd"])
    ap.add_argument("--examples", type=int, default=8)
    ap.add_argument("--rows-per-page", type=int, default=16)
    args = ap.parse_args()
    cfg = load_config(args.config)
    root, d = data_root(cfg), cfg["data"]
    x, y, y_src = load_external(root, args.dataset, d["image_size"], d["crop_roi"],
                                mapped_only=False, allow_unverified=True)
    gx, gy = load_gtsrb_cfg(cfg, root, "test")
    rng = np.random.RandomState(0)
    out = REPO_ROOT / "results" / "mappings"
    out.mkdir(parents=True, exist_ok=True)

    classes = np.unique(y_src)
    k = args.examples
    pages = [classes[i:i + args.rows_per_page] for i in range(0, len(classes), args.rows_per_page)]
    from matplotlib.backends.backend_pdf import PdfPages
    with PdfPages(out / f"{args.dataset}_mapping.pdf") as pdf:
        for pi, page in enumerate(pages):
            fig, axes = plt.subplots(len(page), k + 1, figsize=((k + 1) * 0.95 + 1.6, len(page) * 0.95), squeeze=False)
            for r, c in enumerate(page):
                idx = np.flatnonzero(y_src == c)
                g = int(y[idx[0]])
                for ax in axes[r]:
                    ax.set_xticks([]); ax.set_yticks([])
                label = f"src {c} (n={len(idx)})\n-> " + (f"{g} {GTSRB_SHORT[g]}" if g >= 0 else "unmapped")
                axes[r, 0].set_ylabel(label, rotation=0, ha="right", va="center", fontsize=7)
                if g >= 0:
                    axes[r, 0].imshow(gx[rng.choice(np.flatnonzero(gy == g))])
                else:
                    axes[r, 0].axis("off")
                    axes[r, 0].text(-0.1, 0.5, label, ha="right", va="center", fontsize=7,
                                    transform=axes[r, 0].transAxes)
                for j, i in enumerate(rng.choice(idx, min(k, len(idx)), replace=False)):
                    axes[r, 1 + j].imshow(x[i])
                for j in range(min(k, len(idx)), k):
                    axes[r, 1 + j].axis("off")
            axes[0, 0].set_title("GTSRB", fontsize=7)
            fig.tight_layout(h_pad=0.1, w_pad=0.1)
            pdf.savefig(fig)
            fig.savefig(out / f"{args.dataset}_mapping_p{pi}.png", dpi=100)
            plt.close(fig)
    print(f"wrote {out / (args.dataset + '_mapping.pdf')}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/RoadLens/scripts/exp1_clean.py
"""Experiment 1 - clean performance: train / test accuracy, macro-F1 and ECE of every model.

    python scripts/exp1_clean.py

Trains whatever is missing (resumable), then writes to results/exp1_clean/:
    per_seed.csv   one row per model and seed
    clean.csv/.txt/.tex   mean +- std over seeds (booktabs table)
"""

import _bootstrap  # noqa: F401
import pandas as pd

from roadlens.eval.metrics import mean_std
from roadlens.eval.tables import over_seeds, write_table
from roadlens.experiment import Experiment, make_parser


def main():
    exp = Experiment(make_parser(__doc__, "configs/experiments/exp1_clean.yaml").parse_args(), "exp1_clean")
    runs = exp.cfg["runs"]
    info = exp.train(runs).set_index(["run", "seed"])
    long, _ = exp.evaluate(runs, "gtsrb_clean")

    rows = []
    for (run, seed), g in long.groupby(["run", "seed"], sort=False):
        s = g.set_index("split")
        rows.append({"run": run, "model": exp.label(run), "seed": seed,
                     "params_m": info.loc[(run, seed), "n_params"] / 1e6,
                     "train_acc": 100 * s.acc["train"], "val_acc": 100 * s.acc["val"],
                     "test_acc": 100 * s.acc["test"], "test_f1": 100 * s.macro_f1["test"],
                     "test_ece": 100 * s.ece["test"],
                     "train_min": info.loc[(run, seed), "train_time_s"] / 60})
    per_seed = pd.DataFrame(rows)
    per_seed.to_csv(exp.out / "per_seed.csv", index=False, float_format="%.4f")
    summary = mean_std(per_seed, ["run", "model"])
    table = write_table(exp.out, "clean", summary, {
        "params_m": ("Params (M)", 1, None),
        "train_acc": ("Train acc.", 2, None),
        "test_acc": ("Test acc.", 2, "max"),
        "test_f1": ("Test macro-F1", 2, "max"),
        "test_ece": ("Test ECE", 2, "min"),
    }, caption=f"Clean GTSRB performance (\\%, {over_seeds(len(exp.seeds))}).", label="tab:clean")
    print(table.to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/RoadLens/scripts/exp2_corruption.py
"""Experiment 2 - full corruption evaluation on GTSRB-C (14 corruptions x 5 severities).

    python scripts/exp2_corruption.py

Writes to results/exp2_corruption/:
    per_seed.csv            accuracy, macro-F1, ECE, NLL for every model / seed / corruption / severity
    per_severity.csv        the same, mean +- std over seeds
    summary_per_seed.csv    clean acc, mean corruption acc, mCE, relative mCE, macro-F1, ECE per model and seed
    main.csv/.txt/.tex      main results table (mean +- std over seeds)
    per_corruption.csv/.txt/.tex   accuracy per corruption (mean over severities and seeds)
    per_category.csv        accuracy per corruption category
"""

import _bootstrap  # noqa: F401

from roadlens.eval.metrics import mean_std
from roadlens.eval.tables import over_seeds, tex_escape, write_latex, write_table
from roadlens.experiment import Experiment, make_parser
from roadlens.utils import corruption_categories, corruption_names

MAIN_COLUMNS = {
    "clean_acc": ("Clean acc.", 2, "max"),
    "corr_acc": ("Corr. acc.", 2, "max"),
    "mce": ("mCE", 1, "min"),
    "rel_mce": ("Rel. mCE", 1, "min"),
    "clean_f1": ("Clean F1", 2, "max"),
    "corr_f1": ("Corr. F1", 2, "max"),
    "clean_ece": ("Clean ECE", 2, "min"),
    "corr_ece": ("Corr. ECE", 2, "min"),
}
SHORT = {"gaussian_noise": "Gauss.", "impulse_noise": "Impulse", "shot_noise": "Shot", "motion_blur": "Motion",
         "zoom_blur": "Zoom", "defocus_blur": "Defocus", "snow": "Snow", "fog": "Fog", "frost": "Frost",
         "brightness": "Bright.", "contrast": "Contr.", "pixelate": "Pixel", "elastic_transform": "Elastic",
         "jpeg_compression": "JPEG"}


def main():
    exp = Experiment(make_parser(__doc__, "configs/experiments/exp2_corruption.yaml").parse_args(), "exp2_corruption")
    runs = exp.cfg["runs"]
    long, _ = exp.evaluate(runs, "gtsrb_c")
    long["model"] = long.run.map(exp.label)
    cols = ["run", "model", "seed", "corruption", "severity", "n", "acc", "macro_f1", "ece", "nll"]
    long[cols].to_csv(exp.out / "per_seed.csv", index=False, float_format="%.5f")
    mean_std(long, ["run", "model", "corruption", "severity"], ["acc", "macro_f1", "ece"]).to_csv(
        exp.out / "per_severity.csv", index=False, float_format="%.5f")

    per_seed, summary = exp.summarize(long, "gtsrb_c")
    per_seed.insert(1, "model", per_seed.run.map(exp.label))
    per_seed.to_csv(exp.out / "summary_per_seed.csv", index=False, float_format="%.4f")
    summary.insert(1, "model", summary.run.map(exp.label))
    n = len(exp.seeds)
    table = write_table(exp.out, "main", summary, MAIN_COLUMNS,
                        caption=f"GTSRB and GTSRB-C results (\\%, {over_seeds(n)}). Corr.: mean over 14 "
                                f"corruptions $\\times$ 5 severities. mCE and relative mCE are normalised by "
                                f"{tex_escape(exp.label(exp.cfg['normalizer']))} (= 100).",
                        label="tab:main")
    print(table.to_string(index=False))

    # accuracy per corruption: mean over severities, then over seeds
    names = corruption_names(exp.cfg)
    corr = long[long.corruption != "clean"]
    per_corr = (100 * corr.groupby(["run", "seed", "corruption"], sort=False).acc.mean()).reset_index()
    wide = per_corr.groupby(["run", "corruption"], sort=False).acc.mean().unstack("corruption").reindex(runs)[names]
    wide.insert(0, "model", [exp.label(r) for r in wide.index])
    wide.to_csv(exp.out / "per_corruption.csv", float_format="%.2f")
    (exp.out / "per_corruption.txt").write_text(wide.round(1).to_string(index=False) + "\n", encoding="utf-8")
    best = wide[names].round(1).max()
    rows = [[tex_escape(r.model)] + [(r"\textbf{%.1f}" if round(r[c], 1) == best[c] else "%.1f") % r[c] for c in names]
            for _, r in wide.iterrows()]
    write_latex(exp.out / "per_corruption.tex", ["Model"] + [SHORT.get(c, c) for c in names], rows,
                caption=f"Accuracy (\\%) per corruption on GTSRB-C, averaged over 5 severities and {n} seed(s).",
                label="tab:per-corruption", align="l" + "r" * len(names))

    cats = corruption_categories(exp.cfg)
    per_cat = per_corr.assign(category=per_corr.corruption.map(cats)).groupby(["run", "seed", "category"], sort=False).acc.mean()
    cat_summary = mean_std(per_cat.reset_index(), ["run", "category"], ["acc"])
    cat_summary.insert(1, "model", cat_summary.run.map(exp.label))
    cat_summary.to_csv(exp.out / "per_category.csv", index=False, float_format="%.3f")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/RoadLens/scripts/exp3_ablation.py
"""Experiment 3 - cumulative ablation on GTSRB-C:
baseline -> + mixing -> + JSD -> + Fourier/PRIME ops -> + fusion -> + TENT.

    python scripts/exp3_ablation.py

Writes to results/exp3_ablation/:
    per_seed.csv              summary metrics per step and seed
    ablation.csv/.txt/.tex    mean +- std over seeds (booktabs table)
    per_severity.csv          accuracy per step / corruption / severity (mean +- std over seeds)
"""

import _bootstrap  # noqa: F401
import pandas as pd

from roadlens.eval.metrics import mean_std
from roadlens.eval.tables import over_seeds, tex_escape, write_table
from roadlens.experiment import Experiment, make_parser

COLUMNS = {
    "clean_acc": ("Clean acc.", 2, "max"),
    "corr_acc": ("Corr. acc.", 2, "max"),
    "mce": ("mCE", 1, "min"),
    "rel_mce": ("Rel. mCE", 1, "min"),
    "corr_f1": ("Corr. F1", 2, "max"),
    "corr_ece": ("Corr. ECE", 2, "min"),
}


def main():
    exp = Experiment(make_parser(__doc__, "configs/experiments/exp3_ablation.yaml").parse_args(), "exp3_ablation")
    frames = []
    for step in exp.cfg["steps"]:
        long, _ = exp.evaluate([step["run"]], "gtsrb_c", tta=step.get("tta"))
        frames.append(long.assign(step=step["label"]))
    long = pd.concat(frames, ignore_index=True)
    mean_std(long, ["step", "corruption", "severity"], ["acc", "macro_f1", "ece"]).to_csv(
        exp.out / "per_severity.csv", index=False, float_format="%.5f")

    per_seed, summary = exp.summarize(long, "gtsrb_c", key="step")
    per_seed.to_csv(exp.out / "per_seed.csv", index=False, float_format="%.4f")
    table = write_table(exp.out, "ablation", summary, COLUMNS, label_col="step", label_header="Configuration",
                        caption=f"Cumulative ablation on GTSRB-C (\\%, {over_seeds(len(exp.seeds))}). Each row adds one "
                                f"component to the row above. mCE is normalised by "
                                f"{tex_escape(exp.label(exp.cfg['normalizer']))}.",
                        label="tab:ablation")
    print(table.to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/RoadLens/scripts/exp4_leave_category_out.py
"""Experiment 4 - leave-category-out: does an augmentation choice tuned on some corruption
categories transfer to the others?

    python scripts/exp4_leave_category_out.py

For each direction (e.g. tune on noise+blur, test on weather+digital) the candidate op set with the
best mean accuracy on the tuning categories is selected, and its accuracy on the held-out categories
is compared with the baseline and with the candidate that would have been best on the held-out
categories themselves (the oracle). Writes to results/exp4_leave_category_out/:
    candidates.csv/.txt/.tex   accuracy of every candidate per corruption category
    selection.csv/.txt/.tex    selected vs. oracle vs. baseline on the held-out categories
"""

import _bootstrap  # noqa: F401
import pandas as pd

from roadlens.eval.metrics import mean_std
from roadlens.eval.tables import over_seeds, pm, tex_escape, write_latex, write_table
from roadlens.experiment import Experiment, make_parser
from roadlens.utils import corruption_categories


def main():
    exp = Experiment(make_parser(__doc__, "configs/experiments/exp4_leave_category_out.yaml").parse_args(),
                     "exp4_leave_category_out")
    cfg = exp.cfg
    baseline, candidates = cfg["baseline"], list(cfg["candidates"])
    runs = [baseline] + candidates
    long, _ = exp.evaluate(runs, "gtsrb_c")
    cats = corruption_categories(cfg)
    categories = list(cfg["corruptions"]["test"])
    corr = long[long.corruption != "clean"].assign(category=lambda d: d.corruption.map(cats))

    def group_acc(names):   # accuracy (%) over all test sets of the given categories, per run and seed
        return 100 * corr[corr.category.isin(names)].groupby(["run", "seed"], sort=False).acc.mean()

    per_seed = pd.DataFrame({c: group_acc([c]) for c in categories})
    per_seed["all"] = group_acc(categories)
    per_seed["clean"] = 100 * long[long.corruption == "clean"].set_index(["run", "seed"]).acc
    for i, d in enumerate(cfg["directions"]):
        per_seed[f"tune{i}"], per_seed[f"test{i}"] = group_acc(d["tune"]), group_acc(d["test"])
    per_seed = per_seed.reset_index()
    per_seed.insert(1, "model", per_seed.run.map(exp.label))
    per_seed.to_csv(exp.out / "per_seed.csv", index=False, float_format="%.4f")
    summary = mean_std(per_seed, ["run", "model"])
    n = len(exp.seeds)
    table = write_table(exp.out, "candidates", summary,
                        {"clean": ("Clean", 2, "max"), **{c: (c.capitalize(), 2, "max") for c in categories},
                         "all": ("All 14", 2, "max")},
                        caption=f"Accuracy (\\%) per corruption category for each augmentation op set "
                                f"(ResNet-18, {over_seeds(n)}).", label="tab:lco-candidates")
    print(table.to_string(index=False))

    s = summary.set_index("run")
    rows, tex_rows = [], []
    for i, d in enumerate(cfg["directions"]):
        cand = s.loc[candidates]
        selected = cand[f"tune{i}_mean"].idxmax()
        oracle = cand[f"test{i}_mean"].idxmax()

        def cell(run, tex=False):
            return pm(s.loc[run, f"test{i}_mean"], s.loc[run, f"test{i}_std"], 2, tex)

        row = {"tuned_on": "+".join(d["tune"]), "tested_on": "+".join(d["test"]),
               "selected": exp.label(selected), "selected_test_acc": cell(selected),
               "oracle": exp.label(oracle), "oracle_test_acc": cell(oracle),
               "baseline_test_acc": cell(baseline),
               "gap_to_oracle": round(float(s.loc[oracle, f"test{i}_mean"] - s.loc[selected, f"test{i}_mean"]), 3)}
        rows.append(row)
        tex_rows.append([tex_escape(row["tuned_on"]), tex_escape(row["tested_on"]), tex_escape(row["selected"]),
                         cell(selected, True), tex_escape(row["oracle"]), cell(oracle, True), cell(baseline, True),
                         f"{row['gap_to_oracle']:.2f}"])
    sel = pd.DataFrame(rows)
    sel.to_csv(exp.out / "selection.csv", index=False)
    (exp.out / "selection.txt").write_text(sel.to_string(index=False) + "\n", encoding="utf-8")
    write_latex(exp.out / "selection.tex",
                ["Tuned on", "Tested on", "Selected op set", "Acc.", "Oracle op set", "Acc.", "Baseline acc.", "Gap"],
                tex_rows, align="lllclccc",
                caption=f"Leave-category-out selection of the augmentation op set (\\%, {over_seeds(n)}). "
                        f"The op set is chosen on the tuning categories only; accuracies are on the held-out "
                        f"categories. Gap = oracle minus selected.", label="tab:lco")
    print(sel.to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/RoadLens/scripts/exp5_cross_dataset.py
"""Experiment 5 - cross-dataset: models trained on GTSRB, tested on BTSD / CTSD (classes shared with
GTSRB only), clean and corrupted.

    python scripts/prepare_data.py --datasets btsd          # once
    python scripts/make_gtsrb_c.py --dataset btsd           # once (corrupted BTSD, 2.3 GB)
    python scripts/exp5_cross_dataset.py

The models keep all 43 outputs ("Acc."); "Acc. (shared)" restricts the arg-max to the classes that
exist in the target dataset. A dataset that is not prepared, or whose class map is not verified, is
skipped with a message. Writes to results/exp5_cross_dataset/:
    <dataset>_per_seed.csv, <dataset>_summary_per_seed.csv, <dataset>.csv/.txt/.tex
"""

import _bootstrap  # noqa: F401

from roadlens.data.external import load_mapping
from roadlens.data.loading import external_unavailable
from roadlens.eval.metrics import mean_std
from roadlens.eval.tables import over_seeds, tex_escape, write_table
from roadlens.experiment import Experiment, make_parser
from roadlens.utils import get_logger

log = get_logger()
COLUMNS = {
    "clean_acc": ("Clean acc.", 2, "max"),
    "clean_acc_shared": ("Clean acc. (shared)", 2, "max"),
    "clean_f1": ("Clean F1", 2, "max"),
    "corr_acc": ("Corr. acc.", 2, "max"),
    "corr_acc_shared": ("Corr. acc. (shared)", 2, "max"),
    "mce": ("mCE", 1, "min"),
    "corr_ece": ("Corr. ECE", 2, "min"),
}


def main():
    exp = Experiment(make_parser(__doc__, "configs/experiments/exp5_cross_dataset.yaml").parse_args(),
                     "exp5_cross_dataset")
    runs, done = exp.cfg["runs"], []
    for dataset in exp.cfg["datasets"]:
        reason = external_unavailable(exp.cfg, dataset)
        if reason:
            log.warning(f"skipping {dataset}: {reason}")
            continue
        suite = f"{dataset}_c"
        long, _ = exp.evaluate(runs, suite)
        long["model"] = long.run.map(exp.label)
        long.to_csv(exp.out / f"{dataset}_per_seed.csv", index=False, float_format="%.5f")
        per_seed, _ = exp.summarize(long, suite)
        shared = long.assign(kind=long.corruption.eq("clean").map({True: "clean_acc_shared", False: "corr_acc_shared"}))
        shared = (100 * shared.groupby(["run", "seed", "kind"], sort=False).acc_shared.mean()).unstack("kind").reset_index()
        per_seed = per_seed.merge(shared, on=["run", "seed"])
        per_seed.insert(1, "model", per_seed.run.map(exp.label))
        per_seed.to_csv(exp.out / f"{dataset}_summary_per_seed.csv", index=False, float_format="%.4f")
        summary = mean_std(per_seed, ["run", "model"])
        n_img = int(long[long.corruption == "clean"].n.iloc[0])
        n_cls = len(set(load_mapping(dataset)["map"].values()))
        table = write_table(exp.out, dataset, summary, COLUMNS,
                            caption=f"Cross-dataset transfer GTSRB $\\rightarrow$ {dataset.upper()} ({n_img} test images "
                                    f"of the {n_cls} classes shared with GTSRB; \\%, {over_seeds(len(exp.seeds))}). "
                                    f"No {dataset.upper()} image is used for training. mCE is normalised by "
                                    f"{tex_escape(exp.label(exp.cfg['normalizer']))} on the same data.",
                            label=f"tab:cross-{dataset}")
        print(f"\n{dataset.upper()}\n" + table.to_string(index=False))
        done.append(dataset)
    if not done:
        log.warning("no cross-dataset test set was available; nothing written")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/RoadLens/scripts/exp6_tent_batchsize.py
"""Experiment 6 - test-time adaptation vs. test batch size (1, 16, 64, 200).

    python scripts/exp6_tent_batchsize.py

Methods: none (source model), bn (test-time BatchNorm statistics), tent (TENT). Each corruption of
GTSRB-C at the configured severities is an independent, shuffled online stream; the model is reset
before each stream. Writes to results/exp6_tent_batchsize/:
    per_seed.csv              accuracy / ECE per model, seed, method, batch size and corruption
    sweep.csv/.txt/.tex       mean corruption accuracy per model x method x batch size
    tent_batchsize.pdf        the same as a figure
"""

import _bootstrap  # noqa: F401
import pandas as pd

from roadlens.eval.metrics import mean_std
from roadlens.eval.tables import over_seeds, pm, tex_escape, write_latex
from roadlens.experiment import Experiment, make_parser
from roadlens.runs import ensure_trained, evaluate_run
from roadlens.utils import get_logger

log = get_logger()
METHOD_LABEL = {"none": "No adaptation", "bn": "Test-time BN", "tent": "TENT"}


def main():
    exp = Experiment(make_parser(__doc__, "configs/experiments/exp6_tent_batchsize.yaml").parse_args(),
                     "exp6_tent_batchsize")
    cfg = exp.cfg
    sizes, sevs, n_per_set = list(cfg["batch_sizes"]), list(cfg["severities"]), int(cfg["n_per_set"])
    n_bs1 = int(cfg.get("n_per_set_bs1", n_per_set))
    frames = []
    for seed in exp.seeds:
        for run in cfg["runs"]:
            ensure_trained(run, seed, exp.profile, exp.device, exp.wandb)
            for method in cfg["methods"]:
                for bs in sizes:
                    tta = None if method == "none" else {"method": method, "batch_size": bs, "lr": cfg["lr"]}
                    n = n_bs1 if bs == 1 else n_per_set     # every method in a column sees the same images
                    try:
                        df, _ = evaluate_run(run, seed, "gtsrb_c", exp.profile, exp.device, tta=tta,
                                             severities=sevs, n_per_set=n)
                    except ValueError as err:       # model without BatchNorm
                        log.warning(f"{run}: {err}")
                        break
                    frames.append(df.assign(run=run, seed=seed, method=method, batch_size=bs))
    long = pd.concat(frames, ignore_index=True)
    long["model"] = long.run.map(exp.label)
    long.to_csv(exp.out / "per_seed.csv", index=False, float_format="%.5f")

    corr = long[long.corruption != "clean"]
    per_seed = (100 * corr.groupby(["run", "model", "method", "batch_size", "seed"], sort=False)[["acc", "ece"]].mean()).reset_index()
    summary = mean_std(per_seed, ["run", "model", "method", "batch_size"], ["acc", "ece"])
    summary.to_csv(exp.out / "sweep.csv", index=False, float_format="%.4f")

    rows, tex_rows = [], []
    for (run, method), g in summary.groupby(["run", "method"], sort=False):
        g = g.set_index("batch_size")
        cells = {bs: (g.acc_mean[bs], g.acc_std[bs]) for bs in sizes}
        rows.append({"Model": exp.label(run), "Method": METHOD_LABEL.get(method, method),
                     **{f"bs={bs}": pm(m, s, 2) for bs, (m, s) in cells.items()}})
        tex_rows.append([tex_escape(exp.label(run)), METHOD_LABEL.get(method, method)]
                        + [pm(m, s, 2, tex=True) for m, s in cells.values()])
    table = pd.DataFrame(rows)
    (exp.out / "sweep.txt").write_text(table.to_string(index=False) + "\n", encoding="utf-8")
    sev_txt = ", ".join(str(s) for s in sevs)
    write_latex(exp.out / "sweep.tex", ["Model", "Method"] + [f"Batch {bs}" for bs in sizes], tex_rows,
                align="ll" + "c" * len(sizes),
                caption=f"Test-time adaptation on GTSRB-C (severity {sev_txt}): mean accuracy over 14 corruptions (\\%, "
                        f"{over_seeds(len(exp.seeds))}) as a function of the test batch size. Online adaptation on "
                        f"{n_per_set or 'all'} images per corruption ({n_bs1} for batch size 1), model reset "
                        f"between corruptions.",
                label="tab:tta-batchsize")
    print(table.to_string(index=False))

    from roadlens.eval.plots import plot_tta_sweep
    plot_tta_sweep(summary, sizes, METHOD_LABEL, exp.out / "tent_batchsize.pdf",
                   title=f"GTSRB-C severity {sev_txt}: accuracy vs. test batch size")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/RoadLens/scripts/exp7_improved.py
"""Experiment 7 - the v2 approach, one mechanism per row.

    python scripts/exp7_improved.py

Rows (configs/experiments/exp7_improved.yaml): the v1 models, then per-image input standardisation,
view experts (median / Gaussian low-pass copies of the input), the stacked degradation-aware gate,
and post-hoc temperature scaling. Every row is evaluated on GTSRB-C and on the cross-dataset sets
that are prepared. A second table measures test-time BatchNorm with a source-statistics prior at
small batch sizes.

"Val-C (sel.)" is the model-selection metric (validation corruptions on the select half of the
validation split; no test corruption, no test image). Choose between rows with that column.

Writes to results/exp7_improved/:
    per_seed.csv                  all summary metrics per step and seed
    improved.csv/.txt/.tex        mean +- std over seeds (booktabs table)
    per_corruption.csv/.txt       GTSRB-C accuracy per step and corruption (mean over severities and seeds)
    severity5.csv/.txt            the same at severity 5 only
    selection.csv                 trained runs: selected epoch (0 = the untrained model), temperature
    adaptation.csv/.txt/.tex      test-time BN, with and without the prior, per batch size
"""

import json

import _bootstrap  # noqa: F401
import pandas as pd

from roadlens.data.loading import external_unavailable
from roadlens.eval.metrics import mean_std
from roadlens.eval.tables import over_seeds, pm, tex_escape, write_latex, write_table
from roadlens.experiment import Experiment, make_parser
from roadlens.runs import ensure_trained, evaluate_run, load_run_config, run_dir, temperature
from roadlens.utils import corruption_names, get_logger

log = get_logger()
COLUMNS = {
    "sel_valc_acc": ("Val-C (sel.)", 2, "max"),
    "clean_acc": ("Clean acc.", 2, "max"),
    "corr_acc": ("Corr. acc.", 2, "max"),
    "mce": ("mCE", 1, "min"),
    "corr_f1": ("Corr. F1", 2, "max"),
    "clean_ece": ("Clean ECE", 2, "min"),
    "corr_ece": ("Corr. ECE", 2, "min"),
}
METHODS = [("none", "No adaptation", None), ("bn", "Test-time BN", 0.0), ("bn_prior", "Test-time BN + prior", None)]


def step_eval(exp, step, suite, **kw):
    long, _ = exp.evaluate([step["run"]], suite, tta=step.get("tta"), calibrate=bool(step.get("calibrate", False)), **kw)
    return long.assign(step=step["label"])


def ladder(exp, stem: str = "improved", title: str = "The v2 approach"):
    """Evaluate every step of exp.cfg['steps']; write <stem>.*, per_corruption.* and severity5.*. Returns per_seed."""
    steps, names = exp.cfg["steps"], corruption_names(exp.cfg)
    long = pd.concat([step_eval(exp, s, "gtsrb_c") for s in steps], ignore_index=True)
    per_seed, _ = exp.summarize(long, "gtsrb_c", key="step")

    sel = pd.concat([step_eval(exp, s, "gtsrb_sel") for s in steps], ignore_index=True)
    sel = sel.assign(kind=sel.corruption.map({"clean": "sel_clean_acc", "val_c": "sel_valc_acc"}))
    sel = (100 * sel.groupby(["step", "seed", "kind"], sort=False).acc.mean()).unstack("kind").reset_index()
    per_seed = per_seed.merge(sel, on=["step", "seed"])

    columns = dict(COLUMNS)
    for dataset in exp.cfg.get("datasets", []):
        reason = external_unavailable(exp.cfg, dataset)
        if reason:
            log.warning(f"skipping {dataset}: {reason}")
            continue
        suite = f"{dataset}_c"
        ext = pd.concat([step_eval(exp, s, suite) for s in steps], ignore_index=True)
        summ, _ = exp.summarize(ext, suite, key="step")
        shared = (100 * ext[ext.corruption != "clean"].groupby(["step", "seed"], sort=False).acc_shared.mean())
        summ = summ.merge(shared.rename("corr_acc_shared").reset_index(), on=["step", "seed"])
        keep = {"corr_acc": f"{dataset}_corr_acc", "corr_acc_shared": f"{dataset}_corr_acc_shared", "mce": f"{dataset}_mce"}
        per_seed = per_seed.merge(summ[["step", "seed", *keep]].rename(columns=keep), on=["step", "seed"])
        columns[f"{dataset}_corr_acc"] = (f"{dataset.upper()} corr.", 2, "max")
        columns[f"{dataset}_corr_acc_shared"] = (f"{dataset.upper()} corr. (shared)", 2, "max")
        columns[f"{dataset}_mce"] = (f"{dataset.upper()} mCE", 1, "min")

    per_seed.to_csv(exp.out / "per_seed.csv", index=False, float_format="%.4f")
    summary = mean_std(per_seed, "step")
    table = write_table(exp.out, stem, summary, columns, label_col="step", label_header="Configuration",
                        caption=f"{title}, one mechanism per row (\\%, {over_seeds(len(exp.seeds))}). "
                                f"Val-C (sel.): accuracy under the validation corruptions on the select half of the "
                                f"validation split, the model-selection metric. Corr.: mean over 14 corruptions "
                                f"$\\times$ 5 severities. mCE is normalised by "
                                f"{tex_escape(exp.label(exp.cfg['normalizer']))}.",
                        label=f"tab:{stem}")
    print(table.to_string(index=False))

    corr = long[long.corruption != "clean"]
    order = [s["label"] for s in steps]
    for part, rows in (("per_corruption", corr), ("severity5", corr[corr.severity == corr.severity.max()])):
        wide = (100 * rows.groupby(["step", "corruption"], sort=False).acc.mean()).unstack("corruption").reindex(order)[names]
        wide.to_csv(exp.out / f"{part}.csv", float_format="%.2f")
        (exp.out / f"{part}.txt").write_text(wide.round(1).to_string() + "\n", encoding="utf-8")
    return per_seed, long


def selection(exp):
    """What model selection did: the epoch kept for every trained run, and the fitted temperatures."""
    rows = []
    for seed in exp.seeds:
        for step in exp.cfg["steps"]:
            cfg = load_run_config(step["run"], exp.profile)
            done = json.loads((run_dir(cfg, seed) / "done.json").read_text())
            rows.append({"step": step["label"], "run": step["run"], "seed": seed,
                         "epochs_run": done.get("epochs_run"), "best_epoch": done.get("best_epoch"),
                         "best_val_c_acc": done.get("best_val_c_acc"),
                         "temperature": temperature(step["run"], seed, exp.profile, exp.device)
                         if step.get("calibrate") else None})
    pd.DataFrame(rows).to_csv(exp.out / "selection.csv", index=False, float_format="%.4f")


def adaptation(exp):
    cfg = exp.cfg
    sizes, sevs, prior = list(cfg["batch_sizes"]), list(cfg["severities"]), float(cfg["prior"])
    n_per_set, n_bs1 = int(cfg["n_per_set"]), int(cfg.get("n_per_set_bs1", cfg["n_per_set"]))
    frames = []
    for seed in exp.seeds:
        for run in cfg["adaptation_runs"]:
            ensure_trained(run, seed, exp.profile, exp.device, exp.wandb)
            for method, _, p in METHODS:
                for bs in sizes:
                    tta = None if method == "none" else {"method": "bn", "batch_size": bs, "prior": prior if p is None else p}
                    df, _ = evaluate_run(run, seed, "gtsrb_c", exp.profile, exp.device, tta=tta, severities=sevs,
                                         n_per_set=n_bs1 if bs == 1 else n_per_set)
                    frames.append(df.assign(run=run, seed=seed, method=method, batch_size=bs))
    long = pd.concat(frames, ignore_index=True)
    corr = long[long.corruption != "clean"]
    per_seed = (100 * corr.groupby(["run", "method", "batch_size", "seed"], sort=False).acc.mean()).reset_index()
    summary = mean_std(per_seed, ["run", "method", "batch_size"], ["acc"])
    summary.insert(1, "model", summary.run.map(exp.label))
    summary.to_csv(exp.out / "adaptation.csv", index=False, float_format="%.4f")

    label = {m: (f"{text} (N = {prior:g})" if m == "bn_prior" else text) for m, text, _ in METHODS}
    rows, tex_rows = [], []
    for (run, method), g in summary.groupby(["run", "method"], sort=False):
        g = g.set_index("batch_size")
        rows.append({"Model": exp.label(run), "Method": label[method],
                     **{f"bs={bs}": pm(g.acc_mean[bs], g.acc_std[bs], 2) for bs in sizes}})
        tex_rows.append([tex_escape(exp.label(run)), label[method]] + [pm(g.acc_mean[bs], g.acc_std[bs], 2, tex=True) for bs in sizes])
    table = pd.DataFrame(rows)
    (exp.out / "adaptation.txt").write_text(table.to_string(index=False) + "\n", encoding="utf-8")
    write_latex(exp.out / "adaptation.tex", ["Model", "Method"] + [f"Batch {bs}" for bs in sizes], tex_rows,
                align="ll" + "c" * len(sizes),
                caption=f"Test-time BatchNorm statistics with and without a source prior of {prior:g} pseudo-samples: "
                        f"mean accuracy over 14 corruptions at severity {', '.join(map(str, sevs))} (\\%, "
                        f"{over_seeds(len(exp.seeds))}); {n_per_set} images per corruption ({n_bs1} for batch size 1).",
                label="tab:tta-prior")
    print("\n" + table.to_string(index=False))


def main():
    exp = Experiment(make_parser(__doc__, "configs/experiments/exp7_improved.yaml").parse_args(), "exp7_improved")
    ladder(exp)
    selection(exp)
    adaptation(exp)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/RoadLens/scripts/exp8_v3.py
"""Experiment 8 - RoadLens v3: one mechanism per row, other architectures, self-supervised
pre-training and distillation into one fast network.

    python scripts/exp8_v3.py --profile pilot

Rows (configs/experiments/exp8_v3.yaml):
    local-global contrast normalisation stem       model.input_norm: local
    amplitude-phase recombination (APR) op set     augment.ops: [..., apr]
    EMA of the weights + GPU speed-ups             train.ema, channels_last, fused AdamW, cuDNN autotune
    native 64 px input                             model.input_size: 64
    SimCLR self-supervised pre-training            model.init_from: resnet18_simclr (roadlens/ssl.py)
    ConvNeXt-Atto / EfficientNet-B0 / MobileNetV3  model.name
    v3 view-expert ensemble                        ensemble_v3
    distilled students                             train.distill (teacher: ensemble_views_in)

Writes to results/exp8_v3/:
    v3.csv/.txt/.tex             accuracy, mCE, ECE, Val-C (sel.), BTSD per row (as in experiment 7)
    per_corruption.*, severity5.*  accuracy per row and corruption
    heatmaps/<row>.png           corruption x severity accuracy heatmap of every row
    per_family.csv/.txt          accuracy per corruption family (noise / blur / weather / digital) and at S5
    efficiency.csv/.txt          parameters, forward passes per image, training minutes, GPU throughput
"""

import json
import time

import _bootstrap  # noqa: F401
import numpy as np
import pandas as pd
import torch
from exp7_improved import ladder

from roadlens.data.loading import load_test
from roadlens.eval.evaluate import predict_logits
from roadlens.eval import plots
from roadlens.experiment import Experiment, make_parser
from roadlens.runs import ensure_trained, load_model, load_run_config, run_dir
from roadlens.utils import corruption_categories, get_logger

log = get_logger()


def per_family(exp, long):
    cats = corruption_categories(exp.cfg)
    corr = long[long.corruption != "clean"].assign(family=lambda d: d.corruption.map(cats))
    order = list(dict.fromkeys(s["label"] for s in exp.cfg["steps"]))
    fam = (100 * corr.groupby(["step", "family"], sort=False).acc.mean()).unstack("family")
    fam = fam[list(exp.cfg["corruptions"]["test"])]
    fam["fog S5"] = 100 * corr[(corr.corruption == "fog") & (corr.severity == 5)].groupby("step").acc.mean()
    fam["all S5"] = 100 * corr[corr.severity == 5].groupby("step").acc.mean()
    fam = fam.reindex(order)
    fam.to_csv(exp.out / "per_family.csv", float_format="%.2f")
    (exp.out / "per_family.txt").write_text(fam.round(1).to_string() + "\n", encoding="utf-8")
    print("\n" + fam.round(1).to_string())


def _sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


@torch.no_grad()
def throughput(model, x: np.ndarray, device, batch_size: int, repeats: int) -> tuple[float, float]:
    """(images per second at `batch_size`, milliseconds per single image at batch size 1), with AMP."""
    amp = device.type == "cuda"
    predict_logits(model, x[:batch_size], device, batch_size, amp)            # warm-up (kernels, allocator)
    best = 0.0
    for _ in range(repeats):
        _sync(device)
        t0 = time.perf_counter()
        predict_logits(model, x, device, batch_size, amp)
        _sync(device)
        best = max(best, len(x) / (time.perf_counter() - t0))
    one = x[:1]
    predict_logits(model, one, device, 1, amp)
    _sync(device)
    t0 = time.perf_counter()
    for _ in range(50):
        predict_logits(model, one, device, 1, amp)
    _sync(device)
    return best, 1000 * (time.perf_counter() - t0) / 50


def train_minutes(run: str, seed: int, profile) -> float:
    """Training time of a run including everything it depends on (branches, teacher, pre-training)."""
    cfg = load_run_config(run, profile)
    own = json.loads((run_dir(cfg, seed) / "done.json").read_text()).get("train_time_s", 0.0) / 60
    deps = list(cfg["model"].get("branches", [])) + [cfg["model"].get("init_from")]
    deps.append((cfg["train"].get("distill") or {}).get("teacher"))
    return own + sum(train_minutes(d, seed, profile) for d in deps if d)


def efficiency(exp):
    sp = exp.cfg.get("speed", {})
    seed = exp.seeds[0]
    runs = list(dict.fromkeys(s["run"] for s in exp.cfg["steps"]))
    x, _ = load_test(load_run_config(runs[0], exp.profile), "gtsrb")
    x = np.asarray(x[:int(sp.get("n_images", 2048))])
    rows = []
    for run in runs:
        ensure_trained(run, seed, exp.profile, exp.device)
        model, cfg = load_model(run, seed, exp.device, exp.profile)
        m = cfg["model"]
        passes = len(m.get("branches", [])) * len((m.get("fusion") or {}).get("views", ["raw"])) if m["name"] == "fusion" else 1
        ips, ms = throughput(model, x, exp.device, int(sp.get("batch_size", 256)), int(sp.get("repeats", 3)))
        rows.append({"run": run, "model": exp.label(run), "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
                     "forward_passes": passes, "input_px": m.get("input_size", 128) if m["name"] != "fusion" else None,
                     "train_min_incl_deps": train_minutes(run, seed, exp.profile),
                     "img_per_s": ips, "ms_per_image_bs1": ms})
        del model
        if exp.device.type == "cuda":
            torch.cuda.empty_cache()
    df = pd.DataFrame(rows)
    df.to_csv(exp.out / "efficiency.csv", index=False, float_format="%.3f")
    gpu = torch.cuda.get_device_name(exp.device) if exp.device.type == "cuda" else "CPU"
    (exp.out / "efficiency.txt").write_text(f"device: {gpu}, AMP, seed {seed}\n" + df.round(2).to_string(index=False)
                                           + "\n", encoding="utf-8")
    print(f"\n{gpu}\n" + df.round(2).to_string(index=False))


def main():
    exp = Experiment(make_parser(__doc__, "configs/experiments/exp8_v3.yaml").parse_args(), "exp8_v3")
    _, long = ladder(exp, stem="v3", title="The v3 approach")
    per_family(exp, long)
    plots.plot_corruption_heatmaps(long, "step", exp.out / "heatmaps")
    efficiency(exp)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/RoadLens/scripts/exp9_classical_unsupervised.py
"""Experiment 9 - other learning algorithms: classical supervised learners, a self-supervised
representation, and an unsupervised (label-free) robustness analysis.

    python scripts/exp9_classical_unsupervised.py --profile pilot

A. Classifiers, evaluated on exactly the test sets of experiment 2 (clean + 14 corruptions x 5):
     HOG + linear SVM, HOG + random forest, HOG + PCA + k-nearest neighbours  (classical, supervised)
     SimCLR features + kNN, SimCLR features + logistic regression            (representation learned
                                                                               without labels; labels
                                                                               only for the classifier)
B. Unsupervised analysis of representations: k-means (k = 43) on the clean test features of each
   model, then
     NMI clean        agreement of the clusters with the true classes (normalised mutual information)
     stability S3/S5  share of corrupted images assigned to the same cluster as their clean original,
                      mean over the 14 corruptions; measured WITHOUT any label
     NMI S5           agreement of the corrupted images' clusters with the classes at severity 5

Writes to results/exp9_classical_unsupervised/: classical.csv/.txt, classical_long.csv, clusters.csv/.txt,
heatmaps/<model>.png (corruption x severity accuracy of every classical learner)
"""

import time
from functools import partial

import _bootstrap  # noqa: F401
import cv2
import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, normalized_mutual_info_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

from roadlens.data.loading import load_split, u8_to_device
from roadlens.eval import plots
from roadlens.eval.evaluate import build_sets
from roadlens.eval.metrics import corruption_summary
from roadlens.experiment import Experiment, make_parser
from roadlens.runs import ensure_trained, load_model
from roadlens.ssl import embed
from roadlens.utils import get_logger

log = get_logger()


def hog_features(x: np.ndarray, p: dict) -> np.ndarray:
    """HOG of the grey-level image, [N, D] float32 (uint8 NHWC input)."""
    win, block, stride, cell = (int(p[k]) for k in ("win", "block", "stride", "cell"))
    hog = cv2.HOGDescriptor((win, win), (block, block), (stride, stride), (cell, cell), int(p["bins"]))
    out = []
    for img in np.asarray(x):
        g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        if g.shape[0] != win:
            g = cv2.resize(g, (win, win), interpolation=cv2.INTER_LINEAR)
        out.append(hog.compute(g).ravel())
    return np.stack(out).astype(np.float32)


def deep_features(model, x: np.ndarray, device) -> np.ndarray:
    return embed(model, u8_to_device(np.asarray(x), device), amp=device.type == "cuda").cpu().numpy()


def evaluate_classifiers(learners, featurize, sets) -> pd.DataFrame:
    """learners: [(name, kind, fitted classifier, fit seconds)]. Every set is featurised once."""
    rows, t_pred, n_pred = [], {n: 0.0 for n, *_ in learners}, 0
    for es in sets:
        x, y = es.load()
        f = featurize(x)
        n_pred += len(y)
        for name, kind, clf, fit_s in learners:
            t0 = time.perf_counter()
            pred = clf.predict(f)
            t_pred[name] += time.perf_counter() - t0
            rows.append({"dataset": es.dataset, "split": es.split, "corruption": es.corruption, "severity": es.severity,
                         "n": len(y), "acc": float((pred == y).mean()), "macro_f1": f1_score(y, pred, average="macro"),
                         "ece": np.nan, "model": name, "kind": kind, "fit_s": fit_s})
    df = pd.DataFrame(rows)
    return df.assign(predict_ms_per_image=df.model.map({n: 1000 * t / n_pred for n, t in t_pred.items()}))


def part_a(exp, sets, norm):
    cfg, c = exp.cfg, exp.cfg["classical"]
    seed = exp.seeds[0]
    xtr, ytr = load_split(cfg, "train")
    t0 = time.perf_counter()
    htr = hog_features(xtr, c["hog"])
    log.info(f"HOG: {htr.shape[1]} features, {len(htr)} training images in {time.perf_counter() - t0:.0f}s")
    learners = [
        ("HOG + linear SVM", "classical supervised",
         make_pipeline(StandardScaler(), LinearSVC(C=float(c["svm_c"]), dual="auto", max_iter=5000, random_state=seed))),
        ("HOG + random forest", "classical supervised",
         RandomForestClassifier(int(c["rf_trees"]), n_jobs=-1, random_state=seed)),
        ("HOG + PCA + kNN", "classical supervised",
         make_pipeline(StandardScaler(), PCA(int(c["pca_dim"]), random_state=seed), KNeighborsClassifier(int(c["knn_k"])))),
    ]
    fitted = []
    for name, kind, clf in learners:
        t0 = time.perf_counter()
        clf.fit(htr, ytr)
        fitted.append((name, kind, clf, time.perf_counter() - t0))
        log.info(f"{name}: fitted in {fitted[-1][3]:.0f}s")
    del htr
    frames = [evaluate_classifiers(fitted, lambda x: hog_features(x, c["hog"]), sets)]

    ssl_run = cfg.get("ssl_run")
    if ssl_run:
        ensure_trained(ssl_run, seed, exp.profile, exp.device)
        model, _ = load_model(ssl_run, seed, exp.device, exp.profile)
        bank = int(cfg.get("knn_bank", 0) or 0)
        idx = np.sort(np.random.RandomState(seed).permutation(len(xtr))[:bank]) if bank and bank < len(xtr) else slice(None)
        ftr = deep_features(model, xtr[idx], exp.device)
        fitted = []
        for name, clf in (("SimCLR features + kNN", KNeighborsClassifier(20, metric="cosine", weights="distance")),
                          ("SimCLR features + logistic regression", LogisticRegression(C=float(c["logreg_c"]), max_iter=2000))):
            t0 = time.perf_counter()
            clf.fit(ftr, ytr[idx])
            fitted.append((name, "self-supervised features (no labels in pre-training)", clf, time.perf_counter() - t0))
        frames.append(evaluate_classifiers(fitted, lambda x: deep_features(model, x, exp.device), sets))
        del model
    long = pd.concat(frames, ignore_index=True)
    long.to_csv(exp.out / "classical_long.csv", index=False, float_format="%.4f")
    plots.plot_corruption_heatmaps(long, "model", exp.out / "heatmaps")
    rows = []
    for name, g in long.groupby("model", sort=False):
        s = corruption_summary(g, norm)
        corr = g[g.corruption != "clean"]
        rows.append({"model": name, "kind": g.kind.iloc[0], "clean_acc": s["clean_acc"], "corr_acc": s["corr_acc"],
                     "s5_acc": 100 * corr[corr.severity == 5].acc.mean(), "mce": s.get("mce"),
                     "corr_f1": s["corr_f1"], "fit_s": g.fit_s.iloc[0],
                     "predict_ms_per_image": g.predict_ms_per_image.iloc[0]})
    table = pd.DataFrame(rows)
    table.to_csv(exp.out / "classical.csv", index=False, float_format="%.3f")
    (exp.out / "classical.txt").write_text(table.round(2).to_string(index=False) + "\n", encoding="utf-8")
    print("\n" + table.round(2).to_string(index=False))


def cluster_scores(f_clean, y, corrupted, k, n_init, seed):
    km = KMeans(k, n_init=n_init, random_state=seed).fit(f_clean)
    base = km.labels_
    out = {"nmi_clean": normalized_mutual_info_score(y, base)}
    for sev, feats in corrupted.items():
        assign = [km.predict(f) for f in feats]
        out[f"stability_s{sev}"] = 100 * float(np.mean([(a == base).mean() for a in assign]))
        out[f"nmi_s{sev}"] = float(np.mean([normalized_mutual_info_score(y, a) for a in assign]))
    return out


def part_b(exp, sets):
    cl, seed = exp.cfg["cluster"], exp.seeds[0]
    clean = next(es for es in sets if es.corruption == "clean")
    x0, y0 = clean.load()
    by_sev = {s: [es for es in sets if es.severity == s] for s in cl["severities"]}
    for group in by_sev.values():
        for es in group:
            assert (es.load()[1] == y0).all(), "corrupted sets must be aligned with the clean test set"
    rows = []
    hogp = exp.cfg["classical"]["hog"]

    def hog_pca(x, pca=None):
        f = hog_features(x, hogp)
        return f if pca is None else pca.transform(f)

    pca = make_pipeline(StandardScaler(), PCA(int(exp.cfg["classical"]["pca_dim"]), random_state=seed)).fit(hog_features(x0, hogp))
    rows.append({"representation": "HOG (+ PCA)", "kind": "hand-crafted",
                 **cluster_scores(hog_pca(x0, pca), y0, {s: [hog_pca(es.load()[0], pca) for es in g] for s, g in by_sev.items()},
                                  int(cl["k"]), int(cl["n_init"]), seed)})
    for run in cl["runs"]:
        try:
            ensure_trained(run, seed, exp.profile, exp.device)
        except Exception as e:                       # a run that cannot be trained here is reported, not fatal
            log.warning(f"skipping {run}: {e}")
            continue
        model, cfg = load_model(run, seed, exp.device, exp.profile)
        feat = partial(deep_features, model, device=exp.device)
        kind = "self-supervised" if cfg["train"].get("objective") == "simclr" else "supervised network"
        rows.append({"representation": exp.label(run), "kind": kind,
                     **cluster_scores(feat(x0), y0, {s: [feat(es.load()[0]) for es in g] for s, g in by_sev.items()},
                                      int(cl["k"]), int(cl["n_init"]), seed)})
        del model
        if exp.device.type == "cuda":
            torch.cuda.empty_cache()
    table = pd.DataFrame(rows)
    table.to_csv(exp.out / "clusters.csv", index=False, float_format="%.4f")
    (exp.out / "clusters.txt").write_text(table.round(3).to_string(index=False) + "\n", encoding="utf-8")
    print("\n" + table.round(3).to_string(index=False))


def main():
    exp = Experiment(make_parser(__doc__, "configs/experiments/exp9_classical_unsupervised.yaml").parse_args(),
                     "exp9_classical_unsupervised")
    exp.cfg.setdefault("labels", {})["resnet18_simclr"] = "SimCLR backbone (no labels)"
    exp.cfg["labels"].update({"resnet18_v3": "ResNet-18 v3", "resnet18_v3_ssl": "ResNet-18 v3, SimCLR pre-trained",
                              "convnext_atto_v3": "ConvNeXt-Atto v3", "student_v3": "v3 student (ResNet-18)"})
    sets = build_sets(exp.cfg, "gtsrb_c")
    norm = exp.normalizer_table("gtsrb_c")
    part_a(exp, sets, norm)
    part_b(exp, sets)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/RoadLens/scripts/make_figures.py
"""All figures, as vector PDF in results/figures/ (with the plotted numbers as CSV next to them).

    python scripts/make_figures.py                      # everything
    python scripts/make_figures.py --only curves confusion

    curves       severity_curves.pdf      accuracy vs. severity per corruption, per model
    confusion    confusion_<model>_<condition>.pdf   clean, and the baseline's worst corruption at severity 5
    reliability  reliability.pdf          reliability diagrams, clean and GTSRB-C
    fourier      fourier_heatmap.pdf      Fourier heat map of error (Yin et al. 2019)
    gradcam      gradcam.pdf              Grad-CAM on clean vs. corrupted samples

Uses the cached evaluations of experiments 1-2 and trains anything that is still missing.
"""

import _bootstrap  # noqa: F401
import numpy as np
import pandas as pd
import torch

from roadlens.data.classes import GTSRB_SHORT
from roadlens.data.loading import as_float, load_corrupted_set, load_test, u8_to_device
from roadlens.eval import plots
from roadlens.eval.fourier import fourier_heatmaps
from roadlens.eval.gradcam import grad_cam
from roadlens.eval.metrics import mean_std
from roadlens.experiment import Experiment, make_parser
from roadlens.runs import ensure_trained, load_model
from roadlens.utils import corruption_names, get_logger

log = get_logger()
ALL = ["curves", "confusion", "reliability", "fourier", "gradcam"]
ARCH = {"resnet18": "ResNet-18", "deit_small_patch16_224": "DeiT-S", "smallcnn": "SmallCNN",
        "swin_tiny_patch4_window7_224": "Swin-T"}


def curves(exp):
    runs = exp.cfg["curve_runs"]
    long, _ = exp.evaluate(runs, "gtsrb_c")
    plots.plot_severity_curves(long, runs, exp.cfg["labels"], corruption_names(exp.cfg), exp.out / "severity_curves.pdf")
    table = mean_std(long.assign(acc=100 * long.acc), ["run", "corruption", "severity"], ["acc"])
    table.insert(1, "model", table.run.map(exp.label))
    table.to_csv(exp.out / "severity_curves.csv", index=False, float_format="%.3f")


def confusion(exp):
    base, prop = exp.cfg["baseline"], exp.cfg["proposed"]
    long, arrays = exp.evaluate([base, prop], "gtsrb_c")
    top = max(long.severity)

    def worst(run):
        d = long[(long.run == run) & (long.severity == top)]
        return d.groupby("corruption").acc.mean().idxmin()

    conditions = {base: [("clean", 0), (worst(base), top)], prop: [("clean", 0), (worst(base), top)]}
    if worst(prop) != worst(base):
        conditions[prop].append((worst(prop), top))
    for run, conds in conditions.items():
        order = long[(long.run == run) & (long.seed == exp.seeds[0])].reset_index(drop=True)
        for corruption, sev in conds:
            row = order.index[(order.corruption == corruption) & (order.severity == sev)][0]
            cm = sum(arrays[(run, seed)]["confusion"][row] for seed in exp.seeds)
            name = "clean" if corruption == "clean" else f"{corruption}_s{sev}"
            title = f"{exp.label(run)}: " + ("clean test set" if corruption == "clean"
                                              else f"{corruption.replace('_', ' ')}, severity {sev}")
            if len(exp.seeds) > 1:
                title += f", {len(exp.seeds)} seeds pooled"
            plots.plot_confusion(cm, GTSRB_SHORT, title, exp.out / f"confusion_{run}_{name}.pdf")
            pd.DataFrame(cm, index=GTSRB_SHORT, columns=GTSRB_SHORT).to_csv(exp.out / f"confusion_{run}_{name}.csv")


def reliability(exp):
    runs = exp.cfg["reliability_runs"]
    long, arrays = exp.evaluate(runs, "gtsrb_c")
    bins, rows = {}, []
    for run in runs:
        order = long[(long.run == run) & (long.seed == exp.seeds[0])].reset_index(drop=True)
        clean_rows = order.index[order.corruption == "clean"].tolist()
        corr_rows = order.index[order.corruption != "clean"].tolist()
        for cond, sel in (("clean", clean_rows), ("GTSRB-C", corr_rows)):
            b = sum(arrays[(run, seed)]["bins"][sel].sum(0) for seed in exp.seeds)
            bins[(run, cond)] = b
            for i, (count, conf_sum, correct) in enumerate(b):
                rows.append({"run": run, "condition": cond, "bin": i, "count": int(count),
                             "mean_confidence": conf_sum / count if count else np.nan,
                             "accuracy": correct / count if count else np.nan})
    plots.plot_reliability(bins, ["clean", "GTSRB-C"], runs, exp.cfg["labels"], exp.out / "reliability.pdf")
    pd.DataFrame(rows).to_csv(exp.out / "reliability.csv", index=False, float_format="%.5f")


def fourier(exp):
    f = exp.cfg["fourier"]
    models = {}
    for run in f["runs"]:
        ensure_trained(run, f["seed"], exp.profile, exp.device, exp.wandb)
        models[run] = load_model(run, f["seed"], exp.device, exp.profile)[0]
    x, y = load_test(exp.cfg)
    idx = np.sort(np.random.RandomState(0).permutation(len(x))[:int(f["n_images"])])
    maps, clean = fourier_heatmaps(models, x[idx], y[idx], exp.device, eps=float(f["eps"]), stride=int(f["stride"]))
    plots.plot_fourier(maps, clean, exp.cfg["labels"], float(f["eps"]), exp.out / "fourier_heatmap.pdf")
    for run, m in maps.items():
        pd.DataFrame(m).to_csv(exp.out / f"fourier_heatmap_{run}.csv", index=False, header=False, float_format="%.4f")


def gradcam(exp):
    g = exp.cfg["gradcam"]
    x, y = load_test(exp.cfg)
    idx = np.sort(np.random.RandomState(1).permutation(len(x))[:int(g["n_samples"])])
    sev = int(g["severity"])
    names = [g["corruptions"][j % len(g["corruptions"])] for j in range(len(idx))]
    clean = as_float(u8_to_device(x[idx], exp.device))
    corrupted = as_float(u8_to_device(
        np.stack([np.asarray(load_corrupted_set(exp.cfg, "gtsrb", n, sev)[0][i]) for n, i in zip(names, idx)]), exp.device))
    labels = torch.from_numpy(y[idx])

    def to_img(t):
        return t.permute(1, 2, 0).cpu().numpy()

    rows = {"clean": [], "corrupted": []}       # condition -> list of (row label, [cell per sample])
    for run in g["runs"]:
        ensure_trained(run, g["seed"], exp.profile, exp.device, exp.wandb)
        model = load_model(run, g["seed"], exp.device, exp.profile)[0]
        targets = model.cam_targets()
        if not targets:
            log.warning(f"{run}: no Grad-CAM layer known for this architecture; skipped")
            continue
        for cond, batch in (("clean", clean), ("corrupted", corrupted)):
            cams, logits = grad_cam(model, targets, batch)
            pred = logits.argmax(dim=1).cpu()
            for t, cam in zip(targets, cams):
                label = exp.label(run) if len(targets) == 1 else f"{exp.label(run)}\n{ARCH.get(t.name, t.name)} branch"
                cells = [(plots.cam_overlay(to_img(batch[j]), cam[j].cpu().numpy()),
                          f"{GTSRB_SHORT[pred[j]]} " + ("✓" if pred[j] == labels[j] else "✗"))
                         for j in range(len(idx))]
                rows[cond].append((label, cells))
        del model
    row_labels, columns = [], [{"title": f"{GTSRB_SHORT[int(labels[j])]}", "cells": []} for j in range(len(idx))]
    for cond, batch in (("clean", clean), ("corrupted", corrupted)):
        row_labels.append(f"input\n({cond})")
        for j in range(len(idx)):
            columns[j]["cells"].append((to_img(batch[j]), "clean" if cond == "clean" else f"{names[j].replace('_', ' ')} s{sev}"))
        for label, cells in rows[cond]:
            row_labels.append(label)
            for j in range(len(idx)):
                columns[j]["cells"].append(cells[j])
    plots.plot_gradcam(columns, row_labels, exp.out / "gradcam.pdf")


def main():
    ap = make_parser(__doc__, "configs/experiments/figures.yaml")
    ap.add_argument("--only", nargs="+", choices=ALL, default=ALL)
    args = ap.parse_args()
    exp = Experiment(args, "figures")
    for name in args.only:
        log.info(f"figure: {name}")
        globals()[name](exp)
    log.info(f"figures -> {exp.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/RoadLens/scripts/make_gtsrb_c.py
"""Generate GTSRB-C (14 corruptions x 5 severities of the test set) and validation-C, once.

    python scripts/make_gtsrb_c.py                 # full: 70 sets, ~10.9 GB at 64 px
    python scripts/make_gtsrb_c.py --profile pilot # the 2,000-image subset used by --profile pilot

Re-running skips finished sets. Cross-dataset corrupted sets: add --dataset btsd / ctsd.
"""

import argparse
import time

import _bootstrap  # noqa: F401

from roadlens.data.corrupted import generate_corrupted_sets, generate_val_c, val_c_path
from roadlens.data.gtsrb import load_gtsrb_cfg
from roadlens.data.loading import corrupted_dir, load_test
from roadlens.utils import apply_profile, corruption_names, data_root, get_logger, load_config

log = get_logger()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/data.yaml")
    ap.add_argument("--dataset", default="gtsrb", choices=["gtsrb", "btsd", "ctsd"])
    ap.add_argument("--workers", type=int, default=0, help="0 = all cores (max 8)")
    ap.add_argument("--limit", type=int, default=None, help="only the first N test images (same as data.test_subset=N)")
    ap.add_argument("--profile", default=None, help="smoke | pilot: generate that profile's test subset")
    ap.add_argument("--overwrite", action="store_true")
    ap.add_argument("--skip-val-c", action="store_true")
    ap.add_argument("--val-c-only", action="store_true", help="only build validation-C (needed before training)")
    ap.add_argument("-o", "--override", dest="overrides", action="append", default=[],
                    metavar="KEY=VALUE", help="config override, repeatable (e.g. -o corruptions.seed=1)")
    args = ap.parse_args()
    cfg = apply_profile(load_config(args.config, args.overrides), args.profile)
    if args.limit is not None:
        cfg["data"]["test_subset"] = args.limit
    root, c = data_root(cfg), cfg["corruptions"]

    if args.val_c_only:
        xv, yv = load_gtsrb_cfg(cfg, root, "val")
        generate_val_c(xv, yv, val_c_path(root, cfg["data"]["image_size"]), c["val"], c["val_severities"],
                       c["seed"], overwrite=args.overwrite)
        return

    x, y = load_test(cfg, args.dataset)
    out_dir = corrupted_dir(cfg, args.dataset)
    names = corruption_names(cfg)
    gb = len(names) * len(c["severities"]) * x.nbytes / 1e9
    log.info(f"{args.dataset}-C: {len(names)} corruptions x {len(c['severities'])} severities x "
             f"{len(x)} images -> {out_dir} ({gb:.1f} GB)")
    t0 = time.time()
    generate_corrupted_sets(x, y, out_dir, names, c["severities"], c["seed"],
                            workers=args.workers, overwrite=args.overwrite)
    log.info(f"done in {(time.time() - t0) / 60:.1f} min")

    if args.dataset == "gtsrb" and not args.skip_val_c:
        # validation-C is always built from the full validation split; debug runs take a prefix of it
        xv, yv = load_gtsrb_cfg(cfg, root, "val")
        generate_val_c(xv, yv, val_c_path(root, cfg["data"]["image_size"]), c["val"], c["val_severities"],
                       c["seed"], overwrite=args.overwrite)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/RoadLens/scripts/make_heatmaps.py
"""Corruption x severity accuracy heatmaps (PNG), one per model, from the cached evaluations.

    python scripts/make_heatmaps.py --profile pilot                      # every trained run that has GTSRB-C results
    python scripts/make_heatmaps.py --profile pilot --runs fusion_v2 student_v3
    python scripts/make_heatmaps.py --from-csv results/exp9_classical_unsupervised/classical_long.csv

No model is evaluated: the figures are drawn from <run dir>/eval/gtsrb_c.csv (written by the experiment
scripts) or from a long CSV with columns model, corruption, severity, acc. Output:
    results[_profile]/heatmaps/<run>.png
"""

import argparse
from pathlib import Path

import _bootstrap  # noqa: F401
import pandas as pd

from roadlens.eval import plots
from roadlens.runs import load_run_config, run_dir
from roadlens.utils import REPO_ROOT, apply_profile, load_config, results_root


def label_of(cfg_labels: dict, run: str) -> str:
    return cfg_labels.get(run, run)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--profile", default=None, help="smoke | pilot")
    ap.add_argument("--runs", nargs="+", default=None, help="run names (default: every run with GTSRB-C results)")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--from-csv", default=None, help="long CSV (model, corruption, severity, acc) instead of run folders")
    ap.add_argument("--dataset", default="gtsrb", choices=["gtsrb", "btsd"])
    ap.add_argument("--out", default=None, help="output folder (default: results[_profile]/heatmaps)")
    args = ap.parse_args()

    base = apply_profile(load_config(REPO_ROOT / "configs" / "experiments" / "exp8_v3.yaml"), args.profile)
    labels = base.get("labels", {})
    out = Path(args.out) if args.out else results_root(base) / "heatmaps"
    name = "GTSRB" if args.dataset == "gtsrb" else "BTSD"
    written = []

    if args.from_csv:
        long = pd.read_csv(args.from_csv)
        for model, g in long.groupby("model", sort=False):
            safe = "".join(ch if ch.isalnum() else "_" for ch in model).strip("_").lower()
            written.append(plots.plot_corruption_heatmap(g, f"{name} - {model} accuracy (%)", out / f"{safe}.png"))
    else:
        runs = args.runs
        if runs is None:
            runs = sorted(p.name for p in (run_dir(load_run_config("resnet18_base", args.profile), 0).parent.parent).iterdir()
                          if p.is_dir())
        for run in runs:
            cfg = load_run_config(run, args.profile)
            csv = run_dir(cfg, args.seed) / "eval" / f"{args.dataset}_c.csv"
            if not csv.exists():
                print(f"skip {run}: no {csv.name}")
                continue
            written.append(plots.plot_corruption_heatmap(pd.read_csv(csv), f"{name} - {label_of(labels, run)} accuracy (%)",
                                                         out / f"{run}.png"))
    for p in written:
        print("wrote", p)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/RoadLens/scripts/plot_corruptions.py
"""Visual + numeric sanity check of the corruptions at the working resolution.

    python scripts/plot_corruptions.py

Writes to results/corruptions/:
  grid_<i>.pdf            14 test corruptions x 5 severities for one test image (several images)
  grid_val_c.pdf          the 4 held-out validation corruptions x 5 severities
  package_vs_roadlens.pdf imagecorruptions defaults vs. the 64 px calibration used here
  corruption_stats.csv    mean PSNR (dB) vs. clean per corruption/severity, both parameter sets
"""

import argparse

import _bootstrap  # noqa: F401
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from roadlens.data import corruptions as C
from roadlens.data.classes import GTSRB_SHORT
from roadlens.data.gtsrb import load_gtsrb_cfg
from roadlens.utils import REPO_ROOT, data_root, get_logger, load_config, corruption_names

log = get_logger()


def _show(ax, img):
    ax.imshow(img, interpolation="nearest")
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


def severity_grid(img, names, seed, title, path):
    fig, axes = plt.subplots(len(names), 6, figsize=(6 * 1.05 + 1.3, len(names) * 1.05 + 0.5), squeeze=False)
    for r, name in enumerate(names):
        _show(axes[r, 0], img)
        axes[r, 0].set_ylabel(name.replace("_", " "), rotation=0, ha="right", va="center", fontsize=8)
        for s in range(1, 6):
            _show(axes[r, s], C.corrupt(img, name, s, C.derive_seed(seed, name, s, 0)))
    for s, t in enumerate(["clean"] + [f"severity {i}" for i in range(1, 6)]):
        axes[0, s].set_title(t, fontsize=8)
    fig.suptitle(title, fontsize=9)
    fig.tight_layout(rect=(0, 0, 1, 0.99), h_pad=0.15, w_pad=0.15)
    fig.savefig(path)
    fig.savefig(path.with_suffix(".png"), dpi=110)
    plt.close(fig)


def comparison_grid(img, names, seed, path):
    sevs = [1, 3, 5]
    fig, axes = plt.subplots(len(names), 7, figsize=(7 * 1.05 + 1.3, len(names) * 1.05 + 0.7), squeeze=False)
    for r, name in enumerate(names):
        _show(axes[r, 0], img)
        axes[r, 0].set_ylabel(name.replace("_", " "), rotation=0, ha="right", va="center", fontsize=8)
        for k, s in enumerate(sevs):
            sd = C.derive_seed(seed, name, s, 0)
            _show(axes[r, 1 + k], C.package_default(img, name, s, sd))
            _show(axes[r, 4 + k], C.corrupt(img, name, s, sd))
    axes[0, 0].set_title("clean", fontsize=8)
    for k, s in enumerate(sevs):
        axes[0, 1 + k].set_title(f"package s{s}", fontsize=8)
        axes[0, 4 + k].set_title(f"RoadLens s{s}", fontsize=8)
    fig.suptitle("imagecorruptions defaults (ImageNet-scale) vs. 64 px calibration", fontsize=9)
    fig.tight_layout(rect=(0, 0, 1, 0.99), h_pad=0.15, w_pad=0.15)
    fig.savefig(path)
    fig.savefig(path.with_suffix(".png"), dpi=110)
    plt.close(fig)


def psnr(a, b):
    mse = np.mean((a.astype(np.float32) - b.astype(np.float32)) ** 2, axis=(1, 2, 3))
    return 10 * np.log10(255.0 ** 2 / np.maximum(mse, 1e-8))


def stats(x, names, seed):
    rows = []
    for name in names:
        for s in range(1, 6):
            ours = np.stack([C.corrupt(im, name, s, C.derive_seed(seed, name, s, i)) for i, im in enumerate(x)])
            pkg = np.stack([C.package_default(im, name, s, C.derive_seed(seed, name, s, i))
                            for i, im in enumerate(x)])
            rows.append({"corruption": name, "severity": s,
                         "psnr_roadlens": float(psnr(x, ours).mean()),
                         "psnr_package_default": float(psnr(x, pkg).mean())})
    return pd.DataFrame(rows)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/data.yaml")
    ap.add_argument("--classes", type=int, nargs="+", default=[2, 14, 25, 38],
                    help="GTSRB class ids to draw one example each from")
    ap.add_argument("--stats-n", type=int, default=300, help="test images used for the PSNR table (0 = skip)")
    ap.add_argument("-o", "--override", dest="overrides", action="append", default=[],
                    metavar="KEY=VALUE", help="config override, repeatable")
    args = ap.parse_args()
    cfg = load_config(args.config, args.overrides)
    seed = cfg["corruptions"]["seed"]
    x, y = load_gtsrb_cfg(cfg, data_root(cfg), "test")
    out = REPO_ROOT / "results" / "corruptions"
    out.mkdir(parents=True, exist_ok=True)
    names = corruption_names(cfg)

    rng = np.random.RandomState(0)
    picks = [int(rng.choice(np.flatnonzero(y == c))) for c in args.classes]
    for k, i in enumerate(picks):
        severity_grid(x[i], names, seed, f"GTSRB-C at {x.shape[1]} px - test image {i} ({GTSRB_SHORT[y[i]]})",
                      out / f"grid_{k}.pdf")
    severity_grid(x[picks[0]], cfg["corruptions"]["val"], seed,
                  "validation-C corruptions (held out from the test set)", out / "grid_val_c.pdf")
    comparison_grid(x[picks[0]], names, seed, out / "package_vs_roadlens.pdf")
    log.info(f"figures -> {out}")

    if args.stats_n:
        sub = x[rng.choice(len(x), args.stats_n, replace=False)]
        df = stats(sub, names + cfg["corruptions"]["val"], seed)
        df.to_csv(out / "corruption_stats.csv", index=False, float_format="%.2f")
        print(df.pivot(index="corruption", columns="severity", values="psnr_roadlens").round(1).to_string())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/RoadLens/scripts/prepare_data.py
"""Download datasets and cache them at the working resolution.

    python scripts/prepare_data.py                       # GTSRB only
    python scripts/prepare_data.py --datasets gtsrb btsd ctsd
"""

import argparse

import _bootstrap  # noqa: F401
import numpy as np

from roadlens.data.external import load_external, prepare_external
from roadlens.data.gtsrb import load_gtsrb_cfg, prepare_gtsrb
from roadlens.utils import data_root, get_logger, load_config

log = get_logger()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/data.yaml")
    ap.add_argument("--datasets", nargs="+", default=["gtsrb"], choices=["gtsrb", "btsd", "ctsd"])
    ap.add_argument("-o", "--override", dest="overrides", action="append", default=[],
                    metavar="KEY=VALUE", help="config override, repeatable (e.g. -o data.image_size=32)")
    args = ap.parse_args()
    cfg = load_config(args.config, args.overrides)
    root, d = data_root(cfg), cfg["data"]
    log.info(f"data root: {root}")

    if "gtsrb" in args.datasets:
        prepare_gtsrb(root, d["image_size"], d["crop_roi"])
        for split in ("train", "val", "test"):
            x, y = load_gtsrb_cfg(cfg, root, split)
            counts = np.bincount(y, minlength=d["num_classes"])
            log.info(f"gtsrb {split:5s}: {x.shape}  classes={len(np.unique(y))}  "
                     f"min/max per class={counts.min()}/{counts.max()}")
    for name in ("btsd", "ctsd"):
        if name in args.datasets:
            prepare_external(root, name, d["image_size"], d["crop_roi"])
            x, y, y_src = load_external(root, name, d["image_size"], d["crop_roi"],
                                        mapped_only=False, allow_unverified=True)
            log.info(f"{name}: {len(x)} images, {len(np.unique(y_src))} source classes; "
                     f"{int((y >= 0).sum())} images in {len(np.unique(y[y >= 0]))} classes shared with GTSRB")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/RoadLens/scripts/train.py
"""Train one named run (configs/runs/<name>.yaml) for one or more seeds.

    python scripts/train.py --run resnet18_base --seeds 0 1 2
    python scripts/train.py --run fusion_full --seeds 0          # trains its two branches first if needed

Finished runs are skipped and interrupted ones resume from their last completed epoch. The experiment
scripts call the same code, so running this first is optional.
"""

import argparse

import _bootstrap  # noqa: F401

from roadlens.runs import ensure_trained
from roadlens.utils import get_device, get_logger

log = get_logger()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", required=True, nargs="+", help="run name(s): file stems in configs/runs/")
    ap.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
    ap.add_argument("--profile", default=None, help="smoke | pilot (configs/profiles/); default: full scale")
    ap.add_argument("--device", default=None)
    ap.add_argument("--wandb", action="store_true", help="also log to Weights & Biases")
    args = ap.parse_args()
    device = get_device(args.device)
    for name in args.run:
        for seed in args.seeds:
            info = ensure_trained(name, seed, args.profile, device, args.wandb)
            if info.get("skipped"):
                log.info(f"{name} seed {seed}: nothing to train")
            else:
                log.info(f"{name} seed {seed}: best val-C {100 * info['best_val_c_acc']:.2f}% at epoch "
                         f"{info['best_epoch']}/{info['epochs_run']}, {info['train_time_s'] / 60:.1f} min")


if __name__ == "__main__":
    main()


### `tests/`

In [ ]:
%%writefile /content/RoadLens/tests/conftest.py
import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
warnings.filterwarnings("ignore", message=".*pkg_resources.*")


In [ ]:
%%writefile /content/RoadLens/tests/test_augment.py
"""Batched augmentation ops against PIL / torchvision references, and AugMix invariants."""

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from torchvision.transforms import functional as TVF
from torchvision.transforms import InterpolationMode

from roadlens.augment import AugMix, ColorJitter, TrainAugment, jsd_loss, resolve_ops
from roadlens.augment import ops as O


def batch(n=6, size=64, seed=0):
    g = torch.Generator().manual_seed(seed)
    low = torch.rand(n, 3, 8, 8, generator=g) * 0.8 + 0.1          # keep away from 0/1 so ranges are not full
    x = F.interpolate(low, size=size, mode="bicubic", align_corners=False).clamp(0.05, 0.9)
    return (x * 255).round() / 255


def to_pil(x):
    return [Image.fromarray((img.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)) for img in x]


def from_pil(imgs):
    return torch.stack([torch.from_numpy(np.asarray(im)).permute(2, 0, 1).float() / 255 for im in imgs])


def max_diff_255(a, b):
    return float((a - b).abs().max() * 255)


def test_autocontrast_matches_pil():
    x = batch()
    ref = from_pil([ImageOps.autocontrast(im) for im in to_pil(x)])
    assert max_diff_255(O.autocontrast(x, torch.zeros(len(x))), ref) <= 1.01


def test_equalize_matches_pil():
    x = batch()
    ref = from_pil([ImageOps.equalize(im) for im in to_pil(x)])
    assert max_diff_255(O.equalize(x, torch.zeros(len(x))), ref) <= 1.01


def test_posterize_matches_pil():
    x = batch(4)
    for level, bits in [(0.0, 4), (0.3, 3), (0.5, 2), (0.76, 1), (1.0, 1)]:
        ref = from_pil([ImageOps.posterize(im, bits) for im in to_pil(x)])
        assert max_diff_255(O.posterize(x, torch.full((4,), level)), ref) == 0, (level, bits)


def test_solarize_matches_pil():
    x = batch(4)
    for level in (0.0, 0.25, 0.5, 1.0):
        thr = 256 - int(level * 256)
        ref = from_pil([ImageOps.solarize(im, thr) for im in to_pil(x)])
        assert max_diff_255(O.solarize(x, torch.full((4,), level)), ref) < 1e-3, level


def test_ops_are_identity_at_level_zero():
    x = batch()
    zero = torch.zeros(len(x))
    for name in ["solarize", "rotate", "shear_x", "shear_y", "translate_x", "translate_y",
                 "fourier_noise", "spectral_filter", "spatial_warp", "color_curve"]:
        assert max_diff_255(O.OPS[name](x, zero), x) < 0.01, name


def test_rotate_matches_torchvision():
    x = batch(2)
    torch.manual_seed(0)
    out = O.rotate(x, torch.ones(2))                                  # +-30 degrees
    refs = [TVF.rotate(x, a, interpolation=InterpolationMode.BILINEAR) for a in (30.0, -30.0)]
    for i in range(2):
        assert min(max_diff_255(out[i], r[i]) for r in refs) < 1.0


def test_translate_moves_by_a_third():
    x = torch.zeros(1, 3, 60, 60)
    x[:, :, 30, 30] = 1.0
    torch.manual_seed(0)
    out = O.translate_x(x, torch.ones(1))
    col = int(out[0, 0, 30].argmax())
    assert abs(abs(col - 30) - 20) <= 1 and float(out[0, 0, 30].max()) > 0.9


def test_all_ops_keep_shape_range_and_batch_independence():
    x = batch(8)
    for name, op in O.OPS.items():
        torch.manual_seed(1)
        out = op(x, torch.full((8,), 0.3))
        assert out.shape == x.shape and out.dtype == x.dtype, name
        assert float(out.min()) >= 0 and float(out.max()) <= 1, name
        assert torch.isfinite(out).all(), name


def test_spectral_filter_keeps_flat_images_flat():
    x = torch.full((4, 3, 32, 32), 0.4)
    torch.manual_seed(0)
    assert max_diff_255(O.spectral_filter(x, torch.ones(4)), x) < 0.01


def test_spatial_warp_displacement_is_bounded():
    # warp a horizontal ramp: the value change equals the horizontal displacement (as a fraction of width)
    w = 64
    ramp = ((torch.arange(w) + 0.5) / w).view(1, 1, 1, w).expand(4, 3, w, w).contiguous()
    torch.manual_seed(0)
    out = O.spatial_warp(ramp, torch.ones(4), max_disp=0.1)
    shift = (out - ramp).abs().amax(dim=(1, 2, 3))
    assert float(shift.max()) <= 0.1 + 1e-3 and float(shift.min()) > 0.01


def test_color_curve_fixes_black_and_white():
    x = torch.zeros(2, 3, 8, 8)
    x[:, :, 4:] = 1.0
    torch.manual_seed(0)
    assert max_diff_255(O.color_curve(x, torch.ones(2)), x) < 0.01


def test_fourier_noise_amplitude():
    x = torch.full((16, 3, 64, 64), 0.5)
    torch.manual_seed(0)
    out = O.fourier_noise(x, torch.full((16,), 0.4), max_amp=0.5)
    amp = (out - x).abs().amax(dim=(2, 3))
    assert float(amp.max()) <= 0.2 + 1e-4 and float(amp.min()) > 0.19


def test_augmix_is_deterministic_bounded_and_changes_images():
    x = batch(16)
    aug = AugMix(resolve_ops(["augmix", "prime"]), width=3, depth=(1, 3), alpha=1.0, severity=3)
    torch.manual_seed(7)
    a, a2 = aug(x, 2)
    torch.manual_seed(7)
    b, b2 = aug(x, 2)
    assert torch.equal(a, b) and torch.equal(a2, b2) and not torch.equal(a, a2)
    assert a.shape == x.shape and float(a.min()) >= 0 and float(a.max()) <= 1
    per_image = (a - x).abs().flatten(1).mean(1)
    assert (per_image > 1e-4).float().mean() > 0.8


def test_merged_affine_equals_the_separate_ops():
    x = batch(6)
    level = torch.linspace(0.1, 1.0, 6)
    for name, kind in O.AFFINE_KIND.items():
        torch.manual_seed(3)
        separate = O.OPS[name](x, level)
        torch.manual_seed(3)
        merged = O.affine(x, level, torch.full((6,), kind))
        assert torch.allclose(separate, merged, atol=1e-6), name


def test_augmix_applies_each_op_only_to_its_own_rows():
    # one marker op per group type: rows must come back in their original order after the per-op sorting
    from roadlens.augment import ops
    ops.OPS["_plus"] = lambda x, level: (x + 0.25).clamp(0, 1)
    ops.OPS["_zero"] = lambda x, level: torch.zeros_like(x)
    try:
        x = torch.rand(32, 3, 8, 8) * 0.5
        torch.manual_seed(0)
        out = AugMix(["_plus", "_zero"], width=1, depth=(1, 1), alpha=1.0)(x)[0]
        # width 1, depth 1: out = (1 - m) x + m op(x) with op in {x + .25, 0} -> per image either brighter or darker
        delta = (out - x).flatten(1)
        brighter, darker = (delta >= -1e-6).all(1), (delta <= 1e-6).all(1)
        assert bool((brighter | darker).all()) and bool(brighter.any()) and bool(darker.any())
        ratio = delta[darker] / x.flatten(1)[darker]                 # = -m, constant within an image
        assert float(ratio.std(dim=1).max()) < 1e-4
    finally:
        del ops.OPS["_plus"], ops.OPS["_zero"]


def test_augmix_convexity_with_identity_ops():
    # with ops that do nothing, any Dirichlet/Beta mixing must return the input
    from roadlens.augment import ops
    ops.OPS["_identity"] = lambda x, level: x
    try:
        x = batch(8)
        out = AugMix(["_identity"], width=3, depth=(1, 3))(x)[0]
        assert max_diff_255(out, x) < 1e-2
    finally:
        del ops.OPS["_identity"]


def test_resolve_ops():
    assert len(resolve_ops("augmix")) == 9
    assert len(resolve_ops(["augmix", "prime"])) == 13
    no_geom = resolve_ops(["augmix", "prime"], exclude=["geometric"])
    assert "rotate" not in no_geom and "spatial_warp" not in no_geom and "fourier_noise" in no_geom
    assert resolve_ops(["rotate", "rotate", "augmix"]).count("rotate") == 1


def test_train_augment_views():
    x = batch(8)
    cj = {"brightness": 0.2, "contrast": 0.2, "saturation": 0.2, "hue": 0.02}
    assert len(TrainAugment({"mode": "none", "color_jitter": cj})(x)) == 1
    assert len(TrainAugment({"mode": "mix", "ops": "augmix", "color_jitter": cj})(x)) == 1
    views = TrainAugment({"mode": "jsd", "ops": ["augmix", "prime"], "color_jitter": cj})(x)
    assert len(views) == 3 and not torch.equal(views[1], views[2])


def test_color_jitter_is_mild_and_identity_when_disabled():
    x = batch(8)
    torch.manual_seed(0)
    out = ColorJitter(0.2, 0.2, 0.2, 0.02)(x)
    assert float((out - x).abs().max()) < 0.5 and float((out - x).abs().mean()) > 1e-3
    assert torch.equal(ColorJitter(0, 0, 0, 0)(x), x)


def test_jsd_loss():
    z = torch.randn(10, 43)
    assert float(jsd_loss([z, z, z])) < 1e-6
    assert float(jsd_loss([z, torch.randn(10, 43), torch.randn(10, 43)])) > 0.05


In [ ]:
%%writefile /content/RoadLens/tests/test_engine.py
"""Training loop on synthetic data: it learns, early-stops, and an interrupted run resumes exactly."""

import numpy as np
import pandas as pd
import pytest
import torch

import roadlens.engine as engine
from roadlens.models import SmallCNN


def synthetic(n, seed):
    """Three classes that differ in which colour channel is bright."""
    rng = np.random.RandomState(seed)
    y = rng.randint(0, 3, n)
    x = rng.randint(0, 80, (n, 16, 16, 3))
    x[np.arange(n), :, :, y] += 120
    return x.astype(np.uint8), y.astype(np.int64)


@pytest.fixture
def cfg(monkeypatch):
    data = {"train": synthetic(256, 0), "val": synthetic(64, 1)}
    monkeypatch.setattr(engine, "load_split", lambda cfg, split: data[split])
    monkeypatch.setattr(engine, "load_val_c_split", lambda cfg: (*synthetic(64, 2), None, None))
    return {
        "name": "toy", "data": {"num_classes": 3},
        "train": {"epochs": 4, "batch_size": 32, "lr": 3e-3, "weight_decay": 0.01, "warmup_epochs": 1,
                  "label_smoothing": 0.1, "grad_clip": 1.0, "patience": 0, "amp": False, "eval_batch_size": 64},
        "augment": {"mode": "jsd", "ops": ["augmix", "prime"], "width": 2, "depth": [1, 2], "severity": 3,
                    "jsd_weight": 12.0, "color_jitter": {"brightness": 0.1, "contrast": 0.1, "saturation": 0.1, "hue": 0.01}},
    }


def model(seed=0):
    torch.manual_seed(seed)
    return SmallCNN(3, width=8, dropout=0.1)


def test_training_learns_and_writes_outputs(cfg, tmp_path):
    info = engine.train_run(cfg, 0, tmp_path, model(), torch.device("cpu"))
    assert info["epochs_run"] == 4 and info["best_val_c_acc"] > 0.9
    assert (tmp_path / "best.pt").exists() and (tmp_path / "done.json").exists() and not (tmp_path / "last.pt").exists()
    log = pd.read_csv(tmp_path / "log.csv")
    assert list(log.epoch) == [1, 2, 3, 4] and log.jsd.iloc[0] > 0 and log.loss.iloc[-1] < log.loss.iloc[0]
    assert engine.train_run(cfg, 0, tmp_path, model(), torch.device("cpu")) == info      # finished run: no-op


def test_same_seed_same_result_and_different_seed_differs(cfg, tmp_path):
    a = engine.train_run(cfg, 0, tmp_path / "a", model(), torch.device("cpu"))
    b = engine.train_run(cfg, 0, tmp_path / "b", model(), torch.device("cpu"))
    sa = torch.load(tmp_path / "a" / "best.pt", weights_only=False)["model"]
    sb = torch.load(tmp_path / "b" / "best.pt", weights_only=False)["model"]
    assert a["best_val_c_acc"] == b["best_val_c_acc"] and all(torch.equal(sa[k], sb[k]) for k in sa)
    engine.train_run(cfg, 1, tmp_path / "c", model(), torch.device("cpu"))
    sc = torch.load(tmp_path / "c" / "best.pt", weights_only=False)["model"]
    assert any(not torch.equal(sa[k], sc[k]) for k in sa)


def test_interrupted_run_resumes_exactly(cfg, tmp_path, monkeypatch):
    engine.train_run(cfg, 0, tmp_path / "straight", model(), torch.device("cpu"))

    real_save, calls = engine._save, {"n": 0}

    def flaky_save(obj, path):
        real_save(obj, path)
        if path.name == "last.pt":
            calls["n"] += 1
            if calls["n"] == 2:                      # die right after the checkpoint of epoch 2
                raise KeyboardInterrupt

    monkeypatch.setattr(engine, "_save", flaky_save)
    with pytest.raises(KeyboardInterrupt):
        engine.train_run(cfg, 0, tmp_path / "resumed", model(), torch.device("cpu"))
    assert (tmp_path / "resumed" / "last.pt").exists() and not (tmp_path / "resumed" / "done.json").exists()
    monkeypatch.setattr(engine, "_save", real_save)
    info = engine.train_run(cfg, 0, tmp_path / "resumed", model(seed=123), torch.device("cpu"))   # weights come from the checkpoint
    assert info["epochs_run"] == 4

    a = pd.read_csv(tmp_path / "straight" / "log.csv").drop(columns=["epoch_s", "img_per_s"])
    b = pd.read_csv(tmp_path / "resumed" / "log.csv").drop(columns=["epoch_s", "img_per_s"])
    pd.testing.assert_frame_equal(a, b)
    sa = torch.load(tmp_path / "straight" / "best.pt", weights_only=False)["model"]
    sb = torch.load(tmp_path / "resumed" / "best.pt", weights_only=False)["model"]
    assert all(torch.equal(sa[k], sb[k]) for k in sa)


def test_scratch_dir_keeps_per_epoch_checkpoints_out_of_the_run_dir(cfg, tmp_path, monkeypatch):
    engine.train_run(cfg, 0, tmp_path / "straight", model(), torch.device("cpu"))
    run, scratch = tmp_path / "run", tmp_path / "scratch"
    real_save, calls = engine._save, {"n": 0}

    def flaky_save(obj, path):
        real_save(obj, path)
        calls["n"] += path.name == "last.pt"
        if calls["n"] == 2 and path.name == "last.pt":
            raise KeyboardInterrupt

    monkeypatch.setattr(engine, "_save", flaky_save)
    with pytest.raises(KeyboardInterrupt):
        engine.train_run(cfg, 0, run, model(), torch.device("cpu"), scratch_dir=scratch, fingerprint="a")
    assert (scratch / "last.pt").exists() and (scratch / "best.pt").exists()
    assert not (run / "last.pt").exists() and not (run / "best.pt").exists()      # nothing big in the run dir yet
    monkeypatch.setattr(engine, "_save", real_save)

    info = engine.train_run(cfg, 0, run, model(seed=9), torch.device("cpu"), scratch_dir=scratch, fingerprint="a")
    assert info["epochs_run"] == 4 and (run / "best.pt").exists() and (run / "done.json").exists()
    assert not (scratch / "last.pt").exists() and not (scratch / "best.pt").exists()
    sa = torch.load(tmp_path / "straight" / "best.pt", weights_only=False)["model"]
    sb = torch.load(run / "best.pt", weights_only=False)["model"]
    assert all(torch.equal(sa[k], sb[k]) for k in sa)                             # resumed from scratch, exactly


def test_resume_checkpoint_of_another_configuration_is_ignored(cfg, tmp_path, monkeypatch):
    real_save = engine._save

    def die_after_first_epoch(obj, path):
        real_save(obj, path)
        if path.name == "last.pt":
            raise KeyboardInterrupt

    monkeypatch.setattr(engine, "_save", die_after_first_epoch)
    with pytest.raises(KeyboardInterrupt):
        engine.train_run(cfg, 0, tmp_path / "old", model(), torch.device("cpu"), scratch_dir=tmp_path / "s", fingerprint="old")
    monkeypatch.setattr(engine, "_save", real_save)
    engine.train_run(cfg, 0, tmp_path / "new", model(), torch.device("cpu"), scratch_dir=tmp_path / "s", fingerprint="new")
    engine.train_run(cfg, 0, tmp_path / "ref", model(), torch.device("cpu"))
    a = pd.read_csv(tmp_path / "new" / "log.csv").drop(columns=["epoch_s", "img_per_s"])
    b = pd.read_csv(tmp_path / "ref" / "log.csv").drop(columns=["epoch_s", "img_per_s"])
    pd.testing.assert_frame_equal(a, b)                                           # started from epoch 1, not resumed


def test_early_stopping(cfg, tmp_path, monkeypatch):
    cfg["train"].update(epochs=12, patience=2)
    accs = iter([0.5, 0.6] + [0.4] * 40)             # (val, val-C) pairs: val-C = 0.6, then never better
    monkeypatch.setattr(engine, "accuracy", lambda *a, **k: next(accs))
    info = engine.train_run(cfg, 0, tmp_path, model(), torch.device("cpu"))
    assert info["early_stopped"] and info["best_epoch"] == 1 and info["epochs_run"] == 3


In [ ]:
%%writefile /content/RoadLens/tests/test_metrics.py
"""Metrics on hand-checkable inputs."""

import numpy as np
import pandas as pd
import pytest
import torch

from roadlens.eval import metrics as M
from roadlens.eval.tables import format_summary, pm


def test_confusion_and_macro_f1():
    y = torch.tensor([0, 0, 0, 1, 1, 2])
    pred = torch.tensor([0, 0, 1, 1, 2, 2])
    cm = M.confusion_matrix(pred, y, 4).numpy()
    assert cm.tolist() == [[2, 1, 0, 0], [0, 1, 1, 0], [0, 0, 1, 0], [0, 0, 0, 0]]
    # F1: class0 = 2*2/(3+2) = .8, class1 = 2*1/(2+2) = .5, class2 = 2*1/(1+2) = 2/3; class 3 absent -> ignored
    assert M.macro_f1(cm) == pytest.approx((0.8 + 0.5 + 2 / 3) / 3)


def test_macro_f1_counts_predictions_of_absent_classes_as_errors():
    cm = np.array([[5, 0, 5], [0, 10, 0], [0, 0, 0]])     # class 2 never occurs but is predicted 5 times
    assert M.macro_f1(cm) == pytest.approx((2 * 5 / (10 + 5) + 1.0) / 2)


def test_ece():
    conf = torch.full((100,), 0.8)
    correct = torch.zeros(100, dtype=torch.bool)
    correct[:60] = True
    bins = M.reliability_bins(conf, correct).numpy()
    assert bins[:, 0].sum() == 100 and bins[11, 0] == 100           # 0.8 falls in (11/15, 12/15]
    assert M.ece_from_bins(bins) == pytest.approx(0.2, abs=1e-6)
    perfect = M.reliability_bins(torch.ones(10), torch.ones(10, dtype=torch.bool)).numpy()
    assert M.ece_from_bins(perfect) == pytest.approx(0.0, abs=1e-6)


def test_summarize_logits():
    logits = torch.tensor([[4.0, 0, 0], [0, 4.0, 0], [0, 4.0, 0], [0, 0, 4.0]])
    y = torch.tensor([0, 1, 2, 2])
    stats, cm, bins = M.summarize_logits(logits, y, 3, shared=[0, 2])
    assert stats["acc"] == pytest.approx(0.75) and stats["n"] == 4
    assert cm.sum() == 4 and bins[:, 0].sum() == 4
    # restricted to classes {0, 2}: sample 1 (true 1) can no longer be right, sample 2 (true 2) ties 0 vs 2 -> class 0
    assert stats["acc_shared"] == pytest.approx(0.5)


def _table(clean, accs):
    rows = [{"corruption": "clean", "severity": 0, "acc": clean, "macro_f1": clean, "ece": 0.0}]
    for name, per_sev in accs.items():
        rows += [{"corruption": name, "severity": s + 1, "acc": a, "macro_f1": a, "ece": 0.1} for s, a in enumerate(per_sev)]
    return pd.DataFrame(rows)


def test_mce_and_relative_mce():
    ref = _table(0.9, {"noise": [0.8, 0.6], "blur": [0.7, 0.5]})          # errors: noise .2/.4, blur .3/.5
    assert M.corruption_summary(ref, ref)["mce"] == pytest.approx(100.0)
    assert M.corruption_summary(ref, ref)["rel_mce"] == pytest.approx(100.0)
    model = _table(0.95, {"noise": [0.9, 0.8], "blur": [0.85, 0.75]})     # errors: noise .1/.2, blur .15/.25
    s = M.corruption_summary(model, ref)
    assert s["mce"] == pytest.approx(100 * 0.5)                           # exactly half the errors everywhere
    # relative: noise (0.15 - 0.05) / (0.3 - 0.1) = 0.5 ; blur (0.2 - 0.05) / (0.4 - 0.1) = 0.5
    assert s["rel_mce"] == pytest.approx(50.0)
    assert s["clean_acc"] == pytest.approx(95.0) and s["corr_acc"] == pytest.approx(100 * np.mean([0.9, 0.8, 0.85, 0.75]))


def test_mean_std_and_formatting():
    df = pd.DataFrame({"run": ["a", "a", "a", "b"], "seed": [0, 1, 2, 0], "acc": [1.0, 2.0, 3.0, 5.0]})
    out = M.mean_std(df, "run").set_index("run")
    assert out.loc["a", "acc_mean"] == pytest.approx(2.0) and out.loc["a", "acc_std"] == pytest.approx(1.0)
    assert out.loc["a", "n_seeds"] == 3 and np.isnan(out.loc["b", "acc_std"])
    assert pm(2.0, 1.0, 1) == "2.0 +- 1.0" and pm(5.0, float("nan"), 1) == "5.0"
    plain, tex = format_summary(out.reset_index().rename(columns={"run": "model"}), {"acc": ("Acc", 1, "max")})
    assert plain["Acc"].tolist() == ["2.0 +- 1.0", "5.0"]
    assert tex[1][1] == r"\textbf{5.0}" and "textbf" not in tex[0][1]


In [ ]:
%%writefile /content/RoadLens/tests/test_models_tta.py
"""Models, fusion, Grad-CAM and test-time adaptation invariants (random weights, CPU)."""

import pytest
import torch
import torch.nn.functional as F

from roadlens.eval.gradcam import grad_cam
from roadlens.models import Fusion, SmallCNN, build_model
from roadlens.tta import BNAdapt, NoAdapt, Tent, batchnorm_layers, build_tta, entropy


def images(n=8, seed=0):
    return torch.rand(n, 3, 64, 64, generator=torch.Generator().manual_seed(seed))


def small(seed):
    torch.manual_seed(seed)
    return SmallCNN(43, width=8).eval()


def test_smallcnn_and_timm_shapes():
    x = images(2)
    feat, logits = small(0).features_logits(x)
    assert feat.shape == (2, 64) and logits.shape == (2, 43)
    for name in ("resnet18", "deit_small_patch16_224"):
        m = build_model({"name": name, "pretrained": False, "input_size": 64}, 43).eval()
        feat, logits = m.features_logits(x)
        assert feat.shape == (2, m.feature_dim) and logits.shape == (2, 43)
        assert len(m.cam_targets()) == 1


def test_fusion_modes():
    x = images(4)
    a, b = small(1), small(2)
    pa, pb = F.softmax(a(x), 1), F.softmax(b(x), 1)
    avg = Fusion([a, b], 43, mode="average").eval()
    assert torch.allclose(avg(x).exp(), (pa + pb) / 2, atol=1e-5)
    conf = Fusion([a, b], 43, mode="confidence").eval()
    w = torch.stack([pa.amax(1), pb.amax(1)], 1)
    w = w / w.sum(1, keepdim=True)
    assert torch.allclose(conf(x).exp(), w[:, :1] * pa + w[:, 1:] * pb, atol=1e-5)
    gate = Fusion([a, b], 43, mode="gate").eval()
    out = gate.forward_detailed(x)
    assert torch.allclose(out["weights"], torch.full((4, 2), 0.5))          # zero-initialised gate = average
    assert torch.allclose(out["logits"].exp().sum(1), torch.ones(4), atol=1e-5)
    assert torch.allclose(out["logits"].exp(), (pa + pb) / 2, atol=1e-5)


def test_fusion_freezes_branches_and_checkpoints_only_the_gate():
    f = Fusion([small(1), small(2)], 43, mode="gate")
    f.train()
    assert all(not m.training for m in f.branches.modules())
    assert f.gate.training
    trainable = [n for n, p in f.named_parameters() if p.requires_grad]
    assert trainable and all(not n.startswith("branches.") for n in trainable)
    state = f.checkpoint_state()
    assert state and all(not k.startswith("branches.") for k in state)
    g = Fusion([small(1), small(2)], 43, mode="gate")
    torch.nn.init.normal_(f.gate[-1].weight)
    g.load_checkpoint_state(f.checkpoint_state())
    x = images(3)
    assert torch.allclose(f.eval()(x), g.eval()(x), atol=1e-6)
    with pytest.raises(RuntimeError):
        g.load_checkpoint_state({"gate.0.weight": state["gate.0.weight"]})


def test_gradcam_shapes_and_range():
    x = images(3)
    cnn = small(0)
    cams, logits = grad_cam(cnn, cnn.cam_targets(), x)
    assert cams[0].shape == (3, 64, 64) and float(cams[0].min()) >= 0 and float(cams[0].max()) <= 1 + 1e-6
    vit = build_model({"name": "deit_small_patch16_224", "pretrained": False, "input_size": 64}, 43).eval()
    cams, _ = grad_cam(vit, vit.cam_targets(), x)
    assert cams[0].shape == (3, 64, 64) and torch.isfinite(cams[0]).all()
    fusion = Fusion([cnn, vit], 43, mode="gate").eval()                    # frozen branches still give gradients
    cams, _ = grad_cam(fusion, fusion.cam_targets(), x)
    assert len(cams) == 2 and all(float(c.abs().sum()) > 0 for c in cams)


def test_bn_adapt_uses_batch_statistics_and_leaves_source_untouched():
    model = small(0)
    before = {k: v.clone() for k, v in model.state_dict().items()}
    x = images(16) * 0.3 + 0.6                                             # shifted input distribution
    src = NoAdapt(model, amp=False).predict(x)
    assert torch.allclose(src, model(x))
    adapted = BNAdapt(model, amp=False)
    out = adapted.predict(x)
    assert not torch.allclose(out, src, atol=1e-3)
    assert torch.allclose(adapted.predict(x), out)                         # stateless across batches
    assert all(torch.equal(v, model.state_dict()[k]) for k, v in before.items() if "num_batches" not in k)
    assert not any(m.training for m in model.modules())
    assert all(torch.equal(v, adapted.model.state_dict()[k]) for k, v in before.items() if "running" in k)


def test_tent_updates_only_batchnorm_affine_and_resets():
    model = small(0)
    tent = Tent(model, lr=1e-2, amp=False)
    names = {n for n, p in tent.model.named_parameters() if p.requires_grad}
    bn_names = {f"{n}.{k}" for n, m in tent.model.named_modules() if m in batchnorm_layers(tent.model) for k in ("weight", "bias")}
    assert names == bn_names and len(names) == 2 * len(batchnorm_layers(model))
    x = images(16)
    first = tent.predict(x)
    assert torch.allclose(first, BNAdapt(model, amp=False).predict(x), atol=1e-5)   # logits from before the step
    second = tent.predict(x)
    assert not torch.allclose(first, second, atol=1e-6)                    # the step changed the model
    src, new = model.state_dict(), tent.model.state_dict()
    changed = {k for k in src if not torch.equal(src[k], new[k]) and "num_batches" not in k}
    assert changed and changed <= bn_names
    ent_before = float(F.softmax(first, 1).mul(F.log_softmax(first, 1)).sum(1).mean().neg())
    for _ in range(10):
        last = tent.predict(x)
    ent_after = float(F.softmax(last, 1).mul(F.log_softmax(last, 1)).sum(1).mean().neg())
    assert ent_after < ent_before                                          # entropy is being minimised
    tent.reset()
    assert torch.allclose(tent.predict(x), first, atol=1e-5)


def test_tent_on_fusion_adapts_only_the_cnn_branch_and_works_at_batch_size_one():
    cnn = small(0)
    vit = build_model({"name": "deit_small_patch16_224", "pretrained": False, "input_size": 64}, 43).eval()
    fusion = Fusion([cnn, vit], 43, mode="gate").eval()
    tent = build_tta("tent", fusion, amp=False, lr=1e-2)
    trainable = [n for n, p in tent.model.named_parameters() if p.requires_grad]
    assert trainable and all(n.startswith("branches.0.") for n in trainable)
    out = tent.predict(images(1))
    assert out.shape == (1, 43) and torch.isfinite(out).all()

    # default 'branch' objective: the CNN branch's own entropy goes down even if the gate ignores that branch
    with torch.no_grad():
        fusion.gate[-1].bias.copy_(torch.tensor([-30.0, 30.0]))            # all weight on the ViT branch
    x = images(8)
    tent = build_tta("tent", fusion, amp=False, lr=1e-2)

    def cnn_entropy():
        with torch.no_grad():
            return float(entropy(tent.model.forward_detailed(x)["branch_logits"][0]).mean())

    before = cnn_entropy()
    for _ in range(10):
        tent.predict(x)
    assert cnn_entropy() < before
    assert build_tta("tent", fusion, amp=False, objective="fused").branch_ids is None
    with pytest.raises(ValueError):
        build_tta("bn", vit)
    with pytest.raises(ValueError):
        build_tta("tent", vit)


In [ ]:
%%writefile /content/RoadLens/tests/test_v2.py
"""The v2 components: instance input normalisation, view experts, the stacked gate, the BatchNorm
statistics prior and temperature scaling (random weights / synthetic data, CPU)."""

import numpy as np
import pytest
import torch
import torch.nn.functional as F

import roadlens.engine as engine
from roadlens.data.gtsrb import halve_by_track
from roadlens.eval.calibration import fit_temperature, nll_at
from roadlens.models import Fusion, SmallCNN, build_model
from roadlens.models.views import VIEWS, expert_stats, gauss, image_stats, median3
from roadlens.runs import eval_tag
from roadlens.tta import BNAdapt, NoAdapt, PriorBatchNorm, Tent, batchnorm_layers, build_tta


def images(n=8, seed=0, size=32):
    return torch.rand(n, 3, size, size, generator=torch.Generator().manual_seed(seed))


def small(seed, classes=43, **kw):
    torch.manual_seed(seed)
    return SmallCNN(classes, width=8, **kw).eval()


# ---------------------------------------------------------------------------- instance normalisation
def gap(model, a, b) -> float:
    """Largest change of a logit between two inputs, relative to the largest logit."""
    with torch.no_grad():
        return float((model(a) - model(b)).abs().max() / model(a).abs().max())


def test_instance_norm_removes_per_channel_gain_and_offset():
    x = images(6) * 0.5 + 0.2
    gain = torch.tensor([0.05, 0.4, 1.7]).view(1, 3, 1, 1)
    offset = torch.tensor([0.3, -0.1, 0.05]).view(1, 3, 1, 1)
    model, plain = small(0, input_norm="instance"), small(0)
    assert gap(model, x, x * gain + offset) < 0.03
    mean = x.mean(dim=(2, 3), keepdim=True)
    for c in (0.4, 0.05):                                   # the contrast corruption, severities 1 and 5
        faint = (x - mean) * c + mean
        assert gap(model, x, faint) < 0.03
        assert gap(plain, x, faint) > 5 * gap(model, x, faint)
    assert torch.isfinite(model(torch.full((2, 3, 32, 32), 0.5))).all()         # a constant image is not a division by zero


def test_instance_norm_reaches_timm_models_and_rejects_unknown_names():
    m = build_model({"name": "resnet18", "pretrained": False, "input_size": 64, "input_norm": "instance"}, 43).eval()
    assert m.pre.norm == "instance"
    x = images(2, size=64)
    assert gap(m, x, 0.3 * x + 0.1) < 0.03
    assert build_model({"name": "resnet18", "pretrained": False, "input_size": 64}, 43).pre.norm == "dataset"
    with pytest.raises(ValueError):
        build_model({"name": "smallcnn", "input_norm": "batch"}, 43)


# ---------------------------------------------------------------------------- views and image statistics
def test_views_denoise_and_keep_flat_images():
    flat = torch.full((2, 3, 16, 16), 0.4)
    for name, fn in VIEWS.items():
        assert torch.allclose(fn(flat), flat, atol=1e-6), name
    spiky = flat.clone()
    spiky[0, :, 5, 7], spiky[1, 0, 0, 0] = 1.0, 0.0                              # isolated salt and pepper
    assert torch.equal(median3(spiky), flat)
    clean = images(4) * 0.2 + 0.4
    noisy = (clean + 0.15 * torch.randn(clean.shape, generator=torch.Generator().manual_seed(1))).clamp(0, 1)
    for fn in (median3, gauss):
        assert (fn(noisy) - fn(clean)).std() < 0.6 * (noisy - clean).std()        # the noise shrinks ...
        assert fn(noisy).shape == noisy.shape
    step = torch.zeros(1, 3, 16, 16)
    step[..., 8:] = 1.0
    assert torch.equal(median3(step), step)                                       # ... and a median keeps an edge


def test_image_stats_react_to_noise_blur_and_contrast():
    smooth = gauss(images(8, size=32), sigma=2.0)
    noisy = (smooth + 0.1 * torch.randn(smooth.shape, generator=torch.Generator().manual_seed(2))).clamp(0, 1)
    mean = smooth.mean(dim=(1, 2, 3), keepdim=True)
    faint = (smooth - mean) * 0.2 + mean
    s, n, f = image_stats(smooth), image_stats(noisy), image_stats(faint)
    assert s.shape == (8, 5) and torch.isfinite(s).all()
    assert (n[:, 2] > s[:, 2]).all() and (n[:, 3] > s[:, 3]).all()               # noise raises the high-frequency terms
    assert (f[:, 1] < 0.5 * s[:, 1]).all()                                       # low contrast lowers the contrast term
    assert (image_stats(gauss(noisy))[:, 2] < n[:, 2]).all()                     # blur lowers them again


def test_expert_stats():
    p = torch.tensor([[0.7, 0.2, 0.1], [1 / 3, 1 / 3, 1 / 3]])
    q = torch.tensor([[0.1, 0.8, 0.1], [0.9, 0.05, 0.05]])
    s = expert_stats([p, q])
    assert s.shape == (2, 7)
    assert torch.allclose(s[0, :3], torch.tensor([0.7, 0.5, float(-(p[0] * p[0].log()).sum() / np.log(3))]), atol=1e-6)
    assert torch.allclose(s[1, 2], torch.tensor(1.0), atol=1e-6)                 # uniform = maximal entropy
    assert torch.allclose(s[:, 6], torch.tensor([0.5, 1.0]))                     # row 0 disagrees, row 1 agrees
    assert float(s.min()) >= 0 and float(s.max()) <= 1 + 1e-6


# ---------------------------------------------------------------------------- view-expert fusion
def test_fusion_averages_over_branches_and_views():
    x = images(4)
    a, b = small(1), small(2)
    views = ("raw", "median3", "gauss")
    avg = Fusion([a, b], 43, mode="average", views=views).eval()
    expected = torch.stack([F.softmax(m(VIEWS[v](x)), 1) for v in views for m in (a, b)]).mean(0)
    out = avg.forward_detailed(x)
    assert avg.n_experts == 6 and out["weights"].shape == (4, 6) and len(out["branch_logits"]) == 6
    assert torch.allclose(out["logits"].exp(), expected, atol=1e-5)
    assert out["features"].shape == (4, avg.feature_dim)                         # features of the first view only
    assert torch.allclose(out["branch_logits"][0], a(x)) and torch.allclose(out["branch_logits"][3], b(median3(x)))
    assert torch.allclose(Fusion([a, b], 43, mode="average").eval()(x).exp(),
                          (F.softmax(a(x), 1) + F.softmax(b(x), 1)) / 2, atol=1e-5)   # default: one view, as before
    with pytest.raises(ValueError):
        Fusion([a, b], 43, views=("raw", "sharpen"))
    with pytest.raises(ValueError):
        Fusion([a, b], 43, gate_inputs=("logits",))


def test_stacked_gate_starts_as_the_average_and_trains_only_the_gate():
    x = images(6)
    a, b = small(1), small(2)
    kw = dict(views=("raw", "median3", "gauss"))
    gate = Fusion([a, b], 43, mode="gate", gate_inputs=("stats", "image"), hidden=8, dropout=0.0, **kw)
    avg = Fusion([a, b], 43, mode="average", **kw).eval()
    assert gate.gate[0].in_features == 3 * 6 + 1 + 5
    assert torch.allclose(gate.eval()(x), avg(x), atol=1e-5)
    assert set(gate.checkpoint_state()) == {"gate.0.weight", "gate.0.bias", "gate.3.weight", "gate.3.bias"}

    gate.train()
    assert all(not m.training for m in gate.branches.modules())
    opt = torch.optim.SGD([p for p in gate.parameters() if p.requires_grad], lr=1.0)
    before = [p.detach().clone() for p in gate.branches.parameters()]
    for _ in range(2):                                  # the first step only moves the zero-initialised output layer
        loss = F.cross_entropy(gate(x), torch.arange(6))
        opt.zero_grad()
        loss.backward()
        opt.step()
    assert all(torch.equal(p, q) for p, q in zip(gate.branches.parameters(), before))
    w = gate.eval().forward_detailed(x)["weights"]
    assert torch.allclose(w.sum(1), torch.ones(6), atol=1e-5) and not torch.allclose(w, torch.full_like(w, 1 / 6))
    with pytest.raises(ValueError):
        gate.branch_weights([None] * 6, [torch.zeros(6, 43)] * 6)                # image statistics need the input


def test_tent_on_a_view_fusion_adapts_the_cnn_branch():
    cnn = small(0)
    vit = build_model({"name": "deit_small_patch16_224", "pretrained": False, "input_size": 32}, 43).eval()
    fusion = Fusion([cnn, vit], 43, mode="average", views=("raw", "gauss")).eval()
    tent = build_tta("tent", fusion, amp=False, lr=1e-2, prior=8)
    assert tent.branch_ids == [0]
    out = tent.predict(images(1))
    assert out.shape == (1, 43) and torch.isfinite(out).all()


# ---------------------------------------------------------------------------- BatchNorm statistics prior
def test_prior_batchnorm_interpolates_between_source_and_batch_statistics():
    model = small(0)
    with torch.no_grad():                               # give the source statistics some structure
        for m in batchnorm_layers(model):
            m.running_mean.normal_(0, 0.2)
            m.running_var.uniform_(0.5, 1.5)
    before = {k: v.clone() for k, v in model.state_dict().items()}
    x = images(16) * 0.3 + 0.6
    src, plain = NoAdapt(model, amp=False).predict(x), BNAdapt(model, amp=False).predict(x)
    assert torch.allclose(BNAdapt(model, amp=False, prior=1e9).predict(x), src, atol=1e-4)      # all prior = source model
    assert torch.allclose(BNAdapt(model, amp=False, prior=1e-9).predict(x), plain, atol=1e-4)   # no prior = plain test-time BN
    mid = BNAdapt(model, amp=False, prior=16).predict(x)
    assert (mid - src).abs().mean() < (plain - src).abs().mean() and not torch.allclose(mid, src, atol=1e-4)
    assert all(torch.equal(v, model.state_dict()[k]) for k, v in before.items())                # caller's model untouched
    assert not any(isinstance(m, PriorBatchNorm) for m in model.modules())

    one = x[:1]                                          # a single image: its own statistics are useless
    src1 = NoAdapt(model, amp=False).predict(one)
    with_prior = BNAdapt(model, amp=False, prior=16).predict(one)
    assert (with_prior - src1).abs().mean() < 0.5 * (BNAdapt(model, amp=False).predict(one) - src1).abs().mean()


def test_tent_with_prior_updates_the_wrapped_layers_and_resets():
    model = small(0)
    tent = Tent(model, lr=1e-2, amp=False, prior=16)
    assert sum(isinstance(m, PriorBatchNorm) for m in tent.model.modules()) == len(batchnorm_layers(model))
    assert len(tent.params) == 2 * len(batchnorm_layers(model))
    x = images(4)
    first = tent.predict(x)
    assert torch.allclose(first, BNAdapt(model, amp=False, prior=16).predict(x), atol=1e-5)
    assert not torch.allclose(tent.predict(x), first, atol=1e-6)
    tent.reset()
    assert torch.allclose(tent.predict(x), first, atol=1e-5)


def test_eval_tags_keep_old_names_and_separate_the_new_options():
    assert eval_tag("gtsrb_c") == "gtsrb_c"
    assert eval_tag("gtsrb_c", {"method": "bn", "batch_size": 16, "lr": 1e-3}, [5], 500) == "gtsrb_c__bn_bs16__sev5__n500"
    assert eval_tag("gtsrb_c", {"method": "bn", "batch_size": 16, "prior": 0.0}, [5], 500) == "gtsrb_c__bn_bs16__sev5__n500"
    assert eval_tag("gtsrb_c", {"method": "bn", "batch_size": 1, "prior": 16}, [5], 100) == "gtsrb_c__bn_bs1_p16__sev5__n100"
    assert eval_tag("gtsrb_c", {"method": "tent", "batch_size": 64, "lr": 1e-3}) == "gtsrb_c__tent_bs64_lr0.001"
    assert eval_tag("gtsrb_c", calibrate=True) == "gtsrb_c__cal"


# ---------------------------------------------------------------------------- temperature scaling
def test_fit_temperature_recovers_a_known_scale_and_never_raises_the_nll():
    g = torch.Generator().manual_seed(0)
    z = 2.0 * torch.randn(20000, 10, generator=g)
    y = torch.multinomial(F.softmax(z, 1), 1, generator=g).squeeze(1)            # z is calibrated at T = 1
    for scale in (0.25, 1.0, 4.0):
        t = fit_temperature(z * scale, y)
        assert abs(t / scale - 1) < 0.05
        assert nll_at(z * scale, y, t) <= nll_at(z * scale, y, 1.0) + 1e-9
        assert torch.equal((z * scale / t).argmax(1), z.argmax(1))               # accuracy cannot change
    with pytest.raises(ValueError):
        fit_temperature(torch.zeros(0, 10), torch.zeros(0, dtype=torch.long))


# ---------------------------------------------------------------------------- stacking: split and training
def test_halve_by_track_shares_no_track_and_is_balanced():
    rng = np.random.RandomState(0)
    y = np.repeat(np.arange(6), [30, 60, 90, 30, 150, 240])                      # 1, 2, 3, 1, 5, 8 tracks of 30 frames
    tracks = y * 10000 + np.concatenate([np.repeat(np.arange(n // 30), 30) for n in np.bincount(y)])
    order = rng.permutation(len(y))
    y, tracks = y[order], tracks[order]
    a, b = halve_by_track(y, tracks, seed=0)
    assert len(np.intersect1d(a, b)) == 0 and len(a) + len(b) == len(y)
    assert not set(tracks[a]) & set(tracks[b])
    assert abs(len(set(tracks[a])) - len(set(tracks[b]))) <= 1                   # 20 tracks -> 10 + 10
    a2, b2 = halve_by_track(y, tracks, seed=0)
    assert np.array_equal(a, a2) and np.array_equal(b, b2)


def synthetic(n, seed):
    """Three classes that differ in which colour channel is bright."""
    rng = np.random.RandomState(seed)
    y = rng.randint(0, 3, n)
    x = rng.randint(0, 80, (n, 16, 16, 3))
    x[np.arange(n), :, :, y] += 120
    return x.astype(np.uint8), y.astype(np.int64)


@pytest.fixture
def stack_cfg(monkeypatch):
    data = (*synthetic(96, 0), *synthetic(64, 1), *synthetic(64, 2))
    monkeypatch.setattr(engine, "load_stack_data", lambda cfg: data)
    monkeypatch.setattr(engine, "load_split", lambda *a: pytest.fail("a stacked run must not load the training split"))
    return {
        "name": "toy_gate", "data": {"num_classes": 3},
        "train": {"fit_split": "val", "epochs": 3, "batch_size": 32, "lr": 2e-3, "weight_decay": 0.0, "warmup_epochs": 1,
                  "label_smoothing": 0.0, "patience": 0, "amp": False, "eval_batch_size": 64,
                  "ce_all_views": True, "eval_at_init": True},
        "augment": {"mode": "jsd", "ops": ["augmix", "prime"], "width": 2, "depth": [1, 2], "severity": 6, "jsd_weight": 0.0},
    }


def toy_gate():
    return Fusion([small(1, 3), small(2, 3)], 3, mode="gate", gate_inputs=("stats", "image"), hidden=8, dropout=0.0,
                  views=("raw", "gauss"))


def test_stacked_gate_trains_on_held_out_data_and_logs_every_epoch(stack_cfg, tmp_path):
    info = engine.train_run(stack_cfg, 0, tmp_path, toy_gate(), torch.device("cpu"))
    assert info["epochs_run"] == 3 and 0 <= info["best_epoch"] <= 3
    state = torch.load(tmp_path / "best.pt", weights_only=False)["model"]
    assert set(state) == {"gate.0.weight", "gate.0.bias", "gate.3.weight", "gate.3.bias"}
    again = engine.train_run(stack_cfg, 0, tmp_path / "again", toy_gate(), torch.device("cpu"))
    assert again["best_val_c_acc"] == info["best_val_c_acc"] and again["best_epoch"] == info["best_epoch"]


def test_selection_keeps_the_untrained_average_when_training_does_not_help(stack_cfg, tmp_path, monkeypatch):
    accs = iter([0.9, 0.8] + [0.5] * 20)                # epoch 0: (val, val-C) = (0.9, 0.8); every later epoch is worse
    monkeypatch.setattr(engine, "accuracy", lambda *a, **k: next(accs))
    gate = toy_gate()
    info = engine.train_run(stack_cfg, 0, tmp_path, gate, torch.device("cpu"))
    assert info["best_epoch"] == 0 and info["best_val_c_acc"] == 0.8 and info["val_acc_at_best"] == 0.9
    state = torch.load(tmp_path / "best.pt", weights_only=False)["model"]
    assert float(state["gate.3.weight"].abs().sum()) == 0 and float(state["gate.3.bias"].abs().sum()) == 0   # still the average

    stack_cfg["train"]["eval_at_init"] = False          # without it the first epoch always wins by default
    accs = iter([0.5] * 20)
    info = engine.train_run(stack_cfg, 0, tmp_path / "plain", toy_gate(), torch.device("cpu"))
    assert info["best_epoch"] == 1


def test_ce_all_views_changes_the_loss_only_when_asked(tmp_path):
    torch.manual_seed(0)
    model = small(0, 3)
    views = [images(4, s, size=16) for s in (0, 1, 2)]
    y = torch.tensor([0, 1, 2, 0])
    _, ce_clean, _, _ = engine.training_loss(model, views, y, True, 0.0, 0.0, False)
    _, ce_all, _, _ = engine.training_loss(model, views, y, True, 0.0, 0.0, False, ce_all_views=True)
    logits = [model(v) for v in views]
    assert torch.allclose(ce_clean, F.cross_entropy(logits[0], y), atol=1e-6)
    assert torch.allclose(ce_all, sum(F.cross_entropy(z, y) for z in logits) / 3, atol=1e-6)
    with pytest.raises(ValueError):
        engine.train_run({"name": "x", "data": {"num_classes": 3}, "train": {"fit_split": "test"}, "augment": {}},
                         0, tmp_path, small(0, 3).train(), torch.device("cpu"))


In [ ]:
%%writefile /content/RoadLens/tests/test_v3.py
"""The v3 components: local-global contrast normalisation, amplitude-phase recombination, EMA weights,
distillation, SimCLR and the compatibility of older runs (random weights / synthetic data, CPU)."""

import json

import numpy as np
import pandas as pd
import pytest
import torch
import torch.nn.functional as F

import roadlens.engine as engine
from roadlens.augment import TrainAugment, resolve_ops
from roadlens.augment.ops import amplitude_mix
from roadlens.models import SmallCNN, build_model
from roadlens.models.base import Preprocess
from roadlens.runs import _classifier_prefixes, fingerprint, load_run_config
from roadlens.ssl import knn_accuracy, nt_xent


def images(n=8, seed=0, size=32):
    return torch.rand(n, 3, size, size, generator=torch.Generator().manual_seed(seed))


def small(seed, classes=3, **kw):
    torch.manual_seed(seed)
    return SmallCNN(classes, width=8, **kw)


# ---------------------------------------------------------------------------- local-global normalisation
def test_local_norm_is_invariant_to_gain_and_offset_and_safe_on_flat_images():
    x = images(4) * 0.6 + 0.2
    gain = torch.tensor([0.1, 0.5, 2.0]).view(1, 3, 1, 1)
    offset = torch.tensor([0.3, -0.2, 0.05]).view(1, 3, 1, 1)
    a, b = Preprocess.local_norm(x), Preprocess.local_norm(x * gain + offset)
    assert torch.allclose(a, b, atol=1e-3)
    flat = Preprocess.local_norm(torch.full((2, 3, 32, 32), 0.5))
    assert torch.isfinite(flat).all() and flat.abs().max() < 1e-3


def test_local_norm_removes_a_smooth_additive_veil_better_than_instance_norm():
    x = images(4, size=64) * 0.6 + 0.2
    yy = torch.linspace(0, 1, 64).view(1, 1, 64, 1)
    veil = 0.6 * yy.expand(4, 3, 64, 64)                      # haze that grows from top to bottom
    foggy = 0.5 * x + veil
    inst = lambda z: (z - z.mean((2, 3), keepdim=True)) / z.std((2, 3), keepdim=True, correction=0)  # noqa: E731
    err_local = (Preprocess.local_norm(foggy) - Preprocess.local_norm(x)).abs().mean()
    err_inst = (inst(foggy) - inst(x)).abs().mean()
    assert err_local < 0.5 * err_inst


def test_local_stem_is_a_learnable_blend_and_reaches_timm_models():
    pre = Preprocess(None, norm="local")
    assert torch.allclose(torch.sigmoid(pre.mix), torch.full_like(pre.mix, 0.5))
    out = pre(images(2))
    out.sum().backward()
    assert pre.mix.grad is not None and torch.isfinite(pre.mix.grad).all()
    m = build_model({"name": "resnet18", "pretrained": False, "input_size": 64, "input_norm": "local"}, 43).eval()
    assert m.pre.norm == "local" and "pre.mix" in m.state_dict()
    with pytest.raises(ValueError):
        Preprocess(None, norm="global")


# ---------------------------------------------------------------------------- amplitude-phase recombination
def test_amplitude_mix_keeps_shape_range_and_is_identity_at_level_zero():
    x = images(6)
    same = amplitude_mix(x, torch.zeros(6))
    assert torch.allclose(same, x, atol=1e-4)
    out = amplitude_mix(x, torch.full((6,), 0.3))
    assert out.shape == x.shape and out.min() >= 0 and out.max() <= 1
    assert (out - x).abs().mean() > 1e-3
    assert torch.equal(amplitude_mix(x[:1], torch.full((1,), 0.3)), x[:1])          # nothing to mix with


def test_amplitude_mix_keeps_the_fourier_phase():
    x = images(4) * 0.2 + 0.4                                 # mid-range, so the clamp to [0, 1] stays inactive
    out = amplitude_mix(x, torch.full((4,), 0.2))
    fx, fo = torch.fft.fft2(x), torch.fft.fft2(out)
    strong = (fx.abs() > 1e-2) & (fo.abs() > 1e-2)
    diff = torch.angle(fx * fo.conj())[strong].abs()
    assert diff.median() < 1e-3


def test_apr_op_set_runs_inside_augmix():
    assert resolve_ops(["apr"]) == ["amplitude_mix"]
    aug = TrainAugment({"mode": "jsd", "ops": ["augmix", "prime", "apr"], "width": 2, "depth": [1, 2], "severity": 3})
    views = aug(images(5))
    assert len(views) == 3 and all(v.shape == (5, 3, 32, 32) for v in views)
    assert all(v.min() >= 0 and v.max() <= 1 for v in views)


# ---------------------------------------------------------------------------- EMA and distillation
def test_ema_averages_weights_with_warmup():
    a, b = small(0), small(1)
    ema = engine.ModelEMA(a, decay=0.9)
    ema.update(b)                                             # step 1: d = min(0.9, 2 / 11)
    d = 2 / 11
    k = "fc.weight"
    assert torch.allclose(ema.module.state_dict()[k], d * a.state_dict()[k] + (1 - d) * b.state_dict()[k], atol=1e-6)
    assert not any(p.requires_grad for p in ema.module.parameters())


def test_kd_loss_is_zero_for_identical_logits_and_grows_with_disagreement():
    z = torch.randn(8, 5)
    assert engine.kd_loss(z, z, 2.0).abs() < 1e-6
    assert engine.kd_loss(z, z.flip(1), 2.0) > 0.1


def test_training_loss_adds_the_distillation_term():
    torch.manual_seed(0)
    student, teacher = small(0).eval(), small(1).eval()          # eval: no dropout, so the two calls match
    views = [images(4, s, size=16) for s in (0, 1, 2)]
    y = torch.tensor([0, 1, 2, 0])
    stats = {}
    loss, ce, jsd, _ = engine.training_loss(student, views, y, True, 0.0, 1.0, False, teacher=teacher,
                                            kd_weight=0.5, kd_temperature=2.0, stats=stats)
    assert torch.allclose(loss, ce + jsd + 0.5 * stats["kd"], atol=1e-5) and stats["kd"] > 0
    plain, _, _, _ = engine.training_loss(student, views, y, True, 0.0, 1.0, False)
    assert torch.allclose(plain, ce + jsd, atol=1e-5)


def synthetic(n, seed):
    rng = np.random.RandomState(seed)
    y = rng.randint(0, 3, n)
    x = rng.randint(0, 80, (n, 16, 16, 3))
    x[np.arange(n), :, :, y] += 120
    return x.astype(np.uint8), y.astype(np.int64)


@pytest.fixture
def cfg(monkeypatch):
    data = {"train": synthetic(256, 0), "val": synthetic(64, 1)}
    monkeypatch.setattr(engine, "load_split", lambda cfg, split: data[split])
    monkeypatch.setattr(engine, "load_val_c_split", lambda cfg: (*synthetic(64, 2), None, None))
    return {"name": "toy", "data": {"num_classes": 3},
            "train": {"epochs": 3, "batch_size": 32, "lr": 3e-3, "weight_decay": 0.01, "warmup_epochs": 1,
                      "label_smoothing": 0.1, "grad_clip": 1.0, "patience": 0, "amp": False, "eval_batch_size": 64},
            "augment": {"mode": "jsd", "ops": ["augmix", "prime", "apr"], "width": 2, "depth": [1, 2], "severity": 3,
                        "jsd_weight": 12.0}}


def test_ema_and_distillation_train_and_resume(cfg, tmp_path):
    cfg["train"].update(ema=0.9, distill={"teacher": "t", "weight": 1.0, "temperature": 2.0},
                        cudnn_benchmark=True, fused_optimizer=True, channels_last=True)   # GPU-only flags are ignored on CPU
    plain_cfg = cfg | {"train": {**cfg["train"], "distill": {}}}
    teacher = small(5)
    info = engine.train_run(plain_cfg, 0, tmp_path / "teach", teacher, torch.device("cpu"))     # EMA, no teacher
    assert info["best_val_c_acc"] > 0.8
    info = engine.train_run(cfg, 0, tmp_path / "student", small(0), torch.device("cpu"), teacher=teacher)
    log = pd.read_csv(tmp_path / "student" / "log.csv")
    assert info["best_val_c_acc"] > 0.8 and (log.kd > 0).all()
    with pytest.raises(ValueError):
        engine.train_run(cfg, 0, tmp_path / "no_teacher", small(0), torch.device("cpu"))


# ---------------------------------------------------------------------------- self-supervised learning
def test_nt_xent_prefers_matching_pairs():
    torch.manual_seed(0)
    z = torch.randn(16, 8)
    matched = nt_xent(z, z + 0.01 * torch.randn_like(z))
    shuffled = nt_xent(z, z[torch.randperm(16)])
    assert matched < shuffled


def test_knn_accuracy_on_separable_features():
    centers = F.normalize(torch.eye(4), dim=1)
    y = torch.arange(4).repeat(10)
    bank = F.normalize(centers[y] + 0.05 * torch.randn(40, 4), dim=1)
    assert knn_accuracy(bank, y, bank[:8], y[:8], 4, k=5) == 1.0


# ---------------------------------------------------------------------------- compatibility
def test_classifier_head_is_not_transferred():
    m = build_model({"name": "resnet18", "pretrained": False, "input_size": 64}, 43)
    assert _classifier_prefixes(m) == ["backbone.fc."]


def test_older_runs_keep_their_fingerprints():
    # values written next to the 15 Oct 2026 pilot checkpoints: they must not change, or every
    # finished run would be retrained
    saved = {"resnet18_full": "a30ebc9f2a868891", "fusion_v2": "872a492026715fa9", "ensemble_views_in": "75a3e5b3dd1569f6",
             "smallcnn_base": "058cf6d928c7450b", "deit_s_full": "ed74201c13179c56"}
    for name, fp in saved.items():
        assert fingerprint(load_run_config(name, "pilot")) == fp, name


def test_v3_dependencies_enter_the_fingerprint():
    student = load_run_config("student_v3", "pilot")
    other = json.loads(json.dumps(student))
    other["train"]["distill"]["teacher"] = "ensemble_views"
    assert fingerprint(student) != fingerprint(other)
    ssl = load_run_config("resnet18_v3_ssl", "pilot")
    assert fingerprint(ssl) != fingerprint(load_run_config("resnet18_v3", "pilot"))


# ---------------------------------------------------------------------------- heatmaps
def test_corruption_heatmaps_are_written_one_per_model(tmp_path):
    from roadlens.eval import plots

    rows = []
    for model in ("SVM", "Net A"):
        rows.append({"model": model, "corruption": "clean", "severity": 0, "acc": 0.9})
        for c in plots.HEATMAP_ORDER:
            rows += [{"model": model, "corruption": c, "severity": s, "acc": 1 - 0.1 * s} for s in range(1, 6)]
    paths = plots.plot_corruption_heatmaps(pd.DataFrame(rows), "model", tmp_path)
    assert sorted(p.name for p in paths) == ["net_a.png", "svm.png"] and all(p.stat().st_size > 5000 for p in paths)


### requirements

In [ ]:
%%writefile /content/RoadLens/requirements.txt
# Tested with Python 3.12-3.13, torch 2.8, timm 1.0. On Colab torch, torchvision, timm, numpy,
# scipy, scikit-image and OpenCV are preinstalled; `pip install -r requirements.txt` only adds
# what is missing (it does not downgrade torch).
torch>=2.3
torchvision>=0.18
timm>=1.0
numpy>=1.24
scipy>=1.10
scikit-image>=0.21
opencv-python>=4.8
Pillow>=10.0
imagecorruptions==1.1.2
pandas>=2.0
scikit-learn>=1.3   # experiment 9 (classical learners, k-means)
matplotlib>=3.7
pyyaml>=6.0
tqdm>=4.65
pytest>=7.0          # only for `python -m pytest tests`
# optional: wandb     (enables the --wandb flag)


## 1. Setup and checkpoints

In [ ]:
# ---- Install requirements, import the project, pick the device ----
%cd {CODE}
!pip install -q -r requirements.txt
os.environ['ROADLENS_DATA'] = DATA
os.environ['ROADLENS_RUNS'] = f'{STORE}/runs'           # runs_pilot/ etc. is added from the profile
sys.path.insert(0, CODE)

def run(cmd):
    """Run a shell command with live output; stop the notebook if it fails."""
    print('$', cmd)
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get('_exit_code', 0)
    if code:
        raise RuntimeError(f'command failed (exit code {code}): {cmd}')

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display
from roadlens.runs import ensure_trained, evaluate_run, load_model, load_run_config
from roadlens.eval.evaluate import predict_logits
from roadlens.eval.metrics import corruption_summary
from roadlens.data.loading import corrupted_sets_missing, external_unavailable, load_corrupted_set, load_test
from roadlens.data.classes import GTSRB_CLASSES, GTSRB_SHORT
from roadlens.utils import corruption_names

PROFILE_ARG = PROFILE or None
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Testing works on the CPU but is very slow; Runtime > Change runtime type > T4 GPU')
cfg0 = load_run_config('resnet18_base', PROFILE_ARG)    # data settings shared by every run
CORRUPTIONS = corruption_names(cfg0)
pd.set_option('display.width', 200)

In [ ]:
# ---- Which trained models are on Drive? ----
rows = []
for done in sorted(glob.glob(f'{RUNS_DIR}/*/seed{SEED}/done.json')):
    info, best = json.load(open(done)), os.path.join(os.path.dirname(done), 'best.pt')
    rows.append({'run': info['name'],
                 'kind': 'ensemble (no weights of its own)' if info.get('skipped') else 'trained',
                 'best epoch': info.get('best_epoch'),
                 'best val-C acc %': round(100 * info['best_val_c_acc'], 2) if info.get('best_val_c_acc') else None,
                 'train min': round(info.get('train_time_s', 0) / 60, 1),
                 'best.pt MB': round(os.path.getsize(best) / 2**20, 1) if os.path.exists(best) else None})
found = pd.DataFrame(rows)
display(found)
trained = set(found.run) if len(found) else set()

# ensembles only combine trained models: set them up (nothing is trained) once their parts exist
for name in ENSEMBLES:
    parts = load_run_config(name, PROFILE_ARG)['model'].get('branches', [])
    if name not in trained and parts and all(p in trained for p in parts):
        ensure_trained(name, SEED, PROFILE_ARG, DEVICE)
        trained.add(name)

TEST = [m for m in (MODELS or ORDER) if m in trained]
not_ready = [m for m in (MODELS or []) if m not in trained]
print('testing:', ', '.join(TEST))
if not_ready:
    print('not trained yet, skipped:', ', '.join(not_ready))
if not TEST:
    raise RuntimeError(f'no finished model in {RUNS_DIR} for seed {SEED}: check PROFILE, SEED and STORE')

## 2. Data

In [ ]:
# ---- Test data: GTSRB (+ BTSD), and the corrupted test sets ----
run('python scripts/prepare_data.py --datasets gtsrb' + (' btsd' if TEST_BTSD else ''))

DATASETS = ['gtsrb'] + (['btsd'] if TEST_BTSD else [])
SUB = {'smoke': '_sub200', 'pilot': '_sub2000', '': ''}[PROFILE]
C_DIRS = [f'{d}_c_s64{SUB}' for d in DATASETS]
C_FILTER = ' '.join(f'--include="{d}/***"' for d in C_DIRS) + ' --exclude="*"'
os.makedirs(DATA, exist_ok=True)
if os.path.isdir(CACHE):              # saved on Drive by the training notebook
    run(f'rsync -a {C_FILTER} "{CACHE}/" "{DATA}/"')

generated = False
for ds in DATASETS:
    missing = corrupted_sets_missing(cfg0, ds)
    if not missing:
        print(f'{ds}: all corrupted test sets are ready')
    elif GENERATE_MISSING_C:
        print(f'{ds}: building {len(missing)} missing corrupted sets')
        run(f'python scripts/make_gtsrb_c.py --dataset {ds} {PROF}' + (' --skip-val-c' if ds == 'gtsrb' else ''))
        generated = True
    else:
        print(f'{ds}: {len(missing)} corrupted sets missing; its corrupted tests are skipped')
if generated:                          # keep them for next time
    run(f'mkdir -p "{CACHE}" && rsync -a {C_FILTER} "{DATA}/" "{CACHE}/"')
C_READY = {ds: not corrupted_sets_missing(cfg0, ds) for ds in DATASETS}

## 3. Tests
Tables are also saved as CSV in `test_results*/`.

In [ ]:
# ---- Test 1: clean GTSRB test photos ----
rows = []
for m in TEST:
    df, _ = evaluate_run(m, SEED, 'gtsrb_clean', profile=PROFILE_ARG, device=DEVICE, force=FORCE)
    t = df[df.split == 'test'].iloc[0]
    rows.append({'model': m, 'test images': int(t.n), 'accuracy %': 100 * t.acc,
                 'macro-F1 %': 100 * t.macro_f1, 'ECE %': 100 * t.ece})
clean = pd.DataFrame(rows).set_index('model')
clean.to_csv(f'{OUT}/clean_seed{SEED}.csv', float_format='%.4f')
display(clean.round(2))

In [ ]:
# ---- Test 2: the 14 corruptions x 5 severities (GTSRB-C) ----
long = {}
if not C_READY['gtsrb']:
    print('skipped: the corrupted GTSRB test sets are missing (set GENERATE_MISSING_C = True)')
else:
    for m in TEST:
        long[m], _ = evaluate_run(m, SEED, 'gtsrb_c', profile=PROFILE_ARG, device=DEVICE, force=FORCE)
    norm = long.get('smallcnn_base')            # mCE reference (= 100)
    if norm is None and 'smallcnn_base' in trained:
        norm, _ = evaluate_run('smallcnn_base', SEED, 'gtsrb_c', profile=PROFILE_ARG, device=DEVICE)
    cols = {'clean_acc': 'clean acc %', 'corr_acc': 'corrupted acc %', 'mce': 'mCE',
            'rel_mce': 'relative mCE', 'corr_f1': 'corrupted macro-F1 %', 'corr_ece': 'corrupted ECE %'}
    summary = pd.DataFrame({m: corruption_summary(df, norm) for m, df in long.items()}).T
    summary = summary[[c for c in cols if c in summary]].rename(columns=cols)
    summary.index.name = 'model'
    summary.to_csv(f'{OUT}/corrupted_summary_seed{SEED}.csv', float_format='%.4f')
    print('mCE: mistakes on corrupted photos relative to SmallCNN (= 100); lower is better')
    display(summary.round(2))

    per = pd.DataFrame({m: 100 * df[df.corruption != 'clean'].groupby('corruption').acc.mean()
                        for m, df in long.items()}).reindex(CORRUPTIONS)
    per.to_csv(f'{OUT}/per_corruption_seed{SEED}.csv', float_format='%.3f')
    print('accuracy % per corruption (mean over the 5 severities)')
    display(per.round(1).style.background_gradient(axis=1, cmap='RdYlGn'))

In [ ]:
# ---- Accuracy as the damage gets stronger ----
if long:
    fig, ax = plt.subplots(figsize=(8, 5))
    for m, df in long.items():
        sev = 100 * df[df.corruption != 'clean'].groupby('severity').acc.mean()
        ax.plot([0, *sev.index], [100 * df[df.corruption == 'clean'].acc.iloc[0], *sev.values], marker='o', label=m)
    ax.set(xlabel='severity (0 = clean)', ylabel='accuracy % (mean over 14 corruptions)',
           title=f'GTSRB-C, seed {SEED}', xticks=range(6))
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    fig.savefig(f'{OUT}/severity_curves_seed{SEED}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ---- Test 3: Belgian signs (models trained on German signs only) ----
if TEST_BTSD:
    reason = external_unavailable(cfg0, 'btsd')
    if reason:
        print('BTSD skipped:', reason)
    else:
        rows = []
        for m in TEST:
            df, _ = evaluate_run(m, SEED, 'btsd_c', profile=PROFILE_ARG, device=DEVICE, force=FORCE)
            c, k = df[df.corruption == 'clean'].iloc[0], df[df.corruption != 'clean']
            rows.append({'model': m, 'images': int(c.n), 'clean acc %': 100 * c.acc,
                         'clean acc, shared classes %': 100 * c.acc_shared,
                         'corrupted acc %': 100 * k.acc.mean(), 'corrupted acc, shared classes %': 100 * k.acc_shared.mean()})
        btsd = pd.DataFrame(rows).set_index('model')
        btsd.to_csv(f'{OUT}/btsd_seed{SEED}.csv', float_format='%.4f')
        print("'shared classes': the answer may only be one of the 22 sign types that exist in both countries")
        display(btsd.round(2))

## 4. Look inside one model

In [ ]:
# ---- Which signs get mixed up? (one model, clean test set) ----
SHOW_MODEL = 'fusion_full' if 'fusion_full' in TEST else TEST[-1]     # change to any name in TEST
_, arrays = evaluate_run(SHOW_MODEL, SEED, 'gtsrb_clean', profile=PROFILE_ARG, device=DEVICE)
cm = arrays['confusion'][2]                     # sets are train, val, test
rate = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(11, 10))
ax.imshow(np.sqrt(rate), cmap='Blues')
ax.set_xticks(range(43), GTSRB_SHORT, rotation=90, fontsize=6)
ax.set_yticks(range(43), GTSRB_SHORT, fontsize=6)
ax.set(xlabel='predicted', ylabel='true', title=f'{SHOW_MODEL}: confusion matrix, clean test (sqrt scale)')
plt.show()

off = cm.copy()
np.fill_diagonal(off, 0)
top = np.dstack(np.unravel_index(np.argsort(off, axis=None)[::-1][:10], off.shape))[0]
display(pd.DataFrame([{'true sign': GTSRB_CLASSES[t], 'predicted as': GTSRB_CLASSES[p], 'times': int(off[t, p]),
                       'share of that sign %': round(100 * off[t, p] / cm[t].sum(), 2)}
                      for t, p in top if off[t, p] > 0]))

In [ ]:
# ---- Example predictions: the same photos clean and corrupted ----
CORRUPTION, SEVERITY, N = 'fog', 3, 12          # CORRUPTION: any name printed below
print('corruptions:', ', '.join(CORRUPTIONS))
model, cfg = load_model(SHOW_MODEL, SEED, DEVICE, PROFILE_ARG)
x, y = load_test(cfg, 'gtsrb')
idx = np.sort(np.random.RandomState(1).choice(len(x), N, replace=False))
views = [('clean', x[idx])]
if C_READY['gtsrb']:
    xc, yc = load_corrupted_set(cfg, 'gtsrb', CORRUPTION, SEVERITY)
    assert (yc[idx] == y[idx]).all()
    views.append((f'{CORRUPTION}, severity {SEVERITY}', np.asarray(xc[idx])))

fig, axes = plt.subplots(len(views), N, figsize=(1.6 * N, 2.3 * len(views)), squeeze=False)
for r, (title, imgs) in enumerate(views):
    probs = torch.softmax(predict_logits(model, imgs, DEVICE), 1)
    conf, pred = probs.max(1)
    for i, ax in enumerate(axes[r]):
        ax.imshow(imgs[i])
        ax.axis('off')
        ok = int(pred[i]) == int(y[idx[i]])
        ax.set_title(f'{GTSRB_SHORT[int(pred[i])]}\n{100 * float(conf[i]):.0f}%', fontsize=8,
                     color='green' if ok else 'red')
    axes[r][0].text(-0.15, 0.5, title, transform=axes[r][0].transAxes, rotation=90, va='center', ha='right', fontsize=9)
fig.suptitle(f'{SHOW_MODEL}: prediction and confidence (green = right, red = wrong)', fontsize=10)
plt.tight_layout()
plt.show()
print('true labels:', ', '.join(GTSRB_SHORT[int(t)] for t in y[idx]))

In [ ]:
# ---- Try your own photos ----
# Crop the photo close around one sign (the training photos have a small border), then upload it.
from google.colab import files
from PIL import Image
size = cfg['data']['image_size']
for fname, data in files.upload().items():
    im = Image.open(io.BytesIO(data)).convert('RGB')
    arr = np.asarray(im.resize((size, size), Image.BILINEAR))[None]       # same resizing as the datasets
    p = torch.softmax(predict_logits(model, arr, DEVICE), 1)[0]
    conf, cls = p.topk(5)
    fig, (a, b) = plt.subplots(1, 2, figsize=(9, 2.8), gridspec_kw={'width_ratios': [1, 2.5]})
    a.imshow(arr[0])
    a.axis('off')
    a.set_title(fname, fontsize=8)
    b.barh([GTSRB_CLASSES[int(c)] for c in cls][::-1], 100 * conf.numpy()[::-1], color='#2a78d6')
    b.set(xlim=(0, 100), xlabel='confidence %', title=f'{SHOW_MODEL}: top 5')
    plt.tight_layout()
    plt.show()

## 5. Save

In [ ]:
# ---- Download every table and figure of this notebook ----
print('saved in', OUT)
for f in sorted(os.listdir(OUT)):
    print('  ', f)
shutil.make_archive(f'/content/test_results{SUFFIX}', 'zip', OUT)
from google.colab import files
files.download(f'/content/test_results{SUFFIX}.zip')